In [1]:
!pip install pyHSICLasso

In [2]:
!pip install ucimlrepo

In [3]:
!pip install umap

In [3]:
!pip install shap

In [4]:
!pip install torch

In [5]:
!pip install scikit-learn

In [6]:
import sys
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn
!{sys.executable} -m pip install --no-cache-dir numpy scipy scikit-learn

In [103]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
from time import perf_counter
from typing import Any, Dict, List, Optional, Callable

from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
#from umap import UMAP

from pyHSICLasso import HSICLasso

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split, Subset, Dataset, ConcatDataset
from sklearn.model_selection import train_test_split
from typing import Tuple, Optional, Iterable
import random
import shap
import copy
import pickle  
from sklearn.model_selection import StratifiedShuffleSplit
import math
from collections import defaultdict
from sklearn.datasets import fetch_openml
from ucimlrepo import fetch_ucirepo

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from __future__ import annotations

from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KernelDensity

In [2]:
def compute_feature_bounds(dataset: Dataset, pad: float = 0.25):
    if isinstance(dataset, torch.utils.data.TensorDataset):
        X = dataset.tensors[0].to(dtype=torch.float32)
        fmin = X.amin(dim=0)
        fmax = X.amax(dim=0)
        span = fmax - fmin
        return fmin - pad * span, fmax + pad * span

    if isinstance(dataset, torch.utils.data.Subset) and isinstance(dataset.dataset, torch.utils.data.TensorDataset):
        Xfull = dataset.dataset.tensors[0]
        idx = torch.as_tensor(dataset.indices, device=Xfull.device, dtype=torch.long)
        X = Xfull.index_select(0, idx).to(dtype=torch.float32)
        fmin = X.amin(dim=0)
        fmax = X.amax(dim=0)
        span = fmax - fmin
        return fmin - pad * span, fmax + pad * span

    xs = [dataset[i][0] for i in range(len(dataset))]
    if len(xs) == 0:
        raise ValueError("compute_feature_bounds: empty dataset")
    device = xs[0].device if torch.is_tensor(xs[0]) else torch.device("cpu")
    xs = [x.to(device) if torch.is_tensor(x) else torch.as_tensor(x, device=device) for x in xs]
    X = torch.stack(xs, dim=0).to(dtype=torch.float32)
    fmin = X.min(dim=0).values
    fmax = X.max(dim=0).values
    span = fmax - fmin
    return fmin - pad * span, fmax + pad * span


In [3]:
class UniformNoiseDataset(Dataset):
    def __init__(
        self,
        feature_min: torch.Tensor,
        feature_max: torch.Tensor,
        n: int,
        label: int = 0,
        dtype: torch.dtype | None = None
    ):
        assert feature_min.shape == feature_max.shape

        self.dtype = dtype if dtype is not None else torch.float32

        feature_min = feature_min.to(dtype=self.dtype)
        feature_max = feature_max.to(device=feature_min.device, dtype=self.dtype)

        self.feature_min = feature_min
        self.feature_max = feature_max
        self.n = int(n)
        self.label = int(label)

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        u = torch.rand_like(self.feature_min, dtype=self.dtype)
        x = self.feature_min + u * (self.feature_max - self.feature_min)
        y = torch.tensor(self.label, dtype=torch.long, device=x.device)
        return x, y


In [4]:
class LabelMappedSubset(Dataset):
    def __init__(self, subset: Subset, label_map: dict[int, int]):
        self.subset = subset
        self.label_map = label_map

    def __getitem__(self, idx):
        x, y = self.subset[idx]

        if torch.is_tensor(y):
            y_int = int(y.detach().cpu().item())
        else:
            y_int = int(y)

        y_mapped = self.label_map.get(y_int, y_int)
        return x, torch.tensor(y_mapped, dtype=y.dtype, device=y.device)

    def __len__(self):
        return len(self.subset)


In [5]:
class Abs(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.abs(x)

In [6]:
class MLP(nn.Module):
    def __init__(self, d_in: int, d_hidden: int = 10, n_hidden_layers: int = 2):

        super().__init__()

        layers = []

        layers.append(nn.Linear(d_in, d_hidden))
        layers.append(Abs())


        for _ in range(n_hidden_layers - 1):
            layers.append(nn.Linear(d_hidden, d_hidden))
            layers.append(Abs())

        layers.append(nn.Linear(d_hidden, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [7]:
def train_one_model(
    model: nn.Module,
    dataset_pos_as_1: Dataset,
    feature_min: torch.Tensor,
    feature_max: torch.Tensor,
    num_epochs: int = 20,
    batch_size: int = 128,
    lr: float = 3e-3,
    weight_decay: float = 1e-2,
    seed: int = 0,
):
    """
    Обучает модель различать:
      - реальные (label=1)
      - шумовые (label=0)
    с MSELoss.
    """
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = next(model.parameters()).device
    criterion = nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if isinstance(dataset_pos_as_1, TensorDataset):
        X_real = dataset_pos_as_1.tensors[0].to(device=device, dtype=torch.float32)
        n = int(X_real.shape[0])
        if n == 0:
            raise ValueError("train_one_model: empty TensorDataset")

        fmin = feature_min.to(device=device, dtype=torch.float32)
        fmax = feature_max.to(device=device, dtype=torch.float32)
        span = (fmax - fmin)
        d = int(X_real.shape[1])

        for _ in range(num_epochs):
            model.train()
            perm = torch.randperm(n, device=device)

            for start in range(0, n, batch_size):
                idx = perm[start:start + batch_size]
                xb_real = X_real.index_select(0, idx)


                u = torch.rand(xb_real.shape[0], d, device=device, dtype=torch.float32)
                xb_noise = fmin.unsqueeze(0) + u * span.unsqueeze(0) 

                xb = torch.cat([xb_real, xb_noise], dim=0) 
                yb = torch.cat(
                    [
                        torch.ones(xb_real.shape[0], device=device, dtype=torch.float32),
                        torch.zeros(xb_noise.shape[0], device=device, dtype=torch.float32),
                    ],
                    dim=0,
                ).unsqueeze(1) 

                pred = model(xb)
                loss = criterion(pred, yb)

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

        return model


    n_real = len(dataset_pos_as_1)
    if n_real <= 0:
        raise ValueError("train_one_model: empty dataset_pos_as_1")

    n_noise = max(1, n_real)

    x0, _ = dataset_pos_as_1[0]
    data_device = x0.device if torch.is_tensor(x0) else device

    fmin = feature_min.detach().to(device=data_device, dtype=torch.float32)
    fmax = feature_max.detach().to(device=data_device, dtype=torch.float32)

    noise_ds = UniformNoiseDataset(fmin, fmax, n_noise, label=0, dtype=torch.float32)
    train_ds = ConcatDataset([dataset_pos_as_1, noise_ds])
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

    for _ in range(num_epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device=device, dtype=torch.float32)
            yb = yb.to(device=device).unsqueeze(-1).to(dtype=torch.float32)

            pred = model(xb)
            loss = criterion(pred, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    return model

In [8]:
def model_train(
    d_hidden: int,
    n_hidden_layers: int,
    train_data: Dataset,
    num_epochs: int,
    batch_size: int = 128,
    seed: int = 0,
    output: bool = False,
    device: str | torch.device | None = None,
    feature_min: torch.Tensor | None = None,
    feature_max: torch.Tensor | None = None,
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if isinstance(train_data, TensorDataset):
        X_all, y_all = train_data.tensors


        data_device = X_all.device
        dev = data_device if device is None else torch.device(device)

        X_all = X_all.to(device=dev, dtype=torch.float32)
        y_all = y_all.to(device=dev, dtype=torch.long)

        d_in = int(X_all.shape[1])

        if feature_min is None or feature_max is None:
            fmin, fmax = compute_feature_bounds(TensorDataset(X_all, y_all))
        else:
            fmin, fmax = feature_min, feature_max
        fmin = fmin.to(device=dev, dtype=torch.float32)
        fmax = fmax.to(device=dev, dtype=torch.float32)

        pos_mask = (y_all == 1)
        neg_mask = (y_all == -1)
        X_pos = X_all[pos_mask]
        X_neg = X_all[neg_mask]

        if X_pos.numel() == 0 or X_neg.numel() == 0:
            raise ValueError("model_train: empty class after split (pos or neg)")

        model_pos = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(dev)
        model_neg = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(dev)

        ds_pos = TensorDataset(X_pos, torch.ones(X_pos.size(0), device=dev, dtype=torch.long))
        ds_neg = TensorDataset(X_neg, torch.ones(X_neg.size(0), device=dev, dtype=torch.long))

        model_pos = train_one_model(
            model_pos, ds_pos, fmin, fmax,
            num_epochs=num_epochs, batch_size=batch_size, seed=seed
        )
        model_neg = train_one_model(
            model_neg, ds_neg, fmin, fmax,
            num_epochs=num_epochs, batch_size=batch_size, seed=seed + 1
        )
        return model_pos, model_neg

    ys: list[int] = []
    for i in range(len(train_data)):
        y = train_data[i][1]
        if torch.is_tensor(y):
            ys.append(int(y.detach().cpu().item()))
        else:
            ys.append(int(y))
    labels = torch.tensor(ys, dtype=torch.long)

    pos_idx = (labels == 1).nonzero(as_tuple=True)[0].tolist()
    neg_idx = (labels == -1).nonzero(as_tuple=True)[0].tolist()

    train_pos = Subset(train_data, pos_idx)
    train_neg = Subset(train_data, neg_idx)
    train_neg_as_pos = LabelMappedSubset(train_neg, {-1: 1})

    x0, _ = train_data[0]
    d_in = x0.shape[-1]
    data_device = x0.device if torch.is_tensor(x0) else torch.device("cpu")

    dev = data_device if device is None else torch.device(device)

    fmin, fmax = compute_feature_bounds(train_data)
    fmin = fmin.to(dev)
    fmax = fmax.to(dev)

    model_pos = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(dev)
    model_neg = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(dev)

    model_pos = train_one_model(
        model_pos, train_pos, fmin, fmax,
        num_epochs=num_epochs, batch_size=batch_size, seed=seed
    )
    model_neg = train_one_model(
        model_neg, train_neg_as_pos, fmin, fmax,
        num_epochs=num_epochs, batch_size=batch_size, seed=seed + 1
    )
    return model_pos, model_neg

In [283]:
H_TAU = 1e-3


def metrics_from_sheet(X, y, model_pos, model_neg, beta_pos=0.1, beta_neg=0.1):
    device = next(model_pos.parameters()).device
    X = X.to(device=device, dtype=torch.float32)
    y = y.to(device=device)

    model_pos.eval()
    model_neg.eval()
    with torch.no_grad():
        s1 = model_pos(X).flatten()
        s2 = model_neg(X).flatten()

    pos = (y == 1)
    neg = (y == -1)
    n1 = int(pos.sum().item())
    n2 = int(neg.sum().item())

    n02_1 = int((((s1 >= beta_pos) & (s2 <  beta_neg)) & pos).sum().item())
    n12_1 = int((((s1 >= beta_pos) & (s2 >= beta_neg)) & pos).sum().item())

    n02_2 = int((((s2 >= beta_neg) & (s1 <  beta_pos)) & neg).sum().item())
    n12_2 = int((((s2 >= beta_neg) & (s1 >= beta_pos)) & neg).sum().item())

    a1 = n02_1 / n1 if n1 else 0.0
    a2 = n02_2 / n2 if n2 else 0.0
    F12 = 2 * a1 * a2 / (a1 + a2 + H_TAU)

    b1 = n12_1 / n1 if n1 else 0.0
    b2 = n12_2 / n2 if n2 else 0.0
    G12 = 2 * b1 * b2 / (b1 + b2 + H_TAU)
    #G12 = (b1 + b2) / 2
    #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2

    return dict(
        n1=n1, n2=n2,
        n02_1=n02_1, n12_1=n12_1,
        n02_2=n02_2, n12_2=n12_2,
        a1=a1, a2=a2, F12=F12,
        b1=b1, b2=b2, G12=G12
    )

In [10]:
def pick_beta(scores_pos: torch.Tensor,
              scores_noise: torch.Tensor,
              scores_neg: Optional[torch.Tensor] = None,
              n_quantiles: int = 19) -> Tuple[float, float]:

    s_pos = scores_pos.detach().flatten()
    s_no  = scores_noise.detach().flatten()
    s_neg = scores_neg.detach().flatten() if (scores_neg is not None) else None

    if s_pos.numel() == 0:
        return 0.0, -1.0

    device = s_pos.device

    qs = torch.linspace(0.05, 0.95, steps=n_quantiles, device=device)

    if s_neg is None:
        merged = torch.cat([s_pos, s_no])
    else:
        merged = torch.cat([s_pos, s_no, s_neg])

    pool = torch.quantile(merged, qs)

    best_J, best_b = -1.0, float("nan")

    for b_tensor in pool:
        b = float(b_tensor.item())

        tpr = (s_pos >= b).float().mean().item() if s_pos.numel() else 0.0

        if s_neg is None:
            tnr = (s_no < b).float().mean().item() if s_no.numel() else 0.0
        else:
            tnr_no  = (s_no  < b).float().mean().item() if s_no.numel()  else 0.0
            tnr_neg = (s_neg < b).float().mean().item() if s_neg.numel() else 0.0
            tnr = 0.5 * (tnr_no + tnr_neg)

        J = tpr + tnr - 1.0
        if J > best_J:
            best_J, best_b = J, b

    return best_b, best_J


In [11]:
def choose_coord(
    X: torch.Tensor,
    y: torch.Tensor,
    d_hidden: int,
    n_hidden_layers: int,
    num_epochs: int,
    batch_size: int = 128,
    seed: int = 0,
    output: bool = False,
):

    if torch.is_tensor(y) and y.device != X.device:
        y = y.to(X.device)

    n_features = X.shape[1]


    idx_np = torch.arange(len(y)).detach().cpu().numpy()
    y_np = y.detach().cpu().numpy()

    train_idx_np, val_idx_np = train_test_split(
        idx_np, test_size=0.2, random_state=seed, stratify=y_np
    )


    train_idx = torch.as_tensor(train_idx_np, dtype=torch.long, device=X.device)
    val_idx   = torch.as_tensor(val_idx_np,   dtype=torch.long, device=X.device)


    X_tr, y_tr   = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx],   y[val_idx]

    fmin_all, fmax_all = compute_feature_bounds(TensorDataset(X_tr, y_tr))
    fmin_all = fmin_all.to(X_tr.device)
    fmax_all = fmax_all.to(X_tr.device)

    rows = []
    best_idx, best_key = None, -float("inf")

    for i in range(n_features):

        X_tr_upd  = torch.cat((X_tr[:, :i],  X_tr[:, i+1:]),  dim=1)
        X_val_upd = torch.cat((X_val[:, :i], X_val[:, i+1:]), dim=1)

        fmin_upd = torch.cat((fmin_all[:i], fmin_all[i+1:]))
        fmax_upd = torch.cat((fmax_all[:i], fmax_all[i+1:]))

        n_pos_val = int((y_val == 1).sum().item())
        n_neg_val = int((y_val == -1).sum().item())
        n_noise_for_pos = max(1, n_pos_val)
        n_noise_for_neg = max(1, n_neg_val)

        noise_pos_ds = UniformNoiseDataset(fmin_upd, fmax_upd, n_noise_for_pos, label=0, dtype=torch.float32)
        noise_neg_ds = UniformNoiseDataset(fmin_upd, fmax_upd, n_noise_for_neg, label=0, dtype=torch.float32)

        X_noise_pos = torch.stack([noise_pos_ds[j][0] for j in range(len(noise_pos_ds))], dim=0)
        X_noise_neg = torch.stack([noise_neg_ds[j][0] for j in range(len(noise_neg_ds))], dim=0)

        train_data_upd = TensorDataset(X_tr_upd, y_tr)

        model_pos, model_neg = model_train(
            train_data=train_data_upd,
            d_hidden=d_hidden,
            n_hidden_layers=n_hidden_layers,
            num_epochs=num_epochs,
            batch_size=batch_size,
            seed=seed,
            output=output,
        )

        model_pos.eval()
        model_neg.eval()
        dev_pos = next(model_pos.parameters()).device
        dev_neg = next(model_neg.parameters()).device

        with torch.no_grad():
            s1_pos = model_pos(X_val_upd[(y_val == 1)]).flatten()
            s1_no  = model_pos(X_noise_pos).flatten()
            beta_pos, _ = pick_beta(s1_pos, s1_no)

            s2_neg = model_neg(X_val_upd[(y_val == -1)]).flatten()
            s2_no  = model_neg(X_noise_neg).flatten()
            beta_neg, _ = pick_beta(s2_neg, s2_no)

        cG_grid = [0.1, 0.25, 0.5, 0.75, 1.0]
        best_S, best_tuple, best_m = -float("inf"), None, None

        for cG in cG_grid:
            m_try = metrics_from_sheet(
                X_val_upd, y_val, model_pos, model_neg,
                beta_pos=beta_pos, beta_neg=beta_neg
            )
            S_try = m_try["F12"] - cG * m_try["G12"]
            if S_try > best_S:
                best_S, best_tuple, best_m = S_try, (beta_pos, beta_neg, cG), m_try

        m = dict(best_m)
        m["removed_idx"] = i
        m["beta_pos_best"], m["beta_neg_best"], m["cG_best"] = best_tuple
        m["S"] = best_S
        rows.append(m)

        if best_S > best_key:
            best_key = best_S
            best_idx = i

    df = (
        pd.DataFrame(rows)
          .sort_values(["S", "F12", "G12"], ascending=[False, False, True])
          .reset_index(drop=True)
    )
    return best_idx, df


In [12]:
def set_seed(seed: int = 42):
    import random
    import numpy as np
    import torch

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



In [13]:
def make_dataset_A(seed=42):
    X, y = make_classification(
        n_samples=3000, n_features=8, n_informative=3, n_redundant=2,
        class_sep=1.2, flip_y=0.02, random_state=seed)
    X[:, 0] = np.sin(X[:, 0]) + 0.1 * X[:, 1]
    X[:, 2] = X[:, 2] ** 2
    y = y.astype(int)
    y = 2*y - 1
    return X.astype(np.float64), y.astype(int)

In [261]:
def make_dataset_B(seed=42):
    X, y = make_classification(
        n_samples=6000, n_features=30, n_informative=5, n_redundant=5,
        class_sep=0.8, flip_y=0.01, weights=[0.75, 0.25], random_state=seed)
    X[:, 10] = np.sin(X[:, 0]) * X[:, 1]
    X[:, 11] = X[:, 2] ** 2
    X[:, 12] = np.exp(-X[:, 3]**2)
    X[:, 20:25] = X[:, :5] + 0.05 * np.random.randn(len(X), 5)
    y = y.astype(int)
    y = 2*y - 1
    return X.astype(np.float64), y.astype(int)

In [15]:
def make_dataset_A_real(seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    ds = fetch_ucirepo(id=545) 
    X_df = ds.data.features
    y_df = ds.data.targets

    X = X_df.to_numpy(dtype=np.float64)
    y_raw = np.asarray(y_df.squeeze()) 

    classes = np.unique(y_raw)

    pos_label = "Osmancik" if "Osmancik" in classes else classes[-1]
    y = np.where(y_raw == pos_label, 1, -1).astype(int)

    return X.astype(np.float64), y.astype(int)


In [16]:
def make_dataset_B_real(seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    ds = fetch_ucirepo(id=327) 
    X = ds.data.features.to_numpy(dtype=np.float64)
    y_raw = np.asarray(ds.data.targets.squeeze())

    y_num = y_raw.astype(int)

    y = np.where(y_num > 0, 1, -1).astype(int)

    return X.astype(np.float64), y.astype(int)

In [14]:
def make_dataset_colon(seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    # загрузка
    data = fetch_openml(data_id=45087, as_frame=True)

    # признаки
    X = data.data.to_numpy(dtype=np.float64)

    # таргет
    y_raw = np.asarray(data.target)

    # приведение к int (иногда OpenML хранит как str)
    y_num = y_raw.astype(int)

    # перевод в {-1, 1} (на всякий случай, даже если уже так)
    y = np.where(y_num > 0, 1, -1).astype(int)

    return X.astype(np.float64), y.astype(int)

In [15]:
class ReducerBase:
    def fit(self, X, y=None): ...
    def transform(self, X): ...
    def fit_transform(self, X, y=None):
        self.fit(X, y)
        return self.transform(X)

    @staticmethod
    def _to_numpy(X):
        if torch.is_tensor(X):
            return X.detach().cpu().numpy()
        return np.asarray(X)

    @staticmethod
    def _to_numpy_y(y):
        if y is None:
            return None
        if torch.is_tensor(y):
            return y.detach().cpu().numpy()
        return np.asarray(y)

In [15]:
class PCAReducer(ReducerBase):
    def __init__(self, n_components):
        self.pca = PCA(n_components=n_components, random_state=42)

    def fit(self, X, y=None):
        Xn = self._to_numpy(X)
        self.pca.fit(Xn)
        return self

    def transform(self, X):
        Xn = self._to_numpy(X)
        return self.pca.transform(Xn)

In [16]:
class PLSReducer(ReducerBase):
    def __init__(self, n_components):
        self.pls = PLSRegression(n_components=n_components, scale=False)

    def fit(self, X, y):
        Xn = self._to_numpy(X)
        yn = self._to_numpy_y(y).ravel().astype(float)
        self.pls.fit(Xn, yn)
        return self

    def transform(self, X):
        Xn = self._to_numpy(X)
        return self.pls.transform(Xn)

In [17]:

class UMAPReducer(ReducerBase):
    def __init__(self, n_components):
        self.umap = UMAP(
            n_components=n_components,
            random_state=42,
            n_neighbors=15,
            min_dist=0.1,
            target_metric='categorical'
        )

    def fit(self, X, y):
        Xn = self._to_numpy(X)
        yn = self._to_numpy_y(y)
        self.umap.fit(Xn, yn)
        return self

    def transform(self, X):
        Xn = self._to_numpy(X)
        return self.umap.transform(Xn)

In [18]:
class HSICSelector(ReducerBase):
    def __init__(self, n_select):
        self.n_select = int(n_select)
        self.model = HSICLasso()
        self.mask_ = None

    def fit(self, X, y):
        Xn = self._to_numpy(X)
        yn = self._to_numpy_y(y).ravel().astype(int)

        self.model.input(Xn, yn)
        self.model.classification(self.n_select)

        idx = np.array(self.model.get_index(), dtype=int)
        self.mask_ = np.zeros(Xn.shape[1], dtype=bool)
        self.mask_[idx] = True
        return self

    def transform(self, X):
        Xn = self._to_numpy(X)
        return Xn[:, self.mask_]

In [19]:
class MLPReducer(ReducerBase):
    def __init__(self, d_hidden, n_hidden_layers, num_epochs, batch_size, n_select, seed=0, output=False, device: str = "cpu",):
        self.d_hidden = d_hidden
        self.n_hidden_layers = n_hidden_layers
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.seed = seed
        self.output = output
        self.n_select = n_select

        self.selected_indices_ = None
        self.removed_indices_ = None
        self.history_ = []
        self.device = torch.device(device)
        

    def _to_torch2d(self, X):
        if isinstance(X, torch.Tensor):
            assert X.ndim == 2, "X must be 2D"
            return X.to(device=self.device, dtype=torch.float32)
        X = np.asarray(X)
        assert X.ndim == 2, "X must be 2D"
        return torch.from_numpy(X).to(device=self.device, dtype=torch.float32)

    def _to_torch1d_int(self, y):
        if isinstance(y, torch.Tensor):
            assert y.ndim == 1, "y must be 1D"
            return y.to(device=self.device, dtype=torch.int64)
        y = np.asarray(y).ravel()
        return torch.from_numpy(y.astype(np.int64)).to(device=self.device)


    def fit(self, X, y):
        X_t = self._to_torch2d(X)
        y_t = self._to_torch1d_int(y)

        y_t = y_t.to(device=X_t.device)

        d = X_t.shape[1]
        keep = list(range(d))
        removed = []

        while len(keep) > self.n_select:
            X_cur = X_t[:, keep]

            best_idx, df = choose_coord(
                X=X_cur, y=y_t,
                d_hidden=self.d_hidden,
                n_hidden_layers=self.n_hidden_layers,
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                seed=self.seed,
                output=self.output,
            )

            removed_global = keep[best_idx]
            removed.append(removed_global)
            del keep[best_idx]

            self.history_.append({
                "removed_global": removed_global,
                "n_left": len(keep),
                "df": df,
            })

        self.selected_indices_ = keep
        self.removed_indices_ = removed
        return self

    def transform(self, X):
        assert self.selected_indices_ is not None, "Call fit() before transform()."

        if isinstance(X, torch.Tensor):
            return X[:, self.selected_indices_]
        X = np.asarray(X)
        return X[:, self.selected_indices_]

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

    def get_support(self, indices=False):
        assert self.selected_indices_ is not None, "Call fit() first."
        if indices:
            return np.array(self.selected_indices_, dtype=int)

        if self.selected_indices_:
            mask_len = max(self.selected_indices_) + 1
        else:
            mask_len = self.n_select

        mask = np.zeros(mask_len, dtype=bool)
        if self.selected_indices_:
            mask[self.selected_indices_] = True
        return mask


In [16]:
def sample_uniform(fmin: torch.Tensor, fmax: torch.Tensor, n: int) -> torch.Tensor:

    fmax = fmax.to(device=fmin.device, dtype=fmin.dtype)


    u = torch.rand(n, fmin.numel(), dtype=fmin.dtype, device=fmin.device)
    return fmin.unsqueeze(0) + (fmax - fmin).unsqueeze(0) * u


In [291]:
def get_models(seed: int = 42) -> Dict[str, Any]:
    """
    Реестр всех post-models для evaluate_reduction.
    Все модели должны уметь либо predict_proba, либо decision_function.
    """
    return {
        "MLP": MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=500,
            random_state=seed,
        ),
        "HGBT": HistGradientBoostingClassifier(
            random_state=seed,
        ),
        "SVM_linear": SVC(
            C=1.0,
            kernel="linear",
            probability=True,
            random_state=seed,
        ),
        "SVM_rbf": SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            random_state=seed,
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="uniform",
        ),
        "GaussianNB": GaussianNB(),
        "QDA": QuadraticDiscriminantAnalysis(
            reg_param=0.2,
        ),
        "ShrinkageLDA": LinearDiscriminantAnalysis(
            solver="lsqr",
            shrinkage="auto",
        ),
        "LogRegL2": LogisticRegression(
            penalty="l2",
            solver="liblinear",
            max_iter=5000,
            random_state=seed,
        ),
        "KDEBayes": KDEBayesClassifier(
            bandwidth=0.5,
            kernel="gaussian",
        ),
    }

In [285]:
class BatchedMLP(nn.Module):
    """
    Пакет из B независимых MLP одинаковой архитектуры и одинаковой входной размерности.
    Вход:
        x: [B, N, d]  или [B, d]
    Выход:
        [B, N, 1]     или [B, 1]
    """
    def __init__(
        self,
        num_models: int,
        d_in: int,
        d_hidden: int = 10,
        n_hidden_layers: int = 2,
        device: str | torch.device | None = None,
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()
        self.num_models = int(num_models)
        self.d_in = int(d_in)
        self.d_hidden = int(d_hidden)
        self.n_hidden_layers = int(n_hidden_layers)

        dims = [self.d_in] + [self.d_hidden] * self.n_hidden_layers + [1]

        self.weights = nn.ParameterList()
        self.biases = nn.ParameterList()

        for fan_in, fan_out in zip(dims[:-1], dims[1:]):
            w = nn.Parameter(
                torch.empty(self.num_models, fan_out, fan_in, device=device, dtype=dtype)
            )
            b = nn.Parameter(
                torch.empty(self.num_models, fan_out, device=device, dtype=dtype)
            )
            self.weights.append(w)
            self.biases.append(b)

        self.reset_parameters()

    def reset_parameters(self):
        for w, b in zip(self.weights, self.biases):
            fan_in = int(w.shape[-1])
            bound = 1.0 / math.sqrt(fan_in)
            with torch.no_grad():
                w.uniform_(-bound, bound)
                b.uniform_(-bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze_out = False

        if x.ndim == 2:
            x = x.unsqueeze(1)   # [B, 1, d]
            squeeze_out = True

        if x.ndim != 3:
            raise ValueError(f"BatchedMLP expects 2D or 3D input, got shape={tuple(x.shape)}")
        if x.shape[0] != self.num_models:
            raise ValueError(
                f"Batch size of models mismatch: got x.shape[0]={x.shape[0]}, "
                f"but num_models={self.num_models}"
            )

        for li, (w, b) in enumerate(zip(self.weights, self.biases)):
            # x: [B, N, in], w: [B, out, in] -> [B, N, out]
            x = torch.einsum("bni,boi->bno", x, w) + b.unsqueeze(1)
            if li < len(self.weights) - 1:
                x = torch.abs(x)

        if squeeze_out:
            x = x.squeeze(1)
        return x

    def extract_model(self, idx: int) -> nn.Module:
        """
        Достаёт idx-ю обычную MLP из батча.
        """
        idx = int(idx)
        model = MLP(
            d_in=self.d_in,
            d_hidden=self.d_hidden,
            n_hidden_layers=self.n_hidden_layers
        )

        linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]

        for layer, w, b in zip(linear_layers, self.weights, self.biases):
            layer.weight.data.copy_(w[idx].detach().cpu())
            layer.bias.data.copy_(b[idx].detach().cpu())

        return model
    
    def get_model_state(self, idx: int) -> dict:
        idx = int(idx)
        return {
            "d_in": self.d_in,
            "d_hidden": self.d_hidden,
            "n_hidden_layers": self.n_hidden_layers,
            "weights": [w[idx].detach().cpu().clone() for w in self.weights],
            "biases": [b[idx].detach().cpu().clone() for b in self.biases],
        }
    
    @staticmethod
    def build_model_from_state(state: dict, device: str | torch.device = "cpu") -> nn.Module:
        model = MLP(
            d_in=state["d_in"],
            d_hidden=state["d_hidden"],
            n_hidden_layers=state["n_hidden_layers"],
        ).to(device)

        linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]
        for layer, w, b in zip(linear_layers, state["weights"], state["biases"]):
            layer.weight.data.copy_(w.to(device))
            layer.bias.data.copy_(b.to(device))

        model.eval()
        return model
    
    @staticmethod
    def build_batched_from_states(
        states: list[dict],
        device: str | torch.device = "cpu",
        dtype: torch.dtype | None = None,
    ) -> "BatchedMLP":
        """
        Собирает один BatchedMLP из списка checkpoint-state'ов отдельных моделей.

        Каждый элемент states должен иметь формат, возвращаемый get_model_state():
        {
            "d_in": ...,
            "d_hidden": ...,
            "n_hidden_layers": ...,
            "weights": [tensor(out, in), ...],
            "biases":  [tensor(out), ...],
        }
        """
        if not isinstance(states, list) or len(states) == 0:
            raise ValueError("states must be a non-empty list")

        first = states[0]
        if first is None:
            raise ValueError("states[0] is None")

        d_in = int(first["d_in"])
        d_hidden = int(first["d_hidden"])
        n_hidden_layers = int(first["n_hidden_layers"])
        num_models = len(states)

        if dtype is None:
            dtype = first["weights"][0].dtype

        model = BatchedMLP(
            num_models=num_models,
            d_in=d_in,
            d_hidden=d_hidden,
            n_hidden_layers=n_hidden_layers,
            device=device,
            dtype=dtype,
        )

        n_layers = len(first["weights"])

        for s_idx, st in enumerate(states):
            if st is None:
                raise ValueError(f"states[{s_idx}] is None")

            if int(st["d_in"]) != d_in:
                raise ValueError(f"d_in mismatch at states[{s_idx}]")
            if int(st["d_hidden"]) != d_hidden:
                raise ValueError(f"d_hidden mismatch at states[{s_idx}]")
            if int(st["n_hidden_layers"]) != n_hidden_layers:
                raise ValueError(f"n_hidden_layers mismatch at states[{s_idx}]")
            if len(st["weights"]) != n_layers or len(st["biases"]) != n_layers:
                raise ValueError(f"Number of layers mismatch at states[{s_idx}]")

        with torch.no_grad():
            for li in range(n_layers):
                w_stack = torch.stack(
                    [st["weights"][li].to(device=device, dtype=dtype) for st in states],
                    dim=0,
                )  # [B, out, in]

                b_stack = torch.stack(
                    [st["biases"][li].to(device=device, dtype=dtype) for st in states],
                    dim=0,
                )  # [B, out]

                if model.weights[li].shape != w_stack.shape:
                    raise ValueError(
                        f"Weight shape mismatch at layer {li}: "
                        f"{tuple(model.weights[li].shape)} vs {tuple(w_stack.shape)}"
                    )
                if model.biases[li].shape != b_stack.shape:
                    raise ValueError(
                        f"Bias shape mismatch at layer {li}: "
                        f"{tuple(model.biases[li].shape)} vs {tuple(b_stack.shape)}"
                    )

                model.weights[li].copy_(w_stack)
                model.biases[li].copy_(b_stack)

        model.eval()
        return model


def _gather_subspaces_2d(X: torch.Tensor, subs_idx: torch.Tensor) -> torch.Tensor:
    """
    X:        [N, p]
    subs_idx: [B, d]
    return:   [B, N, d]
    """
    if X.ndim != 2:
        raise ValueError(f"X must be 2D, got shape={tuple(X.shape)}")
    if subs_idx.ndim != 2:
        raise ValueError(f"subs_idx must be 2D, got shape={tuple(subs_idx.shape)}")

    B, d = subs_idx.shape
    N, p = X.shape

    Xb = X.unsqueeze(0).expand(B, N, p)
    idx = subs_idx.unsqueeze(1).expand(B, N, d)
    return torch.gather(Xb, dim=2, index=idx)


def _sample_uniform_batch(
    feature_min_batch: torch.Tensor,
    feature_max_batch: torch.Tensor,
    n: int,
) -> torch.Tensor:
    """
    feature_min_batch, feature_max_batch: [B, d]
    return: [B, n, d]
    """
    feature_min_batch = feature_min_batch.to(dtype=torch.float32)
    feature_max_batch = feature_max_batch.to(
        device=feature_min_batch.device,
        dtype=torch.float32
    )

    B, d = feature_min_batch.shape
    u = torch.rand(B, n, d, device=feature_min_batch.device, dtype=torch.float32)
    span = (feature_max_batch - feature_min_batch).unsqueeze(1)
    return feature_min_batch.unsqueeze(1) + u * span


def _batched_train_one_side(
    X_real_batch: torch.Tensor,
    feature_min_batch: torch.Tensor,
    feature_max_batch: torch.Tensor,
    d_hidden: int,
    n_hidden_layers: int,
    num_epochs: int,
    batch_size: int = 128,
    lr: float = 3e-3,
    weight_decay: float = 1e-2,
    seed: int = 0,
) -> BatchedMLP:
    """
    Одновременно обучает B моделей одной стороны:
      - либо все positive-модели,
      - либо все negative-модели.

    X_real_batch: [B, N_real, d]
    feature_min_batch / feature_max_batch: [B, d]
    """
    if X_real_batch.ndim != 3:
        raise ValueError(f"X_real_batch must be 3D, got shape={tuple(X_real_batch.shape)}")

    B, N_real, d = X_real_batch.shape
    if N_real <= 0:
        raise ValueError("No training objects for batched side training")

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = X_real_batch.device
    dtype = X_real_batch.dtype

    model = BatchedMLP(
        num_models=B,
        d_in=d,
        d_hidden=d_hidden,
        n_hidden_layers=n_hidden_layers,
        device=device,
        dtype=dtype,
    )

    criterion = nn.MSELoss()
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    fmin = feature_min_batch.to(device=device, dtype=dtype).unsqueeze(1)   # [B, 1, d]
    span = (
        feature_max_batch.to(device=device, dtype=dtype)
        - feature_min_batch.to(device=device, dtype=dtype)
    ).unsqueeze(1)                                                          # [B, 1, d]

    for _ in range(num_epochs):
        # Независимая перестановка объектов для каждой модели
        perm = torch.argsort(
            torch.rand(B, N_real, device=device, dtype=dtype),
            dim=1
        )  # [B, N_real]

        for start in range(0, N_real, batch_size):
            idx = perm[:, start:start + batch_size]                         # [B, bs]
            cur_bs = int(idx.shape[1])

            idx_ex = idx.unsqueeze(-1).expand(-1, -1, d)                    # [B, bs, d]
            xb_real = torch.gather(X_real_batch, dim=1, index=idx_ex)       # [B, bs, d]

            u = torch.rand(B, cur_bs, d, device=device, dtype=dtype)
            xb_noise = fmin.expand(-1, cur_bs, -1) + u * span.expand(-1, cur_bs, -1)

            xb = torch.cat([xb_real, xb_noise], dim=1)                      # [B, 2*bs, d]
            yb = torch.cat(
                [
                    torch.ones(B, cur_bs, 1, device=device, dtype=dtype),
                    torch.zeros(B, cur_bs, 1, device=device, dtype=dtype),
                ],
                dim=1,
            )                                                                # [B, 2*bs, 1]

            pred = model(xb)  # [B, 2*bs, 1]

            loss_per_model = ((pred - yb) ** 2).mean(dim=(1, 2))   # [B]
            loss = loss_per_model.sum()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    return model


def _batched_metrics_from_scores(
    s1: torch.Tensor,
    s2: torch.Tensor,
    y: torch.Tensor,
    beta_pos: torch.Tensor,
    beta_neg: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """
    s1, s2: [B, N]
    y:      [N]
    beta_*: [B]
    """
    device = s1.device
    y = y.to(device=device)

    pos = (y == 1)
    neg = (y == -1)

    n1 = max(1, int(pos.sum().item()))
    n2 = max(1, int(neg.sum().item()))

    m1 = (s1 >= beta_pos.unsqueeze(1))
    m2 = (s2 >= beta_neg.unsqueeze(1))

    pos_mask = pos.unsqueeze(0)
    neg_mask = neg.unsqueeze(0)

    n02_1 = ((m1 & ~m2) & pos_mask).sum(dim=1).to(dtype=torch.float32)
    n02_2 = ((m2 & ~m1) & neg_mask).sum(dim=1).to(dtype=torch.float32)

    a1 = n02_1 / n1
    a2 = n02_2 / n2
    F12 = 2.0 * a1 * a2 / (a1 + a2 + H_TAU)

    n12_1 = ((m1 & m2) & pos_mask).sum(dim=1).to(dtype=torch.float32)
    n12_2 = ((m2 & m1) & neg_mask).sum(dim=1).to(dtype=torch.float32)

    b1 = n12_1 / n1
    b2 = n12_2 / n2
    G12 = 2.0 * b1 * b2 / (b1 + b2 + H_TAU)
    #G12 = (b1 + b2) / 2
    #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2

    return {
        "F12": F12,
        "G12": G12,
        "a1": a1,
        "a2": a2,
        "b1": b1,
        "b2": b2,
    }


def _batched_tune_betas_joint(
    model_pos: BatchedMLP,
    model_neg: BatchedMLP,
    X_val_batch: torch.Tensor,
    y_val: torch.Tensor,
    fmin_batch: torch.Tensor,
    fmax_batch: torch.Tensor,
    n_grid: int = 19,
    cG: float = 1,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Подбирает только beta_pos, beta_neg при фиксированном cG.
    Возвращает:
        beta_pos_best: [B]
        beta_neg_best: [B]
        cG_best:       [B]
        S_best:        [B]
    """
    device = X_val_batch.device
    B, N_val, d = X_val_batch.shape

    y_val = y_val.to(device=device, dtype=torch.int64)
    pos = (y_val == 1)
    neg = (y_val == -1)

    n1 = max(1, int(pos.sum().item()))
    n2 = max(1, int(neg.sum().item()))

    model_pos.eval()
    model_neg.eval()

    with torch.no_grad():
        s1_all = model_pos(X_val_batch).squeeze(-1)
        s2_all = model_neg(X_val_batch).squeeze(-1)

        Xno1 = _sample_uniform_batch(fmin_batch, fmax_batch, n1)
        Xno2 = _sample_uniform_batch(fmin_batch, fmax_batch, n2)

        s1_no = model_pos(Xno1).squeeze(-1)
        s2_no = model_neg(Xno2).squeeze(-1)

        q = torch.linspace(0.05, 0.95, steps=n_grid, device=device, dtype=torch.float32)

        merged1 = torch.cat([s1_all[:, pos], s1_all[:, neg], s1_no], dim=1)
        merged2 = torch.cat([s2_all[:, neg], s2_all[:, pos], s2_no], dim=1)

        pool1 = torch.quantile(merged1, q, dim=1).transpose(0, 1).contiguous()  # [B, Q]
        pool2 = torch.quantile(merged2, q, dim=1).transpose(0, 1).contiguous()  # [B, Q]

        m1 = s1_all.unsqueeze(-1) >= pool1.unsqueeze(1)   # [B, N, Q]
        m2 = s2_all.unsqueeze(-1) >= pool2.unsqueeze(1)   # [B, N, Q]

        pos_mask = pos.view(1, N_val, 1, 1)
        neg_mask = neg.view(1, N_val, 1, 1)

        m1e = m1.unsqueeze(-1)  # [B, N, Q, 1]
        m2e = m2.unsqueeze(-2)  # [B, N, 1, Q]

        n02_1 = ((m1e & ~m2e) & pos_mask).sum(dim=1).to(dtype=torch.float32)
        n02_2 = ((m2e & ~m1e) & neg_mask).sum(dim=1).to(dtype=torch.float32)

        a1 = n02_1 / n1
        a2 = n02_2 / n2
        F12 = 2.0 * a1 * a2 / (a1 + a2 + H_TAU)  # [B, Q, Q]

        n12_1 = ((m1e & m2e) & pos_mask).sum(dim=1).to(dtype=torch.float32)
        n12_2 = ((m2e & m1e) & neg_mask).sum(dim=1).to(dtype=torch.float32)

        b1 = n12_1 / n1
        b2 = n12_2 / n2
        G12 = 2.0 * b1 * b2 / (b1 + b2 + H_TAU)  # [B, Q, Q]
        #G12 = (b1 + b2) / 2
        #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2

        S = F12 - float(cG) * G12          # [B, Q, Q]
        S_flat = S.reshape(B, -1)

        best_flat_idx = S_flat.argmax(dim=1)

        Q = int(pool1.shape[1])
        q1_idx = best_flat_idx // Q
        q2_idx = best_flat_idx % Q

        rows = torch.arange(B, device=device)

        beta_pos_best = pool1[rows, q1_idx]
        beta_neg_best = pool2[rows, q2_idx]
        S_best = S_flat[rows, best_flat_idx]

    return beta_pos_best, beta_neg_best, S_best

def _batched_train_with_checkpoints(
    X_pos_tr_batch: torch.Tensor,
    X_neg_tr_batch: torch.Tensor,
    X_par_batch: torch.Tensor,
    y_par: torch.Tensor,
    fmin_batch: torch.Tensor,
    fmax_batch: torch.Tensor,
    d_hidden: int,
    n_hidden_layers: int,
    num_epochs: int,
    batch_size: int = 128,
    checkpoint_every: int = 3,
    checkpoint_n_grid: int = 19,
    cG: float = 1.0,
    lr: float = 3e-3,
    weight_decay: float = 1e-2,
    seed: int = 0,
) -> dict:
    """
    Совместно обучает batched positive- и negative-модели.
    Каждые checkpoint_every эпох:
      - на D_par заново подбирает beta_pos, beta_neg, cG,
      - считает S_par = F12 - cG * G12,
      - сохраняет лучший checkpoint ДЛЯ КАЖДОЙ ПАРЫ моделей.

    Возвращает словарь с лучшими state'ами, beta/cG, epoch и метриками на D_par.
    """

    if X_pos_tr_batch.ndim != 3:
        raise ValueError(f"X_pos_tr_batch must be 3D, got shape={tuple(X_pos_tr_batch.shape)}")
    if X_neg_tr_batch.ndim != 3:
        raise ValueError(f"X_neg_tr_batch must be 3D, got shape={tuple(X_neg_tr_batch.shape)}")
    if X_par_batch.ndim != 3:
        raise ValueError(f"X_par_batch must be 3D, got shape={tuple(X_par_batch.shape)}")

    B, N_pos, d = X_pos_tr_batch.shape
    B2, N_neg, d2 = X_neg_tr_batch.shape
    B3, N_par, d3 = X_par_batch.shape

    if B != B2 or B != B3:
        raise ValueError(
            f"Batch size mismatch: pos={B}, neg={B2}, par={B3}"
        )
    if d != d2 or d != d3:
        raise ValueError(
            f"Subspace dim mismatch: pos={d}, neg={d2}, par={d3}"
        )
    if N_pos <= 0 or N_neg <= 0:
        raise ValueError("Empty class encountered in _batched_train_with_checkpoints")

    checkpoint_every = max(1, int(checkpoint_every))

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = X_pos_tr_batch.device
    dtype = X_pos_tr_batch.dtype

    model_pos = BatchedMLP(
        num_models=B,
        d_in=d,
        d_hidden=d_hidden,
        n_hidden_layers=n_hidden_layers,
        device=device,
        dtype=dtype,
    )

    model_neg = BatchedMLP(
        num_models=B,
        d_in=d,
        d_hidden=d_hidden,
        n_hidden_layers=n_hidden_layers,
        device=device,
        dtype=dtype,
    )

    opt_pos = torch.optim.AdamW(
        model_pos.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    opt_neg = torch.optim.AdamW(
        model_neg.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    fmin_batch = fmin_batch.to(device=device, dtype=dtype)
    fmax_batch = fmax_batch.to(device=device, dtype=dtype)
    y_par = y_par.to(device=device, dtype=torch.int64)

    fmin_e = fmin_batch.unsqueeze(1)                      # [B, 1, d]
    span_e = (fmax_batch - fmin_batch).unsqueeze(1)      # [B, 1, d]

    best_S_par = torch.full((B,), -float("inf"), device=device, dtype=torch.float32)
    best_beta_pos = torch.zeros(B, device=device, dtype=torch.float32)
    best_beta_neg = torch.zeros(B, device=device, dtype=torch.float32)
    best_F12_par = torch.zeros(B, device=device, dtype=torch.float32)
    best_G12_par = torch.zeros(B, device=device, dtype=torch.float32)
    best_epoch = torch.zeros(B, device=device, dtype=torch.int64)

    best_pos_states: list[dict | None] = [None] * B
    best_neg_states: list[dict | None] = [None] * B

    def _train_one_epoch_one_side(
        model: BatchedMLP,
        optimizer: torch.optim.Optimizer,
        X_real_batch: torch.Tensor,
        seed_epoch: int,
    ) -> None:
        B_loc, N_real, d_loc = X_real_batch.shape

        gen = torch.Generator(device=device)
        gen.manual_seed(int(seed_epoch))

        perm = torch.argsort(
            torch.rand(B_loc, N_real, generator=gen, device=device, dtype=dtype),
            dim=1
        )  # [B, N_real]

        model.train()

        for start in range(0, N_real, batch_size):
            idx = perm[:, start:start + batch_size]                       # [B, bs]
            cur_bs = int(idx.shape[1])

            idx_ex = idx.unsqueeze(-1).expand(-1, -1, d_loc)             # [B, bs, d]
            xb_real = torch.gather(X_real_batch, dim=1, index=idx_ex)    # [B, bs, d]

            u = torch.rand(B_loc, cur_bs, d_loc, generator=gen, device=device, dtype=dtype)
            xb_noise = fmin_e.expand(-1, cur_bs, -1) + u * span_e.expand(-1, cur_bs, -1)

            xb = torch.cat([xb_real, xb_noise], dim=1)                   # [B, 2*bs, d]
            yb = torch.cat(
                [
                    torch.ones(B_loc, cur_bs, 1, device=device, dtype=dtype),
                    torch.zeros(B_loc, cur_bs, 1, device=device, dtype=dtype),
                ],
                dim=1,
            )                                                             # [B, 2*bs, 1]

            pred = model(xb)                                              # [B, 2*bs, 1]
            loss_per_model = ((pred - yb) ** 2).mean(dim=(1, 2))          # [B]
            loss = loss_per_model.sum()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

    for epoch in range(1, num_epochs + 1):
        _train_one_epoch_one_side(
            model=model_pos,
            optimizer=opt_pos,
            X_real_batch=X_pos_tr_batch,
            seed_epoch=seed + 100000 + epoch,
        )

        _train_one_epoch_one_side(
            model=model_neg,
            optimizer=opt_neg,
            X_real_batch=X_neg_tr_batch,
            seed_epoch=seed + 200000 + epoch,
        )

        do_checkpoint = (epoch % checkpoint_every == 0) or (epoch == num_epochs)
        if not do_checkpoint:
            continue

        beta_pos_cur, beta_neg_cur, _ = _batched_tune_betas_joint(
            model_pos=model_pos,
            model_neg=model_neg,
            X_val_batch=X_par_batch,
            y_val=y_par,
            fmin_batch=fmin_batch,
            fmax_batch=fmax_batch,
            n_grid=checkpoint_n_grid,
            cG=cG,
        )

        model_pos.eval()
        model_neg.eval()

        with torch.no_grad():
            s1_par = model_pos(X_par_batch).squeeze(-1)   # [B, N_par]
            s2_par = model_neg(X_par_batch).squeeze(-1)   # [B, N_par]

        m_par = _batched_metrics_from_scores(
            s1=s1_par,
            s2=s2_par,
            y=y_par,
            beta_pos=beta_pos_cur,
            beta_neg=beta_neg_cur,
        )

        F12_par_cur = m_par["F12"]
        G12_par_cur = m_par["G12"]
        S_par_cur = F12_par_cur - cG * G12_par_cur

        improved = S_par_cur > best_S_par
        improved_idx = torch.where(improved)[0].tolist()

        if len(improved_idx) > 0:
            for b in improved_idx:
                best_pos_states[b] = model_pos.get_model_state(b)
                best_neg_states[b] = model_neg.get_model_state(b)

            best_S_par[improved] = S_par_cur[improved]
            best_beta_pos[improved] = beta_pos_cur[improved]
            best_beta_neg[improved] = beta_neg_cur[improved]
            best_F12_par[improved] = F12_par_cur[improved]
            best_G12_par[improved] = G12_par_cur[improved]
            best_epoch[improved] = int(epoch)

    # safety: если по какой-то причине checkpoint ни разу не сработал
    for b in range(B):
        if best_pos_states[b] is None:
            best_pos_states[b] = model_pos.get_model_state(b)
        if best_neg_states[b] is None:
            best_neg_states[b] = model_neg.get_model_state(b)

    return {
        "best_pos_states": best_pos_states,
        "best_neg_states": best_neg_states,
        "best_beta_pos": best_beta_pos.detach().cpu().numpy(),
        "best_beta_neg": best_beta_neg.detach().cpu().numpy(),
        "best_S_par": best_S_par.detach().cpu().numpy(),
        "best_F12_par": best_F12_par.detach().cpu().numpy(),
        "best_G12_par": best_G12_par.detach().cpu().numpy(),
        "best_epoch": best_epoch.detach().cpu().numpy(),
    }

In [19]:
class PackedShapBatchMLP(nn.Module):
    """
    Объединяет B независимых MLP одинаковой архитектуры и одинаковой входной
    размерности d в одну multi-output модель:

        input:  [N, B*d]
        output: [N, B]

    b-й выход зависит только от b-го блока признаков длины d.
    """

    def __init__(
        self,
        weights: list[torch.Tensor],
        biases: list[torch.Tensor],
        d_in: int,
        device: torch.device,
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()
        self.d_in = int(d_in)
        self.num_models = int(weights[0].shape[0])

        self.weights = nn.ParameterList(
            [nn.Parameter(w.to(device=device, dtype=dtype), requires_grad=False) for w in weights]
        )
        self.biases = nn.ParameterList(
            [nn.Parameter(b.to(device=device, dtype=dtype), requires_grad=False) for b in biases]
        )

    @classmethod
    def from_models(
        cls,
        models: list[nn.Module],
        device: torch.device,
        dtype: torch.dtype = torch.float32,
    ) -> "PackedShapBatchMLP":
        if len(models) == 0:
            raise ValueError("from_models: empty models list")

        linear_layers_all = []
        for m in models:
            linear_layers = [layer for layer in m.net if isinstance(layer, nn.Linear)]
            if len(linear_layers) == 0:
                raise ValueError("from_models: model has no Linear layers")
            linear_layers_all.append(linear_layers)

        n_layers = len(linear_layers_all[0])
        if any(len(ll) != n_layers for ll in linear_layers_all):
            raise ValueError("from_models: inconsistent number of Linear layers")

        d_in = int(linear_layers_all[0][0].in_features)

        weights: list[torch.Tensor] = []
        biases: list[torch.Tensor] = []

        for li in range(n_layers):
            w_stack = torch.stack(
                [linear_layers_all[b][li].weight.detach() for b in range(len(models))],
                dim=0
            )  # [B, out, in]
            b_stack = torch.stack(
                [linear_layers_all[b][li].bias.detach() for b in range(len(models))],
                dim=0
            )  # [B, out]
            weights.append(w_stack)
            biases.append(b_stack)

        return cls(weights=weights, biases=biases, d_in=d_in, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [N, B*d]
        return: [N, B]
        """
        if x.ndim != 2:
            raise ValueError(f"PackedShapBatchMLP expects [N, B*d], got shape={tuple(x.shape)}")

        N, Dtot = x.shape
        if Dtot != self.num_models * self.d_in:
            raise ValueError(
                f"PackedShapBatchMLP: expected input dim {self.num_models * self.d_in}, got {Dtot}"
            )

        x = x.view(N, self.num_models, self.d_in).transpose(0, 1).contiguous()  # [B, N, d]

        for li, (w, b) in enumerate(zip(self.weights, self.biases)):
            x = torch.einsum("bni,boi->bno", x, w) + b.unsqueeze(1)
            if li < len(self.weights) - 1:
                x = torch.abs(x)

        x = x.squeeze(-1).transpose(0, 1).contiguous()  # [N, B]
        return x


def _pack_inputs_for_shap(X_source: torch.Tensor, subs_list: list[np.ndarray]) -> torch.Tensor:
    """
    X_source: [N, p]
    subs_list: list of arrays, each shape [d]
    return: [N, B*d]
    """
    blocks = []
    for subs in subs_list:
        subs_t = torch.as_tensor(subs, dtype=torch.long, device=X_source.device)
        blocks.append(X_source[:, subs_t])
    return torch.cat(blocks, dim=1)


def _normalize_shap_multioutput(vals, n_outputs: int, n_samples: int, total_features: int) -> np.ndarray:
    """
    Приводит shap output к ndarray shape [B, N, F].
    Поддерживает основные форматы, которые SHAP может вернуть для multi-output.
    """
    if isinstance(vals, list):
        arr = np.stack([np.asarray(v, dtype=float) for v in vals], axis=0)  # [B, N, F]
        return arr

    arr = np.asarray(vals, dtype=float)

    if arr.ndim == 2:
        # [N, F] — допустимо только для single-output
        if n_outputs != 1:
            raise ValueError(f"Expected multi-output SHAP, got 2D array with n_outputs={n_outputs}")
        return arr[None, :, :]

    if arr.ndim != 3:
        raise ValueError(f"Unsupported SHAP output shape: {arr.shape}")

    # Возможные варианты:
    # [B, N, F]
    if arr.shape[0] == n_outputs and arr.shape[1] == n_samples and arr.shape[2] == total_features:
        return arr

    # [N, F, B]
    if arr.shape[0] == n_samples and arr.shape[1] == total_features and arr.shape[2] == n_outputs:
        return np.transpose(arr, (2, 0, 1))

    # [N, B, F]
    if arr.shape[0] == n_samples and arr.shape[1] == n_outputs and arr.shape[2] == total_features:
        return np.transpose(arr, (1, 0, 2))

    raise ValueError(f"Cannot normalize SHAP output shape {arr.shape}")


def _diag_block_positive_means(vals_norm: np.ndarray, block_dim: int) -> np.ndarray:
    """
    vals_norm: [B, N, B*d_total_per_modelblock]
    Берёт для b-го выхода только его собственный блок признаков [b*d : (b+1)*d]
    и считает mean(max(shap, 0)) по объектам.

    return: [B, d]
    """
    B, N, F = vals_norm.shape
    if F != B * block_dim:
        raise ValueError(f"Expected total_features = B*block_dim = {B*block_dim}, got {F}")

    out = np.zeros((B, block_dim), dtype=float)
    for b in range(B):
        sl = slice(b * block_dim, (b + 1) * block_dim)
        block_vals = vals_norm[b, :, sl]         # [N, d]
        out[b] = np.maximum(block_vals, 0.0).mean(axis=0)
    return out


def _batched_gradient_shap_for_group(
    models: list[nn.Module],
    X_bg_pack: torch.Tensor,
    X_eval_pack: torch.Tensor,
    device: torch.device,
) -> np.ndarray:
    """
    Возвращает positive mean SHAP для каждого выхода/модели в группе.

    models: list of B ordinary MLP
    X_bg_pack:   [N_bg, B*d]
    X_eval_pack: [N_ev, B*d]

    return: [B, d]
    """
    if len(models) == 0:
        raise ValueError("_batched_gradient_shap_for_group: empty models")

    B = len(models)
    d = int(X_bg_pack.shape[1] // B)

    wrapper = PackedShapBatchMLP.from_models(models, device=device, dtype=X_bg_pack.dtype).eval()

    expl = shap.GradientExplainer(wrapper, X_bg_pack)
    vals = expl.shap_values(X_eval_pack)

    vals_norm = _normalize_shap_multioutput(
        vals=vals,
        n_outputs=B,
        n_samples=int(X_eval_pack.shape[0]),
        total_features=int(X_eval_pack.shape[1]),
    )  # [B, N, B*d]

    return _diag_block_positive_means(vals_norm, block_dim=d)  # [B, d]

In [286]:
H_TAU = 1e-3


def tune_betas_joint(
    model_pos: torch.nn.Module,
    model_neg: torch.nn.Module,
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    fmin: torch.Tensor,
    fmax: torch.Tensor,
    n_grid: int = 19,
    cG: float = 1.0,
) -> Tuple[float, float, float, float]:
    """
    Совместно подбирает (beta_pos, beta_neg) и cG, максимизируя
    S = F12 - cG * G12 на валидации.
    Возвращает: (beta_pos, beta_neg, best_cG, best_S).
    """

    device = next(model_pos.parameters()).device
    Xv = X_val.to(device=device, dtype=torch.float32)
    yv = y_val.to(device=device, dtype=torch.int64)

    model_pos.eval()
    model_neg.eval()

    with torch.no_grad():
        s1_all = model_pos(Xv).flatten()
        s2_all = model_neg(Xv).flatten()
        pos = (yv == 1)
        neg = (yv == -1)

        n1 = max(1, int(pos.sum().item()))
        n2 = max(1, int(neg.sum().item()))

        Xno1 = sample_uniform(fmin.to(device), fmax.to(device), n1)
        Xno2 = sample_uniform(fmin.to(device), fmax.to(device), n2)
        s1_no = model_pos(Xno1).flatten()
        s2_no = model_neg(Xno2).flatten()

        q = torch.linspace(0.05, 0.95, steps=n_grid, device=device)
        pool1 = torch.quantile(torch.cat([s1_all[pos], s1_all[neg], s1_no]), q)
        pool2 = torch.quantile(torch.cat([s2_all[neg], s2_all[pos], s2_no]), q)

        n1 = max(1, int(pos.sum().item()))
        n2 = max(1, int(neg.sum().item()))

    best_S = -float("inf")
    best_bp = 0.0
    best_bn = 0.0

    for bpos_t in pool1:
        bpos = float(bpos_t.item())
        m1 = (s1_all >= bpos)

        for bneg_t in pool2:
            bneg = float(bneg_t.item())
            m2 = (s2_all >= bneg)

            n02_1 = int(((m1 & ~m2) & pos).sum().item())
            n02_2 = int(((m2 & ~m1) & neg).sum().item())
            a1 = n02_1 / n1
            a2 = n02_2 / n2
            F12 = 2 * a1 * a2 / (a1 + a2 + H_TAU)

            n12_1 = int(((m1 & m2) & pos).sum().item())
            n12_2 = int(((m2 & m1) & neg).sum().item())
            b1 = n12_1 / n1
            b2 = n12_2 / n2
            G12 = 2 * b1 * b2 / (b1 + b2 + H_TAU)
            #G12 = (b1 + b2) / 2
            #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2

            S = F12 - float(cG) * G12
            if S > best_S:
                best_S = S
                best_bp = bpos
                best_bn = bneg

    return best_bp, best_bn, best_S

In [265]:
def _clone_estimator(est):
    """
    Безопасное клонирование sklearn-модели.
    Для sklearn лучше clone, но такой вариант тоже работает,
    если модель корректно реализует get_params().
    """
    return est.__class__(**est.get_params())

def _get_score_vector(model, X):
    """
    Возвращает непрерывный score для ROC AUC / PR AUC.
    """
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(model, "decision_function"):
        out = model.decision_function(X)
        return np.asarray(out).ravel()

    raise ValueError(
        f"Model {model.__class__.__name__} has neither predict_proba nor decision_function"
    )

In [266]:
def evaluate_reduction(
    X: np.ndarray,
    y: np.ndarray,
    reducer_name: str,
    reducer_ctor: Any,
    q: int,
    dataset_name: str,
    n_splits: int = 5,
    reducer_kwargs: Optional[Dict[str, Any]] = None,
    compute_unary_on_reduced: bool = True,
    unary_params: Optional[Dict[str, Any]] = None,
    results_dir: Optional[str | Path] = None,
    model_seed: int = 42,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    reducer_kwargs = reducer_kwargs or {}
    unary_params = unary_params or dict(
        d_hidden=48,
        n_hidden_layers=2,
        num_epochs=60,
        batch_size=256,
        beta_grid=31,
        cG=1.0,
    )

    save_root = Path(results_dir) if results_dir is not None else None
    if save_root is not None:
        save_root.mkdir(parents=True, exist_ok=True)

    y_orig = np.asarray(y).ravel()
    if set(np.unique(y_orig)) == {-1, 1}:
        y_bin = (y_orig == 1).astype(int)
    else:
        y_bin = y_orig.astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    rows_cls: List[Dict[str, Any]] = []
    rows_unary: List[Dict[str, Any]] = []

    for fold, (tr, te) in enumerate(skf.split(X, y_bin), 1):
        fold_dir = None
        if save_root is not None:
            fold_dir = save_root / f"fold_{fold:02d}"
            fold_dir.mkdir(parents=True, exist_ok=True)

        Xtr, Xte = X[tr], X[te]
        ytr_bin, yte_bin = y_bin[tr], y_bin[te]
        ytr_orig, yte_orig = y_orig[tr], y_orig[te]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)

        models = get_models(seed=model_seed + fold)

        # ---------- baseline on full feature space ----------
        base_scores: Dict[str, Tuple[float, float, float]] = {}
        for mname, base_model in models.items():
            m0 = _clone_estimator(base_model)
            t0 = perf_counter()
            m0.fit(Xtr_s, ytr_bin)
            t0 = perf_counter() - t0

            p0 = _get_score_vector(m0, Xte_s)
            auc0 = roc_auc_score(yte_bin, p0)
            ap0 = average_precision_score(yte_bin, p0)
            base_scores[mname] = (auc0, ap0, t0)

        # ---------- reducer ----------
        try:
            red = reducer_ctor(q=q, **reducer_kwargs)
        except TypeError:
            red = reducer_ctor(n_select=q, **reducer_kwargs)

        tR = perf_counter()
        Ztr = red.fit_transform(Xtr_s, ytr_orig)
        Zte = red.transform(Xte_s)
        tR = perf_counter() - tR

        selected_indices = None
        if hasattr(red, "selected_indices_") and red.selected_indices_ is not None:
            selected_indices = np.asarray(red.selected_indices_, dtype=int).tolist()

        if fold_dir is not None:
            fold_meta = {
                "fold": fold,
                "dataset": dataset_name,
                "reducer_name": reducer_name,
                "q": int(q),
                "train_indices": tr.tolist(),
                "test_indices": te.tolist(),
                "selected_indices": selected_indices,
                "train_shape_before": list(Xtr.shape),
                "test_shape_before": list(Xte.shape),
                "train_shape_after": list(Ztr.shape),
                "test_shape_after": list(Zte.shape),
                "t_reducer": float(tR),
            }
            with open(fold_dir / "fold_meta.json", "w", encoding="utf-8") as f:
                json.dump(fold_meta, f, ensure_ascii=False, indent=2)

        # ---------- classical post-models ----------
        fold_cls_rows = []
        for mname, base_model in models.items():
            auc0, ap0, t0 = base_scores[mname]

            m1 = _clone_estimator(base_model)
            t1 = perf_counter()
            m1.fit(Ztr, ytr_bin)
            t1 = perf_counter() - t1

            p1 = _get_score_vector(m1, Zte)
            auc1 = roc_auc_score(yte_bin, p1)
            ap1 = average_precision_score(yte_bin, p1)

            row = {
                "dataset": dataset_name,
                "method": reducer_name,
                "q": q,
                "model": mname,
                "fold": fold,
                "AUC": auc1,
                "PR_AUC": ap1,
                "ΔAUC": auc1 - auc0,
                "ΔPR_AUC": ap1 - ap0,
                "AUC_base": auc0,
                "PR_AUC_base": ap0,
                "t_reducer": tR,
                "t_model": t1,
                "t_model_base": t0,
                "n_selected": None if selected_indices is None else len(selected_indices),
                "selected_indices": None if selected_indices is None else ",".join(map(str, selected_indices)),
            }

            rows_cls.append(row)
            fold_cls_rows.append(row)

        if fold_dir is not None and len(fold_cls_rows) > 0:
            pd.DataFrame(fold_cls_rows).to_csv(
                fold_dir / "classical_metrics.csv",
                index=False,
            )
            if selected_indices is not None:
                with open(fold_dir / "selected_indices.json", "w", encoding="utf-8") as f:
                    json.dump(
                        {"selected_indices": selected_indices},
                        f,
                        ensure_ascii=False,
                        indent=2,
                    )

        # ---------- unary on reduced ----------
        if compute_unary_on_reduced:
            dev_param = unary_params.get("device", None)
            if dev_param is not None:
                unary_device_preferred = torch.device(dev_param)
            else:
                if torch.cuda.is_available():
                    unary_device_preferred = torch.device("cuda")
                elif torch.backends.mps.is_available():
                    unary_device_preferred = torch.device("mps")
                else:
                    unary_device_preferred = torch.device("cpu")

            ytr_pm = np.where(ytr_orig == 1, 1, -1)
            yte_pm = np.where(yte_orig == 1, 1, -1)

            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
            (fit_idx, val_idx), = sss.split(Ztr, (ytr_pm == 1).astype(int))

            Ztr_fit, Ztr_val = Ztr[fit_idx], Ztr[val_idx]
            y_fit_pm, y_val_pm = ytr_pm[fit_idx], ytr_pm[val_idx]

            X_fit_t = torch.as_tensor(Ztr_fit, dtype=torch.float32, device=unary_device_preferred)
            y_fit_t = torch.as_tensor(y_fit_pm, dtype=torch.int64, device=unary_device_preferred)
            train_ds_fit = TensorDataset(X_fit_t, y_fit_t)

            model_pos, model_neg = model_train(
                d_hidden=unary_params.get("d_hidden", 32),
                n_hidden_layers=unary_params.get("n_hidden_layers", 1),
                train_data=train_ds_fit,
                num_epochs=unary_params.get("num_epochs", 50),
                batch_size=unary_params.get("batch_size", 256),
                seed=unary_params.get("seed", 0),
                output=False,
            )
            model_pos.eval()
            model_neg.eval()

            unary_device = next(model_pos.parameters()).device

            X_val_t = torch.as_tensor(Ztr_val, dtype=torch.float32, device=unary_device)
            y_val_t = torch.as_tensor(y_val_pm, dtype=torch.int64, device=unary_device)

            fmin, fmax = compute_feature_bounds(TensorDataset(X_fit_t, y_fit_t))
            fmin = fmin.to(unary_device)
            fmax = fmax.to(unary_device)

            beta_pos, beta_neg, S_val = tune_betas_joint(
                model_pos,
                model_neg,
                X_val_t,
                y_val_t,
                fmin,
                fmax,
                n_grid=unary_params.get("beta_grid", 19),
                cG=unary_params.get("cG", 1.0),
            )

            X_te_t = torch.as_tensor(Zte, dtype=torch.float32, device=unary_device)
            yte_pm_t = torch.as_tensor(yte_pm, dtype=torch.int64, device=unary_device)

            m_un = metrics_from_sheet(
                X_te_t,
                yte_pm_t,
                model_pos,
                model_neg,
                beta_pos=float(beta_pos),
                beta_neg=float(beta_neg),
            )

            with torch.no_grad():
                s1_te = model_pos(X_te_t).flatten()
                s2_te = model_neg(X_te_t).flatten()

            pos_hit = (s1_te >= float(beta_pos)) & (s2_te < float(beta_neg))
            neg_hit = (s2_te >= float(beta_neg)) & (s1_te < float(beta_pos))
            coverage = float((pos_hit | neg_hit).float().mean().item())

            S_test = float(m_un["F12"] - float(unary_params.get("cG", 1.0)) * m_un["G12"])

            unary_row = {
                "dataset": dataset_name,
                "method": reducer_name,
                "q": q,
                "model": "UnaryMLP",
                "fold": fold,
                "F12": float(m_un["F12"]),
                "G12": float(m_un["G12"]),
                "a1": float(m_un["a1"]),
                "a2": float(m_un["a2"]),
                "b1": float(m_un["b1"]),
                "b2": float(m_un["b2"]),
                "n1": int(m_un["n1"]),
                "n2": int(m_un["n2"]),
                "coverage": coverage,
                "beta_pos": float(beta_pos),
                "beta_neg": float(beta_neg),
                "S_val": float(S_val),
                "S_test": S_test,
                "t_reducer": tR,
                "t_model": 0.0,
                "unary_device": str(unary_device),
                "n_selected": None if selected_indices is None else len(selected_indices),
                "selected_indices": None if selected_indices is None else ",".join(map(str, selected_indices)),
            }

            rows_unary.append(unary_row)

            if fold_dir is not None:
                with open(fold_dir / "unary_metrics.json", "w", encoding="utf-8") as f:
                    json.dump(unary_row, f, ensure_ascii=False, indent=2)

    if save_root is not None:
        if len(rows_cls) > 0:
            pd.DataFrame(rows_cls).to_csv(save_root / "all_classical_folds.csv", index=False)
        if len(rows_unary) > 0:
            pd.DataFrame(rows_unary).to_csv(save_root / "all_unary_folds.csv", index=False)

    return rows_cls, rows_unary

In [30]:
def run_all_A(
    include_unary: bool = False,
    unary_params: dict | None = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    configs = [
        ("DS-A", make_dataset_A, [2, 3, 5]),
        # ("DS-B", make_dataset_B, [5, 10, 15]),
    ]

    methods = [
        ("PCA",        lambda q, **_: PCAReducer(q)),
        ("PLS",        lambda q, **_: PLSReducer(q)),
        ("UMAP_sup",   lambda q, **_: UMAPReducer(q)),
        ("HSIC_Lasso", lambda q, **_: HSICSelector(q)),
        ("MLP_unar",   lambda q, **kw: MLPReducer(
            d_hidden=64,
            n_hidden_layers=2,
            num_epochs=40,
            batch_size=256,
            n_select=q,
            seed=0,
            output=False,
            **kw,   
        )),
    ]


    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    else:
        DEVICE = torch.device("cpu")

    print("Using device:", DEVICE)
    if DEVICE.type == "cuda":
        print("CUDA name:", torch.cuda.get_device_name(0))

    base_unary_params = dict(
        d_hidden=32,
        n_hidden_layers=1,
        num_epochs=50,
        batch_size=256,
    )
    if unary_params is not None:
        base_unary_params.update(unary_params)
    base_unary_params["device"] = str(DEVICE)  

    rows_cls_all: List[Dict[str, Any]] = []

    for dname, maker, qgrid in configs:
        X, y = maker()
        for name, ctor in methods:
            for q in qgrid:
                if name == "MLP_unar":
                    reducer_kwargs = {
                        "device": str(DEVICE),  
                    }
                else:
                    reducer_kwargs = {}

                cls_rows, _un_rows = evaluate_reduction(
                    X, y,
                    reducer_name=name,
                    reducer_ctor=ctor,
                    q=q,
                    dataset_name=dname,
                    reducer_kwargs=reducer_kwargs,
                    compute_unary_on_reduced=include_unary,
                    unary_params=base_unary_params,
                )
                rows_cls_all.extend(cls_rows)


    df = pd.DataFrame(rows_cls_all)

    for col in ["AUC", "PR_AUC", "ΔAUC", "ΔPR_AUC",
                "t_reducer", "t_model", "t_model_base"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    agg = (
        df.groupby(["dataset", "method", "q", "model"], dropna=False)
          .agg(
              AUC_mean=("AUC", "mean"),
              PR_AUC_mean=("PR_AUC", "mean"),
              dAUC_mean=("ΔAUC", "mean"),
              dPR_mean=("ΔPR_AUC", "mean"),
              t_red_med=("t_reducer", "median"),
              t_mod_med=("t_model", "median"),
          )
          .reset_index()
          .sort_values(["dataset", "model", "dAUC_mean"], ascending=[True, True, False])
    )

    return df, agg


In [31]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.2.2+cu118
11.8
True


In [33]:
torch.cuda.get_device_name(0)


'NVIDIA GeForce GTX 1080'

In [ ]:
df_A, agg_A = run_all_A()

In [22]:
def dynamic_bounds(
    n: int,
    q: int,
    A_idx: np.ndarray,
    step: int,
    n_steps: int,
    # upper caps
    uA_mult_start: float = 1.10,   # U_A raw = uA_mult / q
    uA_mult_end: float = 1.25,
    u0_mult_start: float = 0.90,   # U_0 = u0_mult / q
    u0_mult_end: float = 1.05,
    # lower masses
    mass_A_start: float = 0.10,
    mass_A_end: float = 0.22,
    mass_0_start: float = 0.12,
    mass_0_end: float = 0.08,
    # explicit outside reserve
    outside_reserve_start: float = 0.30,
    outside_reserve_end: float = 0.25,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Возвращает покоординатные lower/upper bounds для проекции:
      - для j in Ahat:     lb_j = L_A, ub_j = U_A
      - для j not in Ahat: lb_j = L_0, ub_j = U_0

    Дополнительно enforce-ится минимальная внешняя масса:
        sum_{j not in Ahat} pi_j >= reserve_out,
    что эквивалентно
        sum_{j in Ahat} pi_j <= 1 - reserve_out.
    """
    assert n >= 1
    assert 1 <= q <= n

    A = np.unique(np.asarray(A_idx, dtype=int).reshape(-1))
    A = A[(A >= 0) & (A < n)]
    m = int(A.size)
    n_out = n - m

    progress = 0.0 if n_steps <= 1 else step / (n_steps - 1)

    # ---------- explicit outside reserve ----------
    reserve_out = outside_reserve_start + (outside_reserve_end - outside_reserve_start) * progress
    reserve_out = float(np.clip(reserve_out, 0.0, 0.95))

    # ---------- raw upper caps ----------
    U_A_raw = (uA_mult_start + (uA_mult_end - uA_mult_start) * progress) / max(q, 1)
    U_0 = (u0_mult_start + (u0_mult_end - u0_mult_start) * progress) / max(q, 1)

    U_A_raw = float(np.clip(U_A_raw, 1.0 / n + 1e-9, 1.0))
    U_0 = float(np.clip(U_0, 1.0 / n + 1e-9, 1.0))

    # Чтобы outside reserve был достижим, вне Ahat должна быть достаточная capacity
    if n_out > 0:
        U_0 = max(U_0, reserve_out / n_out + 1e-9)
        U_0 = float(np.clip(U_0, 1.0 / n + 1e-9, 1.0))

    # Cap на anchors, чтобы внутри Ahat не утаскивалась вся масса
    if m > 0:
        U_A_reserve = (1.0 - reserve_out) / m
        U_A = min(U_A_raw, U_A_reserve)
        U_A = float(np.clip(U_A, 1.0 / n + 1e-9, 1.0))
    else:
        U_A = U_A_raw

    # ---------- lower masses ----------
    mass_A = 0.0 if m == 0 else mass_A_start + (mass_A_end - mass_A_start) * progress
    mass_A = float(np.clip(mass_A, 0.0, 0.75))

    mass_0 = mass_0_start + (mass_0_end - mass_0_start) * progress
    mass_0 = float(np.clip(mass_0, 0.0, 0.50))

    L_A = (mass_A / m) if m > 0 else 0.0
    L_0 = (mass_0 / n_out) if n_out > 0 else 0.0

    # lower bounds должны быть существенно ниже соответствующих upper bounds
    if m > 0:
        L_A = min(L_A, 0.85 * U_A)
    if n_out > 0:
        L_0 = min(L_0, 0.50 * U_0)

    # Anchor-floor не должен сам по себе убивать outside reserve
    if m > 0:
        max_anchor_mass = max(0.0, 1.0 - reserve_out - 1e-6)
        if m * L_A > max_anchor_mass:
            L_A = max_anchor_mass / m

    # Общая feasibility по lower bounds
    total_lb = m * L_A + n_out * L_0
    if total_lb >= 1.0 - 1e-8:
        scale = (1.0 - 1e-6) / total_lb
        L_A *= scale
        L_0 *= scale

    lb = np.full(n, L_0, dtype=float)
    ub = np.full(n, U_0, dtype=float)

    if m > 0:
        lb[A] = L_A
        ub[A] = U_A

    # safety: coordinatewise feasibility
    bad = lb > ub
    if np.any(bad):
        lb[bad] = np.minimum(lb[bad], ub[bad] - 1e-9)
        lb = np.maximum(lb, 0.0)

    info = {
        "U_A": float(U_A),
        "U0": float(U_0),
        "L_A": float(L_A),
        "L0": float(L_0),
        "anchor_size": int(m),
        "mass_A_lb": float(m * L_A),
        "mass_0_lb": float(n_out * L_0),
        "reserve_out": float(reserve_out),
        "anchor_cap_total": float(m * U_A) if m > 0 else 0.0,
    }
    return lb, ub, info

In [23]:
H_TAU = 1e-3
def _h_tau(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return 2.0 * x * y / (x + y + H_TAU)

In [24]:
def _expand_anchor_set(
    scores: np.ndarray,
    A_prev: np.ndarray,
    q: int,
    step: int,
    n_steps: int,
    min_mult_uniform = 1.1,
    add_frac_start = 0.15,
    add_frac_end   = 0.25,
    cap_mult_start = 1.6,
    cap_mult_end   = 1.3
) -> np.ndarray:
    """
    Накопительно расширяет anchor-set Ahat из текущего step-specific score-вектора.
    Важно: сюда следует подавать score текущего шага (например coord_new_prob),
    а не уже сглаженный/projection-aware вектор.

    Добавляет top новых координат, но ограничивает общий размер Ahat.
    """
    s = np.asarray(scores, dtype=float).reshape(-1)
    n = s.size

    s[~np.isfinite(s)] = 0.0
    s = np.maximum(s, 0.0)

    A_prev = np.unique(np.asarray(A_prev, dtype=int).reshape(-1))
    A_prev = A_prev[(A_prev >= 0) & (A_prev < n)]

    if s.sum() <= 0:
        return A_prev.copy()

    s = s / s.sum()

    progress = 0.0 if n_steps <= 1 else step / (n_steps - 1)

    add_frac = add_frac_start + (add_frac_end - add_frac_start) * progress
    cap_mult = cap_mult_start + (cap_mult_end - cap_mult_start) * progress

    add_count = max(1, int(np.ceil(q * add_frac)))
    cap_size = min(n, max(q, int(np.ceil(q * cap_mult))))

    baseline = min_mult_uniform / max(n, 1)

    current = set(A_prev.tolist())
    ranked = np.argsort(-s)

    # если Ahat ещё пусто — инициализируем top-координатами текущего шага
    if len(current) == 0:
        init_take = min(add_count, cap_size, n)
        return np.sort(ranked[:init_take]).astype(int)

    added = 0
    for idx in ranked:
        idx = int(idx)
        if idx in current:
            continue
        if len(current) >= cap_size:
            break
        if s[idx] < baseline:
            break
        current.add(idx)
        added += 1
        if added >= add_count:
            break

    return np.array(sorted(current), dtype=int)

In [25]:
def _project_box_simplex(
    v: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    tol: float = 1e-10,
    max_iter: int = 200,
) -> np.ndarray:
    """
    Евклидова проекция на множество
        {x : sum_i x_i = 1, lb_i <= x_i <= ub_i}.
    """
    v = np.asarray(v, dtype=float).reshape(-1)
    lb = np.asarray(lb, dtype=float).reshape(-1)
    ub = np.asarray(ub, dtype=float).reshape(-1)

    if not (v.shape == lb.shape == ub.shape):
        raise ValueError("v, lb, ub must have the same shape")
    if np.any(lb > ub):
        raise ValueError("Found lb_i > ub_i")
    if lb.sum() > 1.0 + 1e-12:
        raise ValueError("Infeasible lower bounds: sum(lb) > 1")
    if ub.sum() < 1.0 - 1e-12:
        raise ValueError("Infeasible upper bounds: sum(ub) < 1")

    if abs(lb.sum() - 1.0) <= tol:
        x = lb.copy()
        return x / x.sum()
    if abs(ub.sum() - 1.0) <= tol:
        x = ub.copy()
        return x / x.sum()

    lo = np.min(v - ub)
    hi = np.max(v - lb)

    for _ in range(max_iter):
        tau = 0.5 * (lo + hi)
        x = np.clip(v - tau, lb, ub)
        s = x.sum()

        if s > 1.0:
            lo = tau
        else:
            hi = tau

        if hi - lo <= tol:
            break

    tau = 0.5 * (lo + hi)
    x = np.clip(v - tau, lb, ub)

    s = x.sum()
    if s <= 0 or not np.isfinite(s):
        raise ValueError("Projection failed: non-positive or non-finite sum")

    x /= s
    return x

In [26]:
def _chunk_list(seq, chunk_size: int):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    for start in range(0, len(seq), chunk_size):
        yield seq[start:start + chunk_size]

In [208]:
class EnsembleReducer(ReducerBase):
    def __init__(
        self,
        d_hidden,
        n_hidden_layers,
        num_epochs,
        batch_size,
        n_select,
        num_models,
        num_attempts,
        num_coords,
        num_samples,
        seed=0,
        output=False,
        device: str = "cpu",
        use_gpu_shap: bool = False,
        max_models_per_batch: int = 32,
        max_shap_models_per_batch: int = 16,
        checkpoint_every: int = 3,
        checkpoint_n_grid: int = 19,
        cG: float = 1.0,
    ):
        self.d_hidden = d_hidden
        self.n_hidden_layers = n_hidden_layers
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.seed = seed
        self.output = output
        self.n_select = n_select
        self.num_samples = num_samples
        self.num_models = num_models
        self.num_attempts = num_attempts
        self.num_coords = num_coords
        self.max_models_per_batch = int(max_models_per_batch)
        self.max_shap_models_per_batch = int(max_shap_models_per_batch)

        self.list_model_pos = []
        self.list_model_neg = []
        self.list_model_subs = []
        self.list_beta_pos = []
        self.list_beta_neg = []
        self.list_best_S = []

        self.coord_prob = None
        self.history_ = []
        self.rng = np.random.default_rng(self.seed)
        self.eta_history_ = []
        self.eta_last_ = None
        self.eta_final_ = None
        self.selected_indices_ = None

        self.device = torch.device(device)
        self.use_gpu_shap = use_gpu_shap
        self.checkpoint_every = int(checkpoint_every)
        self.checkpoint_n_grid = int(checkpoint_n_grid)
        self.cG = float(cG)

        self.list_best_epoch = []
        self.list_best_S_par = []
        self.list_best_F12_par = []
        self.list_best_G12_par = []
        self.list_best_F12_val = []
        self.list_best_G12_val = []

    def _to_torch2d(self, X):
        if isinstance(X, torch.Tensor):
            assert X.ndim == 2, "X must be 2D"
            return X.to(dtype=torch.float32)
        X = np.asarray(X)
        assert X.ndim == 2, "X must be 2D"
        return torch.from_numpy(X).to(dtype=torch.float32)

    def _to_torch1d_int(self, y):
        if isinstance(y, torch.Tensor):
            assert y.ndim == 1, "y must be 1D"
            return y.to(dtype=torch.int64)
        y = np.asarray(y).ravel()
        return torch.from_numpy(y.astype(np.int64))

    @staticmethod
    def _shap_mean_pos(expl, X):
        vals = expl.shap_values(X)
        if isinstance(vals, list):
            vals = np.array(vals)
            if vals.ndim == 3:
                vals = vals.mean(axis=0)
        V = np.atleast_2d(np.asarray(vals, float))
        return np.maximum(V, 0.0).mean(axis=0)

    @staticmethod
    def _robust_norm(v: np.ndarray) -> np.ndarray:
        v = np.asarray(v, dtype=float)
        v = np.maximum(v, 0.0)
        if not np.any(np.isfinite(v)):
            return np.full_like(v, 1.0 / max(1, v.size))
        p95 = np.percentile(v[np.isfinite(v)], 95.0)
        if p95 > 0:
            v = np.minimum(v, p95)
        s = v.sum()
        return (v / s) if (np.isfinite(s) and s > 0) else np.full_like(v, 1.0 / max(1, v.size))

    @staticmethod
    def _theory_shap_importance(mu_pos: np.ndarray, mu_neg: np.ndarray, subspace_size: int) -> np.ndarray:
        """
        Практический SHAP-объект, близкий к теории:
        1) positive SHAP для двух моделей,
        2) robust normalization,
        3) регуляризованное гармоническое среднее H_tau,
        4) нормировка внутри подпространства,
        5) clipped object U(j)=min(1, |S| * phi(j)).
        """
        mu_pos = np.squeeze(np.asarray(mu_pos, dtype=float)).reshape(-1)
        mu_neg = np.squeeze(np.asarray(mu_neg, dtype=float)).reshape(-1)

        mu_pos = np.maximum(mu_pos, 0.0)
        mu_neg = np.maximum(mu_neg, 0.0)

        mu_pos = EnsembleReducer._robust_norm(mu_pos)
        mu_neg = EnsembleReducer._robust_norm(mu_neg)

        phi_sub = _h_tau(mu_pos, mu_neg)

        s = phi_sub.sum()
        if not np.isfinite(s) or s <= 0:
            phi_sub = np.full_like(phi_sub, 1.0 / max(1, phi_sub.size))
        else:
            phi_sub = phi_sub / s

        U_sub = np.minimum(1.0, float(subspace_size) * phi_sub)
        return U_sub

    @staticmethod
    def _project_capped_simplex(p: np.ndarray, L: float, U: float) -> np.ndarray:
        p = np.asarray(p, dtype=float)
        n = p.size
        L = float(L)
        U = float(U)
        L = min(L, 1.0 / n)
        U = max(U, 1.0 / n)

        x = np.clip(p, L, U)
        r = 1.0 - x.sum()
        if abs(r) < 1e-12:
            return x / x.sum()

        for _ in range(5):
            if r > 0:
                mask = (x < U - 1e-12)
                if not np.any(mask):
                    break
                w = p[mask].copy()
                w[w < 0] = 0.0
                if not np.any(w):
                    w = np.ones_like(w)
                w = w / w.sum()
                delta = np.minimum(U - x[mask], r * w / (w.sum() + 1e-12))
                add = delta * (r / (delta.sum() + 1e-12))
                x[mask] += add
                r = 1.0 - x.sum()
            else:
                mask = (x > L + 1e-12)
                if not np.any(mask):
                    break
                w = p[mask].copy()
                w[w < 0] = 0.0
                if not np.any(w):
                    w = np.ones_like(w)
                w = w / w.sum()
                delta = np.minimum(x[mask] - L, (-r) * w / (w.sum() + 1e-12))
                sub = delta * ((-r) / (delta.sum() + 1e-12))
                x[mask] -= sub
                r = 1.0 - x.sum()

            if abs(r) < 1e-12:
                break

        x = np.clip(x, L, U)
        x /= x.sum()
        return x

    def _sample_subspace_size(self, D: int, rho: float = 0.25) -> int:
        """
        Выбирает размер подпространства d in {1, ..., D}
        из смеси:
        rho * Uniform({1,...,D}) + (1-rho) * IncreasingLinear.
        """
        D = int(max(1, D))
        d_grid = np.arange(1, D + 1, dtype=int)

        p_unif = np.full(D, 1.0 / D, dtype=float)
        p_inc = 2.0 * d_grid.astype(float) / (D * (D + 1))

        p = rho * p_unif + (1.0 - rho) * p_inc
        p /= p.sum()

        return int(self.rng.choice(d_grid, p=p))

    def _draw_candidate_subspaces(self, n_feat: int) -> list[np.ndarray]:
        """
        Сэмплирует сразу все подпространства для одного outer-slot,
        после чего их можно сгруппировать по размерности и обучать пакетно.
        """
        D_max = min(self.num_coords, n_feat)
        lam = 0.3

        out: list[np.ndarray] = []
        for _ in range(self.num_attempts):
            d_cur = self._sample_subspace_size(D_max, rho=0.25)

            if self.rng.random() < lam:
                subs_idx = self.rng.choice(n_feat, size=d_cur, replace=False)
            else:
                subs_idx = self.rng.choice(
                    n_feat,
                    size=d_cur,
                    replace=False,
                    p=self.coord_prob,
                )

            out.append(np.sort(subs_idx).astype(int))

        return out

    def _compute_step_shap_batched(
        self,
        model_indices: list[int],
        X_tr: torch.Tensor,
        X_par: torch.Tensor,
        y_par: torch.Tensor,
        bg_idx: torch.Tensor,
        ev_idx: torch.Tensor,
        n_feat: int,
    ) -> tuple[np.ndarray, int]:
        """
        Считает SHAP для одного outer-step пакетно:
        - группировка по размерности подпространства,
        - chunking по числу моделей,
        - внутри чанка один GradientExplainer для positive и один для negative.
        """
        coord_shap_sum = np.zeros(n_feat, dtype=float)
        n_shap_blocks = 0

        groups: dict[int, list[int]] = defaultdict(list)
        for idx in model_indices:
            subs = self.list_model_subs[idx]
            groups[len(subs)].append(idx)

        if self.use_gpu_shap and self.device.type == "cuda":
            shap_device = self.device
            X_tr_src = X_tr
            X_par_src = X_par
            y_par_src = y_par
        else:
            shap_device = torch.device("cpu")
            X_tr_src = X_tr.detach().cpu()
            X_par_src = X_par.detach().cpu()
            y_par_src = y_par.detach().cpu()

        X_bg_base = X_tr_src[bg_idx]
        X_eval_base = X_par_src[ev_idx]
        y_eval_base = y_par_src[ev_idx]

        for d_cur, idx_list in groups.items():
            for idx_chunk in _chunk_list(idx_list, self.max_shap_models_per_batch):
                subs_chunk = [self.list_model_subs[idx] for idx in idx_chunk]

                X_bg_pack = _pack_inputs_for_shap(X_bg_base, subs_chunk).to(shap_device)
                X_eval_pack = _pack_inputs_for_shap(X_eval_base, subs_chunk).to(shap_device)

                pos_mask = (y_eval_base == 1)
                neg_mask = (y_eval_base == -1)

                X_pos_pack = X_eval_pack[pos_mask]
                X_neg_pack = X_eval_pack[neg_mask]

                if X_pos_pack.shape[0] == 0:
                    X_pos_pack = X_eval_pack
                if X_neg_pack.shape[0] == 0:
                    X_neg_pack = X_eval_pack

                models_pos = [
                    copy.deepcopy(self.list_model_pos[idx]).to(shap_device).eval()
                    for idx in idx_chunk
                ]
                models_neg = [
                    copy.deepcopy(self.list_model_neg[idx]).to(shap_device).eval()
                    for idx in idx_chunk
                ]

                mu_pos_batch = _batched_gradient_shap_for_group(
                    models=models_pos,
                    X_bg_pack=X_bg_pack,
                    X_eval_pack=X_pos_pack,
                    device=shap_device,
                )

                mu_neg_batch = _batched_gradient_shap_for_group(
                    models=models_neg,
                    X_bg_pack=X_bg_pack,
                    X_eval_pack=X_neg_pack,
                    device=shap_device,
                )

                for loc, idx_model in enumerate(idx_chunk):
                    subs = self.list_model_subs[idx_model]

                    mu_pos = np.asarray(mu_pos_batch[loc], dtype=float).reshape(-1)
                    mu_neg = np.asarray(mu_neg_batch[loc], dtype=float).reshape(-1)

                    if mu_pos.shape != mu_neg.shape:
                        raise ValueError(
                            f"Shape mismatch in batched SHAP: {mu_pos.shape} vs {mu_neg.shape}"
                        )

                    U_sub = self._theory_shap_importance(
                        mu_pos=mu_pos,
                        mu_neg=mu_neg,
                        subspace_size=len(subs),
                    )

                    coord_shap_sum[subs] += U_sub
                    n_shap_blocks += 1

        return coord_shap_sum, n_shap_blocks

    def fit(self, X, y):
        X_t = self._to_torch2d(X).to(self.device)
        y_t = self._to_torch1d_int(y).to(self.device)

        n_feat = X_t.shape[1]
        n_samples_total = X_t.shape[0]

        self.list_model_pos = []
        self.list_model_neg = []
        self.list_model_subs = []
        self.list_beta_pos = []
        self.list_beta_neg = []
        self.list_best_S = []

        self.list_best_epoch = []
        self.list_best_S_par = []
        self.list_best_F12_par = []
        self.list_best_G12_par = []
        self.list_best_F12_val = []
        self.list_best_G12_val = []

        self.anchor_set_ = np.array([], dtype=int)
        self.eta_history_ = []
        self.eta_last_ = None
        self.eta_final_ = None
        self.selected_indices_ = None

        y_np = y_t.detach().cpu().numpy().ravel()
        idx_np = np.arange(n_samples_total)

        tr_idx_np, tmp_idx_np = train_test_split(
            idx_np,
            test_size=0.4,
            random_state=self.seed,
            stratify=y_np,
        )

        y_tmp_np = y_np[tmp_idx_np]

        par_idx_np, val_idx_np = train_test_split(
            tmp_idx_np,
            test_size=0.5,
            random_state=self.seed + 1,
            stratify=y_tmp_np,
        )

        tr_idx = torch.as_tensor(tr_idx_np, dtype=torch.long, device=X_t.device)
        par_idx = torch.as_tensor(par_idx_np, dtype=torch.long, device=X_t.device)
        val_idx = torch.as_tensor(val_idx_np, dtype=torch.long, device=X_t.device)

        X_tr, y_tr = X_t[tr_idx], y_t[tr_idx]
        X_par, y_par = X_t[par_idx], y_t[par_idx]
        X_val, y_val = X_t[val_idx], y_t[val_idx]

        n_bg = min(128, X_tr.shape[0])
        n_ev = min(256, X_par.shape[0])

        bg_idx_np = self.rng.choice(X_tr.shape[0], size=n_bg, replace=False)
        bg_idx = torch.as_tensor(bg_idx_np, dtype=torch.long, device=X_tr.device)

        y_par_np = y_par.detach().cpu().numpy().ravel()
        pos_idx = np.where(y_par_np == 1)[0]
        neg_idx = np.where(y_par_np == -1)[0]

        n_ev_pos = min(len(pos_idx), max(1, n_ev // 2))
        n_ev_neg = min(len(neg_idx), max(1, n_ev - n_ev_pos))

        ev_pos = (
            self.rng.choice(pos_idx, size=n_ev_pos, replace=False)
            if len(pos_idx) > 0 else np.array([], dtype=int)
        )
        ev_neg = (
            self.rng.choice(neg_idx, size=n_ev_neg, replace=False)
            if len(neg_idx) > 0 else np.array([], dtype=int)
        )

        ev_idx_np = np.concatenate([ev_pos, ev_neg])
        if ev_idx_np.size == 0:
            ev_idx_np = self.rng.choice(
                X_par.shape[0],
                size=min(n_ev, X_par.shape[0]),
                replace=False,
            )

        self.rng.shuffle(ev_idx_np)
        ev_idx = torch.as_tensor(ev_idx_np, dtype=torch.long, device=X_par.device)

        fmin_all, fmax_all = compute_feature_bounds(TensorDataset(X_tr, y_tr))
        fmin_all = fmin_all.to(device=X_tr.device, dtype=X_tr.dtype)
        fmax_all = fmax_all.to(device=X_tr.device, dtype=X_tr.dtype)

        self.coord_prob = np.full(n_feat, 1.0 / n_feat, dtype=float)

        for k in range(self.num_samples):
            for i in range(self.num_models):
                best_model_pos = None
                best_model_neg = None
                best_subs_idx = None
                best_beta_pos = None
                best_beta_neg = None
                best_best_S = -float("inf")

                best_epoch = None
                best_S_par = None
                best_F12_par = None
                best_G12_par = None
                best_F12_val = None
                best_G12_val = None

                candidates = self._draw_candidate_subspaces(n_feat)

                groups: dict[int, list[tuple[int, np.ndarray]]] = defaultdict(list)
                for cand_id, subs_idx in enumerate(candidates):
                    groups[len(subs_idx)].append((cand_id, subs_idx))

                pos_tr_mask = (y_tr == 1)
                neg_tr_mask = (y_tr == -1)

                for d_cur, items in groups.items():
                    for chunk_id, items_chunk in enumerate(_chunk_list(items, self.max_models_per_batch)):
                        subs_batch_np = np.stack([subs for _, subs in items_chunk], axis=0)
                        subs_batch_t = torch.as_tensor(
                            subs_batch_np,
                            dtype=torch.long,
                            device=X_tr.device,
                        )

                        X_sub_tr_batch = _gather_subspaces_2d(X_tr, subs_batch_t)
                        X_sub_par_batch = _gather_subspaces_2d(X_par, subs_batch_t)
                        X_sub_val_batch = _gather_subspaces_2d(X_val, subs_batch_t)

                        fmin_batch = fmin_all[subs_batch_t]
                        fmax_batch = fmax_all[subs_batch_t]

                        X_pos_tr_batch = X_sub_tr_batch[:, pos_tr_mask, :]
                        X_neg_tr_batch = X_sub_tr_batch[:, neg_tr_mask, :]

                        if X_pos_tr_batch.shape[1] == 0 or X_neg_tr_batch.shape[1] == 0:
                            raise ValueError("Empty class encountered during batched candidate training")

                        seed_base = self.seed + 100000 * k + 1000 * i + 50 * d_cur + chunk_id

                        ckpt = _batched_train_with_checkpoints(
                            X_pos_tr_batch=X_pos_tr_batch,
                            X_neg_tr_batch=X_neg_tr_batch,
                            X_par_batch=X_sub_par_batch,
                            y_par=y_par,
                            fmin_batch=fmin_batch,
                            fmax_batch=fmax_batch,
                            d_hidden=self.d_hidden,
                            n_hidden_layers=self.n_hidden_layers,
                            num_epochs=self.num_epochs,
                            batch_size=self.batch_size,
                            checkpoint_every=self.checkpoint_every,
                            checkpoint_n_grid=self.checkpoint_n_grid,
                            cG=self.cG,
                            seed=seed_base,
                        )

                        model_pos_best_batch = BatchedMLP.build_batched_from_states(
                            ckpt["best_pos_states"],
                            device=self.device,
                            dtype=X_sub_val_batch.dtype,
                        )
                        model_neg_best_batch = BatchedMLP.build_batched_from_states(
                            ckpt["best_neg_states"],
                            device=self.device,
                            dtype=X_sub_val_batch.dtype,
                        )

                        beta_pos_batch = torch.as_tensor(
                            ckpt["best_beta_pos"],
                            device=self.device,
                            dtype=torch.float32,
                        )
                        beta_neg_batch = torch.as_tensor(
                            ckpt["best_beta_neg"],
                            device=self.device,
                            dtype=torch.float32,
                        )

                        model_pos_best_batch.eval()
                        model_neg_best_batch.eval()

                        with torch.no_grad():
                            s1_val_batch = model_pos_best_batch(X_sub_val_batch).squeeze(-1)
                            s2_val_batch = model_neg_best_batch(X_sub_val_batch).squeeze(-1)

                        m_val_batch = _batched_metrics_from_scores(
                            s1=s1_val_batch,
                            s2=s2_val_batch,
                            y=y_val,
                            beta_pos=beta_pos_batch,
                            beta_neg=beta_neg_batch,
                        )

                        F12_val_batch = m_val_batch["F12"]
                        G12_val_batch = m_val_batch["G12"]
                        S_val_batch = F12_val_batch - float(self.cG) * G12_val_batch

                        loc = int(torch.argmax(S_val_batch).item())
                        S_cur = float(S_val_batch[loc].item())

                        if S_cur > best_best_S:
                            best_best_S = S_cur
                            best_subs_idx = subs_batch_np[loc].copy()

                            best_beta_pos = float(beta_pos_batch[loc].item())
                            best_beta_neg = float(beta_neg_batch[loc].item())

                            best_epoch = int(ckpt["best_epoch"][loc])
                            best_S_par = float(ckpt["best_S_par"][loc])
                            best_F12_par = float(ckpt["best_F12_par"][loc])
                            best_G12_par = float(ckpt["best_G12_par"][loc])

                            best_F12_val = float(F12_val_batch[loc].item())
                            best_G12_val = float(G12_val_batch[loc].item())

                            best_model_pos = BatchedMLP.build_model_from_state(
                                ckpt["best_pos_states"][loc],
                                device=self.device,
                            )
                            best_model_neg = BatchedMLP.build_model_from_state(
                                ckpt["best_neg_states"][loc],
                                device=self.device,
                            )

                            best_model_pos.eval()
                            best_model_neg.eval()

                self.list_model_pos.append(best_model_pos)
                self.list_model_neg.append(best_model_neg)
                self.list_model_subs.append(best_subs_idx)
                self.list_beta_pos.append(best_beta_pos)
                self.list_beta_neg.append(best_beta_neg)
                self.list_best_S.append(best_best_S)

                self.list_best_epoch.append(best_epoch)
                self.list_best_S_par.append(best_S_par)
                self.list_best_F12_par.append(best_F12_par)
                self.list_best_G12_par.append(best_G12_par)
                self.list_best_F12_val.append(best_F12_val)
                self.list_best_G12_val.append(best_G12_val)

                if self.output:
                    print(
                        f"[outer={k+1}/{self.num_samples} model={i+1}/{self.num_models}] "
                        f"d={len(best_subs_idx) if best_subs_idx is not None else -1} "
                        f"epoch_best={best_epoch} "
                        f"S_par={best_S_par:.6f} "
                        f"F12_par={best_F12_par:.6f} "
                        f"G12_par={best_G12_par:.6f} "
                        f"S_val={best_best_S:.6f} "
                        f"F12_val={best_F12_val:.6f} "
                        f"G12_val={best_G12_val:.6f} "
                        f"beta_pos={best_beta_pos:.6f} "
                        f"beta_neg={best_beta_neg:.6f} "
                        f"cG={self.cG:.6f}"
                    )

            start = len(self.list_model_pos) - self.num_models
            end = len(self.list_model_pos)

            all_global_idx = list(range(start, end))

            coord_shap_sum, n_shap_blocks = self._compute_step_shap_batched(
                model_indices=all_global_idx,
                X_tr=X_tr,
                X_par=X_par,
                y_par=y_par,
                bg_idx=bg_idx,
                ev_idx=ev_idx,
                n_feat=n_feat,
            )

            if n_shap_blocks > 0:
                coord_new_prob = coord_shap_sum / float(n_shap_blocks)
            else:
                coord_new_prob = np.full(n_feat, 1.0 / n_feat, dtype=float)

            eta_step = np.asarray(coord_new_prob, dtype=float).copy()
            s = eta_step.sum()
            if not np.isfinite(s) or s <= 0:
                eta_step[:] = 1.0 / n_feat
            else:
                eta_step /= s

            self.eta_last_ = eta_step
            self.eta_history_.append(eta_step.copy())

            progress = 0.0 if self.num_samples <= 1 else k / (self.num_samples - 1)

            gamma = 0.80 + (0.60 - 0.80) * progress
            coord_raw = (1.0 - gamma) * self.coord_prob + gamma * eta_step

            raw_sum = coord_raw.sum()
            if not np.isfinite(raw_sum) or raw_sum <= 0:
                coord_raw[:] = 1.0 / n_feat
            else:
                coord_raw /= raw_sum

            A_prev = self.anchor_set_.copy()

            lb, ub, caps_info = dynamic_bounds(
                n=n_feat,
                q=self.n_select,
                A_idx=A_prev,
                step=k,
                n_steps=self.num_samples,
            )

            coord_proj = _project_box_simplex(coord_raw, lb=lb, ub=ub)
            self.coord_prob = coord_proj

            self.anchor_set_ = _expand_anchor_set(
                scores=eta_step,
                A_prev=A_prev,
                q=self.n_select,
                step=k,
                n_steps=self.num_samples,
            )

            print(f"Finished outer iteration {k+1}/{self.num_samples}")

            if self.output:
                print(
                    f"iter={k+1}/{self.num_samples} "
                    f"|A_prev|={caps_info['anchor_size']} "
                    f"L_A={caps_info['L_A']:.6f} "
                    f"L0={caps_info['L0']:.6f} "
                    f"U_A={caps_info['U_A']:.6f} "
                    f"U0={caps_info['U0']:.6f} "
                    f"reserve_out={caps_info['reserve_out']:.3f} "
                    f"gamma={gamma:.3f}"
                )
                print("A_used:", A_prev)
                print("Ahat_next:", self.anchor_set_)
                print("pi:", np.round(self.coord_prob, 4))
                print("eta:", np.round(eta_step, 4))

        if len(self.eta_history_) == 0:
            self.eta_final_ = self.coord_prob.copy()
        else:
            tail = min(3, len(self.eta_history_))
            #tail = min(5, len(self.eta_history_))
            self.eta_final_ = np.mean(self.eta_history_[-tail:], axis=0)
            s = self.eta_final_.sum()
            if not np.isfinite(s) or s <= 0:
                self.eta_final_ = np.full_like(self.coord_prob, 1.0 / len(self.coord_prob))
            else:
                self.eta_final_ /= s

        if self.output:
            print("final eta:", np.round(self.eta_final_, 4))
            print("final pi :", np.round(self.coord_prob, 4))
            final_k = min(self.n_select, len(self.eta_final_))
            final_idx = np.sort(np.argsort(self.eta_final_)[-final_k:])
            print("final selected by eta:", final_idx)

        return self

    def coord_prob_get(self):
        return self.coord_prob

    def eta_get(self):
        assert self.eta_final_ is not None, "Call fit() before eta_get()."
        return self.eta_final_

    def transform(self, X):
        assert self.coord_prob is not None, "Call fit() before transform()."
        assert self.eta_final_ is not None, "eta_final_ is missing. Call fit() first."
        assert hasattr(self, "n_select"), "n_select must be set in __init__()."

        score = np.asarray(self.eta_final_, dtype=float)
        k = min(self.n_select, len(score))
        topk_idx = np.argsort(score)[-k:]
        topk_idx = np.sort(topk_idx)
        self.selected_indices_ = topk_idx

        if isinstance(X, torch.Tensor):
            assert X.ndim == 2, "X must be 2D tensor"
            return X[:, topk_idx]
        else:
            X = np.asarray(X)
            assert X.ndim == 2, "X must be 2D array"
            return X[:, topk_idx]

36
35
600
60

In [276]:
def run_all_B(
    include_unary: bool = True,
    unary_params: dict | None = None,
    results_root: str | Path = "results_B",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    configs = [
        ("DS-B", make_dataset_B, [5]),
    ]



    methods = [
        ("Ensembleunar", lambda **kw: EnsembleReducer(
            d_hidden=32,
            n_hidden_layers=2,
            num_epochs=40,
            batch_size=512,
            n_select=kw.pop("q"),   # важно: q прокидываем как n_select
            num_models=96,
            num_attempts=500,
            num_coords=4,
            num_samples=4,
            seed=42,
            output=True,
            **kw,
        )),
    ]

    results_root = Path(results_root)
    results_root.mkdir(parents=True, exist_ok=True)

    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    else:
        DEVICE = torch.device("cpu")

    print("Using device:", DEVICE)
    if DEVICE.type == "cuda":
        print("CUDA name:", torch.cuda.get_device_name(0))

    base_unary_params = dict(
        d_hidden=32,
        n_hidden_layers=2,
        num_epochs=100,
        batch_size=512 if DEVICE.type == "cuda" else 128,
        checkpoint_every=5,
        checkpoint_n_grid=19,
        cG=0.25,
        device=str(DEVICE),
    )
    if unary_params is not None:
        base_unary_params.update(unary_params)

    rows_cls_all: List[Dict[str, Any]] = []
    rows_un_all: List[Dict[str, Any]] = []

    for dname, maker, qgrid in configs:
        X, y = maker()

        dataset_dir = results_root / dname
        dataset_dir.mkdir(parents=True, exist_ok=True)

        for name, ctor in methods:
            for q in qgrid:
                method_dir = dataset_dir / f"{name}_q{q}"
                method_dir.mkdir(parents=True, exist_ok=True)

                if name == "Ensembleunar":
                    reducer_kwargs = {
                        "device": str(DEVICE),
                        "use_gpu_shap": (DEVICE.type == "cuda"),
                        "max_models_per_batch": 500 if DEVICE.type == "cuda" else 64,
                        "max_shap_models_per_batch": 128 if DEVICE.type == "cuda" else 16,
                        "checkpoint_every": 5,
                        "checkpoint_n_grid": 19,
                        "cG": 0.25,
                    }
                else:
                    reducer_kwargs = {}

                cls_rows, un_rows = evaluate_reduction(
                    X=X,
                    y=y,
                    reducer_name=name,
                    reducer_ctor=ctor,
                    q=q,
                    dataset_name=dname,
                    reducer_kwargs=reducer_kwargs,
                    compute_unary_on_reduced=include_unary,
                    unary_params=base_unary_params,
                    results_dir=method_dir,
                    model_seed=42,
                )

                rows_cls_all.extend(cls_rows)
                rows_un_all.extend(un_rows)

    df_cls = pd.DataFrame(rows_cls_all)
    df_un = pd.DataFrame(rows_un_all)

    for col in ["AUC", "PR_AUC", "ΔAUC", "ΔPR_AUC", "t_reducer", "t_model", "t_model_base"]:
        if col in df_cls.columns:
            df_cls[col] = pd.to_numeric(df_cls[col], errors="coerce")

    agg = (
        df_cls.groupby(["dataset", "method", "q", "model"], dropna=False)
        .agg(
            AUC_mean=("AUC", "mean"),
            AUC_std=("AUC", "std"),
            PR_AUC_mean=("PR_AUC", "mean"),
            PR_AUC_std=("PR_AUC", "std"),
            dAUC_mean=("ΔAUC", "mean"),
            dAUC_std=("ΔAUC", "std"),
            dPR_mean=("ΔPR_AUC", "mean"),
            dPR_std=("ΔPR_AUC", "std"),
            t_red_med=("t_reducer", "median"),
            t_mod_med=("t_model", "median"),
        )
        .reset_index()
        .sort_values(["dataset", "method", "q", "dAUC_mean"], ascending=[True, True, True, False])
    )

    if not df_un.empty:
        agg_un = (
            df_un.groupby(["dataset", "method", "q", "model"], dropna=False)
            .agg(
                F12_mean=("F12", "mean"),
                F12_std=("F12", "std"),
                G12_mean=("G12", "mean"),
                G12_std=("G12", "std"),
                S_test_mean=("S_test", "mean"),
                S_test_std=("S_test", "std"),
                coverage_mean=("coverage", "mean"),
                coverage_std=("coverage", "std"),
            )
            .reset_index()
        )
        agg_un.to_csv(results_root / "summary_unary.csv", index=False)
    else:
        agg_un = pd.DataFrame()

    df_cls.to_csv(results_root / "all_classical_results.csv", index=False)
    df_un.to_csv(results_root / "all_unary_results.csv", index=False)
    agg.to_csv(results_root / "summary_classical.csv", index=False)

    results = {
        "df_cls": df_cls,
        "df_un": df_un,
        "agg": agg,
        "agg_un": agg_un,
    }
    with open(results_root / "results_B.pkl", "wb") as f:
        pickle.dump(results, f)

    return df_cls, agg

In [ ]:
df_cls, agg = run_all_B(
    include_unary=True,
    unary_params={
        "d_hidden": 32,
        "n_hidden_layers": 2,
        "num_epochs": 100,
        "batch_size": 128,
        "beta_grid": 19,
        "cG": 0.25,
    },
    results_root="results_B_full",
)

Using device: cuda
CUDA name: NVIDIA GeForce GTX 1080


/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer=1/4 model=1/96] d=1 epoch_best=5 S_par=0.521013 F12_par=0.524177 G12_par=0.012657 S_val=0.557475 F12_val=0.560921 G12_val=0.013787 beta_pos=0.499189 beta_neg=0.764204 cG=0.250000
[outer=1/4 model=2/96] d=1 epoch_best=10 S_par=0.484518 F12_par=0.485710 G12_par=0.004768 S_val=0.497805 F12_val=0.498761 G12_val=0.003822 beta_pos=0.646519 beta_neg=0.782254 cG=0.250000
[outer=1/4 model=3/96] d=1 epoch_best=5 S_par=0.544297 F12_par=0.550118 G12_par=0.023288 S_val=0.570562 F12_val=0.579442 G12_val=0.035522 beta_pos=0.612539 beta_neg=0.634544 cG=0.250000
[outer=1/4 model=4/96] d=4 epoch_best=10 S_par=0.578536 F12_par=0.594818 G12_par=0.065130 S_val=0.573996 F12_val=0.584440 G12_val=0.041779 beta_pos=0.200807 beta_neg=0.665209 cG=0.250000
[outer=1/4 model=5/96] d=4 epoch_best=10 S_par=0.550842 F12_par=0.572944 G12_par=0.088405 S_val=0.478312 F12_val=0.502865 G12_val=0.098209 beta_pos=0.491501 beta_neg=0.716549 cG=0.250000
[outer=1/4 model=6/96] d=3 epoch_best=10 S_par=0.525056 F12_par=0.5

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer=1/4 model=1/96] d=1 epoch_best=5 S_par=0.543063 F12_par=0.543063 G12_par=0.000000 S_val=0.488094 F12_val=0.488094 G12_val=0.000000 beta_pos=0.512138 beta_neg=0.777837 cG=0.250000
[outer=1/4 model=2/96] d=2 epoch_best=5 S_par=0.500256 F12_par=0.511745 G12_par=0.045956 S_val=0.483589 F12_val=0.492470 G12_val=0.035522 beta_pos=0.484484 beta_neg=0.655577 cG=0.250000
[outer=1/4 model=3/96] d=1 epoch_best=5 S_par=0.585511 F12_par=0.585511 G12_par=0.000000 S_val=0.586390 F12_val=0.586390 G12_val=0.000000 beta_pos=0.598700 beta_neg=0.665999 cG=0.250000
[outer=1/4 model=4/96] d=4 epoch_best=10 S_par=0.588315 F12_par=0.600686 G12_par=0.049482 S_val=0.549329 F12_val=0.563080 G12_val=0.055006 beta_pos=0.219300 beta_neg=0.657147 cG=0.250000
[outer=1/4 model=5/96] d=2 epoch_best=10 S_par=0.496438 F12_par=0.507582 G12_par=0.044573 S_val=0.466221 F12_val=0.473698 G12_val=0.029906 beta_pos=0.418546 beta_neg=0.826882 cG=0.250000
[outer=1/4 model=6/96] d=1 epoch_best=5 S_par=0.530333 F12_par=0.534

...

[outer=1/4 model=1/96] d=1 epoch_best=5 S_par=0.502218 F12_par=0.507199 G12_par=0.019926 S_val=0.547529 F12_val=0.550399 G12_val=0.011480 beta_pos=0.486616 beta_neg=0.758815 cG=0.250000
[outer=1/4 model=2/96] d=3 epoch_best=5 S_par=0.470223 F12_par=0.496865 G12_par=0.106569 S_val=0.512745 F12_val=0.530947 G12_val=0.072809 beta_pos=0.409437 beta_neg=0.404086 cG=0.250000
[outer=1/4 model=3/96] d=1 epoch_best=5 S_par=0.593104 F12_par=0.597310 G12_par=0.016822 S_val=0.594909 F12_val=0.596101 G12_val=0.004768 beta_pos=0.605663 beta_neg=0.629657 cG=0.250000
[outer=1/4 model=4/96] d=2 epoch_best=5 S_par=0.574038 F12_par=0.579589 G12_par=0.022205 S_val=0.564657 F12_val=0.577777 G12_val=0.052479 beta_pos=0.230449 beta_neg=0.437224 cG=0.250000
[outer=1/4 model=5/96] d=4 epoch_best=10 S_par=0.493473 F12_par=0.529315 G12_par=0.143368 S_val=0.485938 F12_val=0.526865 G12_val=0.163710 beta_pos=0.488255 beta_neg=0.717689 cG=0.250000
[outer=1/4 model=6/96] d=3 epoch_best=10 S_par=0.526472 F12_par=0.541

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer=1/4 model=1/96] d=1 epoch_best=5 S_par=0.515021 F12_par=0.515021 G12_par=0.000000 S_val=0.499205 F12_val=0.499205 G12_val=0.000000 beta_pos=0.502952 beta_neg=0.764870 cG=0.250000
[outer=1/4 model=2/96] d=4 epoch_best=5 S_par=0.507743 F12_par=0.516286 G12_par=0.034174 S_val=0.511344 F12_val=0.518484 G12_val=0.028562 beta_pos=0.300216 beta_neg=0.389203 cG=0.250000
[outer=1/4 model=3/96] d=1 epoch_best=5 S_par=0.549665 F12_par=0.549665 G12_par=0.000000 S_val=0.546929 F12_val=0.548286 G12_val=0.005428 beta_pos=0.598045 beta_neg=0.683658 cG=0.250000
[outer=1/4 model=4/96] d=2 epoch_best=5 S_par=0.602068 F12_par=0.607421 G12_par=0.021415 S_val=0.557837 F12_val=0.560863 G12_val=0.012102 beta_pos=0.229247 beta_neg=0.421560 cG=0.250000
[outer=1/4 model=5/96] d=4 epoch_best=10 S_par=0.522158 F12_par=0.547166 G12_par=0.100029 S_val=0.483923 F12_val=0.501910 G12_val=0.071947 beta_pos=0.521508 beta_neg=0.717832 cG=0.250000
[outer=1/4 model=6/96] d=3 epoch_best=10 S_par=0.540595 F12_par=0.554

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer=1/4 model=1/96] d=1 epoch_best=5 S_par=0.530876 F12_par=0.530876 G12_par=0.000000 S_val=0.563739 F12_val=0.563739 G12_val=0.000000 beta_pos=0.551227 beta_neg=0.750369 cG=0.250000
[outer=1/4 model=2/96] d=4 epoch_best=5 S_par=0.524222 F12_par=0.540892 G12_par=0.066678 S_val=0.532109 F12_val=0.543633 G12_val=0.046094 beta_pos=0.297128 beta_neg=0.387083 cG=0.250000
[outer=1/4 model=3/96] d=1 epoch_best=5 S_par=0.541696 F12_par=0.541696 G12_par=0.000000 S_val=0.571439 F12_val=0.571439 G12_val=0.000000 beta_pos=0.395838 beta_neg=0.809580 cG=0.250000
[outer=1/4 model=4/96] d=2 epoch_best=5 S_par=0.580541 F12_par=0.589211 G12_par=0.034680 S_val=0.601114 F12_val=0.605821 G12_val=0.018828 beta_pos=0.223640 beta_neg=0.482900 cG=0.250000
[outer=1/4 model=5/96] d=4 epoch_best=5 S_par=0.454996 F12_par=0.473767 G12_par=0.075085 S_val=0.485114 F12_val=0.504580 G12_val=0.077862 beta_pos=0.129620 beta_neg=0.434561 cG=0.250000
[outer=1/4 model=6/96] d=3 epoch_best=5 S_par=0.544207 F12_par=0.56097

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>dataset</th>
      <th>method</th>
      <th>q</th>
      <th>model</th>
      <th>AUC_mean</th>
      <th>PR_AUC_mean</th>
      <th>dAUC_mean</th>
      <th>dPR_mean</th>
      <th>t_red_med</th>
      <th>t_mod_med</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>DS-B</td>
      <td>Ensembleunar</td>
      <td>5</td>
      <td>HGBT</td>
      <td>0.918878</td>
      <td>0.822972</td>
      <td>-0.038840</td>
      <td>-0.083135</td>
      <td>262.616889</td>
      <td>0.354890</td>
    </tr>
    <tr>
      <th>1</th>
      <td>DS-B</td>
      <td>Ensembleunar</td>
      <td>5</td>
      <td>MLP</td>
      <td>0.938234</td>
      <td>0.870474</td>
      <td>0.008147</td>
      <td>0.009184</td>
      <td>262.616889</td>
      <td>0.380501</td>
    </tr>
  </tbody>
</table>
</div>

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))

In [235]:
def run_colon_10fold_cv(
    cv_seed: int = 42,
    reducer_seed: int = 42,
    clf_seed: int = 42,
    results_dir: str = "results_colon_cv",
):
    # -----------------------------
    # 1. данные
    # -----------------------------
    X, y = make_dataset_colon()
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y).astype(int).ravel()

    uniq = np.unique(y)
    if set(uniq) != {-1, 1}:
        raise ValueError(f"Expected y in {{-1, 1}}, got {uniq}")

    # -----------------------------
    # 2. папка с результатами
    # -----------------------------
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)

    eta_dir = results_path / "eta_vectors"
    eta_dir.mkdir(exist_ok=True)

    indices_dir = results_path / "test_indices"
    indices_dir.mkdir(exist_ok=True)

    # -----------------------------
    # 3. device
    # -----------------------------
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    # -----------------------------
    # 4. CV: 10 фолдов ~ 90/10
    # -----------------------------
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=cv_seed,
    )

    fold_rows = []

    run_info = {
        "cv_seed": cv_seed,
        "reducer_seed": reducer_seed,
        "clf_seed": clf_seed,
        "device": device,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_splits": 10,
    }
    with open(results_path / "run_info.json", "w", encoding="utf-8") as f:
        json.dump(run_info, f, ensure_ascii=False, indent=2)

    # -----------------------------
    # 5. цикл по фолдам
    # -----------------------------
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        print("=" * 80)
        print(f"Fold {fold_id}/10")

        np.savetxt(
            indices_dir / f"fold_{fold_id:02d}_test_indices.txt",
            test_idx,
            fmt="%d",
        )

        # ---- split ----
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        print("Train shape:", X_train.shape)
        print("Test shape :", X_test.shape)
        print("Test indices:", test_idx.tolist())

        # -------------------------------------------------
        # ВАЖНО: стандартизация СРАЗУ, до reducer
        # fit только на train, test трансформируем тем же scaler
        # -------------------------------------------------
        scaler_full = StandardScaler()
        X_train_sc = scaler_full.fit_transform(X_train)
        X_test_sc = scaler_full.transform(X_test)

        # ---- reducer ----
        reducer = EnsembleReducer(
            d_hidden=64,
            n_hidden_layers=2,
            num_epochs=100,
            batch_size=16,
            n_select=14,
            num_models=100,
            num_attempts=5000,
            num_coords=8,
            num_samples=4,
            seed=reducer_seed + fold_id,
            output=True,
            device=device,
            use_gpu_shap=(device == "cuda"),
            max_models_per_batch=5000,
            max_shap_models_per_batch=1024,
            checkpoint_every=5,
            checkpoint_n_grid=19,
            cG=0.25,
        )

        # reducer обучается уже на стандартизованных признаках
        reducer.fit(X_train_sc, y_train)

        # ---- сохранить полный eta-вектор ----
        eta = reducer.eta_get().copy()
        np.save(eta_dir / f"fold_{fold_id:02d}_eta.npy", eta)

        eta_df = pd.DataFrame({
            "feature_index": np.arange(len(eta), dtype=int),
            "eta": eta,
        })
        eta_df.to_csv(
            eta_dir / f"fold_{fold_id:02d}_eta.csv",
            index=False,
        )

        # ---- отбор координат ----
        # transform тоже на стандартизованных матрицах
        X_train_red = reducer.transform(X_train_sc)
        X_test_red = reducer.transform(X_test_sc)
        selected_indices = reducer.selected_indices_.copy()

        print("Original train shape:", X_train.shape)
        print("Scaled train shape  :", X_train_sc.shape)
        print("Reduced train shape :", X_train_red.shape)
        print("Original test shape :", X_test.shape)
        print("Scaled test shape   :", X_test_sc.shape)
        print("Reduced test shape  :", X_test_red.shape)
        print("Selected indices:", selected_indices)

        # -------------------------------------------------
        # Второй scaler уже НЕ нужен:
        # выбранные признаки взяты из X_train_sc / X_test_sc,
        # то есть они уже стандартизованы по train
        # -------------------------------------------------

        # ---- L1 logistic regression ----
        clf = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=1.0,
            max_iter=5000,
            random_state=clf_seed + fold_id,
        )

        clf.fit(X_train_red, y_train)

        # ---- предсказания ----
        y_pred = clf.predict(X_test_red)

        acc = accuracy_score(y_test, y_pred)
        err = 1.0 - acc
        f1 = f1_score(y_test, y_pred, pos_label=1)

        print(f"Fold {fold_id} accuracy            : {acc:.6f}")
        print(f"Fold {fold_id} classification error: {err:.6f}")
        print(f"Fold {fold_id} F1-score            : {f1:.6f}")

        # ---- сохранить подробности по фолду ----
        fold_row = {
            "fold": fold_id,
            "train_size": int(len(train_idx)),
            "test_size": int(len(test_idx)),
            "accuracy": float(acc),
            "classification_error": float(err),
            "f1_score": float(f1),
            "n_selected": int(len(selected_indices)),
            "selected_indices": ",".join(map(str, selected_indices.tolist())),
            "test_indices_file": str(indices_dir / f"fold_{fold_id:02d}_test_indices.txt"),
            "eta_npy_file": str(eta_dir / f"fold_{fold_id:02d}_eta.npy"),
            "eta_csv_file": str(eta_dir / f"fold_{fold_id:02d}_eta.csv"),
        }
        fold_rows.append(fold_row)

        fold_json = {
            "fold": fold_id,
            "test_indices": test_idx.tolist(),
            "selected_indices": selected_indices.tolist(),
            "accuracy": float(acc),
            "classification_error": float(err),
            "f1_score": float(f1),
        }
        with open(results_path / f"fold_{fold_id:02d}_summary.json", "w", encoding="utf-8") as f:
            json.dump(fold_json, f, ensure_ascii=False, indent=2)

    # -----------------------------
    # 6. итоговая таблица
    # -----------------------------
    results_df = pd.DataFrame(fold_rows)

    summary_row = {
        "fold": "mean",
        "train_size": results_df["train_size"].mean(),
        "test_size": results_df["test_size"].mean(),
        "accuracy": results_df["accuracy"].mean(),
        "classification_error": results_df["classification_error"].mean(),
        "f1_score": results_df["f1_score"].mean(),
        "n_selected": results_df["n_selected"].mean(),
        "selected_indices": "",
        "test_indices_file": "",
        "eta_npy_file": "",
        "eta_csv_file": "",
    }
    std_row = {
        "fold": "std",
        "train_size": results_df["train_size"].std(ddof=1),
        "test_size": results_df["test_size"].std(ddof=1),
        "accuracy": results_df["accuracy"].std(ddof=1),
        "classification_error": results_df["classification_error"].std(ddof=1),
        "f1_score": results_df["f1_score"].std(ddof=1),
        "n_selected": results_df["n_selected"].std(ddof=1),
        "selected_indices": "",
        "test_indices_file": "",
        "eta_npy_file": "",
        "eta_csv_file": "",
    }

    results_df_full = pd.concat(
        [results_df, pd.DataFrame([summary_row, std_row])],
        ignore_index=True,
    )

    results_csv = results_path / "results.csv"
    results_df_full.to_csv(results_csv, index=False)

    print("=" * 80)
    print("CV finished")
    print(f"Mean accuracy: {results_df['accuracy'].mean():.6f}")
    print(f"Mean F1      : {results_df['f1_score'].mean():.6f}")
    print(f"Saved to     : {results_csv}")

    return results_df_full

In [259]:
results_df = run_colon_10fold_cv(
    cv_seed=42,
    reducer_seed=42,
    clf_seed=42,
    results_dir="results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
)

print(results_df)

device: cuda
Fold 1/10
Train shape: (55, 2000)
Test shape : (7, 2000)
Test indices: [13, 17, 30, 34, 37, 41, 43]
[outer=1/4 model=1/100] d=7 epoch_best=80 S_par=0.582394 F12_par=0.587751 G12_par=0.021429 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.350739 beta_neg=0.461046 cG=0.250000
[outer=1/4 model=2/100] d=3 epoch_best=5 S_par=0.922580 F12_par=0.922580 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.262120 beta_neg=0.380318 cG=0.250000
[outer=1/4 model=3/100] d=4 epoch_best=20 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.599381 beta_neg=0.491873 cG=0.250000
[outer=1/4 model=4/100] d=1 epoch_best=5 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.451005 beta_neg=0.572343 cG=0.250000
[outer=1/4 model=5/100] d=5 epoch_best=20 S_par=0.917223 F12_par=0.922580 G12_par=0.021429 S_val=0.922580 F12_val=0.922580 G12_val=0.0000

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[outer=1/4 model=1/100] d=4 epoch_best=10 S_par=0.631114 F12_par=0.631114 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.395908 beta_neg=0.465663 cG=0.250000
[outer=1/4 model=2/100] d=4 epoch_best=5 S_par=0.461042 F12_par=0.461042 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.138770 beta_neg=0.469685 cG=0.250000
[outer=1/4 model=3/100] d=3 epoch_best=10 S_par=0.832847 F12_par=0.832847 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.486915 beta_neg=0.495460 cG=0.250000
[outer=1/4 model=4/100] d=3 epoch_best=10 S_par=0.642801 F12_par=0.648158 G12_par=0.021429 S_val=0.917223 F12_val=0.922580 G12_val=0.021429 beta_pos=0.202556 beta_neg=0.698213 cG=0.250000
[outer=1/4 model=5/100] d=2 epoch_best=5 S_par=0.726810 F12_par=0.726810 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.430587 beta_neg=0.633963 cG=0.250000
[outer=1/4 model=6/100] d=7 epoch_best=45 S_par=0.731208 F12_p

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[outer=1/4 model=1/100] d=3 epoch_best=5 S_par=0.587751 F12_par=0.587751 G12_par=0.000000 S_val=0.807196 F12_val=0.807196 G12_val=0.000000 beta_pos=0.294647 beta_neg=0.504603 cG=0.250000
[outer=1/4 model=2/100] d=4 epoch_best=10 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.856653 F12_val=0.856653 G12_val=0.000000 beta_pos=0.368308 beta_neg=0.219913 cG=0.250000
[outer=1/4 model=3/100] d=5 epoch_best=25 S_par=0.666223 F12_par=0.666223 G12_par=0.000000 S_val=0.856653 F12_val=0.856653 G12_val=0.000000 beta_pos=0.420813 beta_neg=0.384165 cG=0.250000
[outer=1/4 model=4/100] d=7 epoch_best=25 S_par=0.648158 F12_par=0.648158 G12_par=0.000000 S_val=0.932836 F12_val=0.932836 G12_val=0.000000 beta_pos=0.311213 beta_neg=0.478559 cG=0.250000
[outer=1/4 model=5/100] d=1 epoch_best=5 S_par=0.631114 F12_par=0.631114 G12_par=0.000000 S_val=0.932836 F12_val=0.932836 G12_val=0.000000 beta_pos=0.753335 beta_neg=0.461497 cG=0.250000
[outer=1/4 model=6/100] d=4 epoch_best=25 S_par=0.731208 F12_p

...

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[outer=1/4 model=1/100] d=4 epoch_best=15 S_par=0.922580 F12_par=0.922580 G12_par=0.000000 S_val=0.928148 F12_val=0.932836 G12_val=0.018750 beta_pos=0.302912 beta_neg=0.243381 cG=0.250000
[outer=1/4 model=2/100] d=4 epoch_best=25 S_par=0.917223 F12_par=0.922580 G12_par=0.021429 S_val=0.932836 F12_val=0.932836 G12_val=0.000000 beta_pos=0.560255 beta_neg=0.545119 cG=0.250000
[outer=1/4 model=3/100] d=3 epoch_best=5 S_par=0.917223 F12_par=0.922580 G12_par=0.021429 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.224705 beta_neg=0.253389 cG=0.250000
[outer=1/4 model=4/100] d=6 epoch_best=15 S_par=0.648158 F12_par=0.648158 G12_par=0.000000 S_val=0.932836 F12_val=0.932836 G12_val=0.000000 beta_pos=0.380892 beta_neg=0.358716 cG=0.250000
[outer=1/4 model=5/100] d=3 epoch_best=10 S_par=0.999500 F12_par=0.999500 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.173770 beta_neg=0.331827 cG=0.250000
[outer=1/4 model=6/100] d=5 epoch_best=25 S_par=0.999500 F12_

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[outer=1/4 model=1/100] d=2 epoch_best=5 S_par=0.631114 F12_par=0.631114 G12_par=0.000000 S_val=0.856653 F12_val=0.856653 G12_val=0.000000 beta_pos=0.546749 beta_neg=0.353300 cG=0.250000
[outer=1/4 model=2/100] d=5 epoch_best=20 S_par=0.799503 F12_par=0.799503 G12_par=0.000000 S_val=0.851966 F12_val=0.856653 G12_val=0.018750 beta_pos=0.510254 beta_neg=0.527906 cG=0.250000
[outer=1/4 model=3/100] d=1 epoch_best=35 S_par=0.587751 F12_par=0.587751 G12_par=0.000000 S_val=0.856653 F12_val=0.856653 G12_val=0.000000 beta_pos=0.603497 beta_neg=0.639313 cG=0.250000
[outer=1/4 model=4/100] d=8 epoch_best=10 S_par=0.856653 F12_par=0.856653 G12_par=0.000000 S_val=0.932836 F12_val=0.932836 G12_val=0.000000 beta_pos=0.391467 beta_neg=0.213973 cG=0.250000
[outer=1/4 model=5/100] d=2 epoch_best=10 S_par=0.587751 F12_par=0.587751 G12_par=0.000000 S_val=0.856653 F12_val=0.856653 G12_val=0.000000 beta_pos=0.588783 beta_neg=0.431076 cG=0.250000
[outer=1/4 model=6/100] d=6 epoch_best=20 S_par=0.656848 F12_

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [211]:
def run_colon_10fold_cv_upd(
    cv_seed: int = 42,
    reducer_seed: int = 42,
    clf_seed: int = 42,
    results_dir: str = "results_colon_cv",
):
    # -----------------------------
    # 1. данные
    # -----------------------------
    X, y = make_dataset_colon()
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y).astype(int).ravel()

    uniq = np.unique(y)
    if set(uniq) != {-1, 1}:
        raise ValueError(f"Expected y in {{-1, 1}}, got {uniq}")

    # -----------------------------
    # 2. папка с результатами
    # -----------------------------
    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)

    eta_dir = results_path / "eta_vectors"
    eta_dir.mkdir(exist_ok=True)

    indices_dir = results_path / "test_indices"
    indices_dir.mkdir(exist_ok=True)

    # -----------------------------
    # 3. device
    # -----------------------------
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    # -----------------------------
    # 4. CV: 10 фолдов ~ 90/10
    # -----------------------------
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=cv_seed,
    )

    fold_rows = []

    run_info = {
        "cv_seed": cv_seed,
        "reducer_seed": reducer_seed,
        "clf_seed": clf_seed,
        "device": device,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_splits": 10,
    }
    with open(results_path / "run_info.json", "w", encoding="utf-8") as f:
        json.dump(run_info, f, ensure_ascii=False, indent=2)

    # -----------------------------
    # 5. цикл по фолдам
    # -----------------------------
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        print("=" * 80)
        print(f"Fold {fold_id}/10")

        np.savetxt(
            indices_dir / f"fold_{fold_id:02d}_test_indices.txt",
            test_idx,
            fmt="%d",
        )

        # ---- split ----
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        print("Train shape:", X_train.shape)
        print("Test shape :", X_test.shape)
        print("Test indices:", test_idx.tolist())

        # -------------------------------------------------
        # ВАЖНО: стандартизация СРАЗУ, до reducer
        # fit только на train, test трансформируем тем же scaler
        # -------------------------------------------------
        scaler_full = StandardScaler()
        X_train_sc = scaler_full.fit_transform(X_train)
        X_test_sc = scaler_full.transform(X_test)

        # ---- reducer ----
        reducer = EnsembleReducer(
            d_hidden=64,
            n_hidden_layers=2,
            num_epochs=100,
            batch_size=16,
            n_select=14,
            num_models=80,
            num_attempts=5000,
            num_coords=8,
            num_samples=5,
            seed=reducer_seed + fold_id,
            output=True,
            device=device,
            use_gpu_shap=(device == "cuda"),
            max_models_per_batch=5000,
            max_shap_models_per_batch=1024,
            checkpoint_every=5,
            checkpoint_n_grid=19,
            cG=0.25,
        )

        # reducer обучается уже на стандартизованных признаках
        reducer.fit(X_train_sc, y_train)

        # ---- сохранить полный eta-вектор ----
        eta = reducer.eta_get().copy()
        np.save(eta_dir / f"fold_{fold_id:02d}_eta.npy", eta)

        eta_df = pd.DataFrame({
            "feature_index": np.arange(len(eta), dtype=int),
            "eta": eta,
        })
        eta_df.to_csv(
            eta_dir / f"fold_{fold_id:02d}_eta.csv",
            index=False,
        )

        # ---- отбор координат ----
        # transform тоже на стандартизованных матрицах
        X_train_red = reducer.transform(X_train_sc)
        X_test_red = reducer.transform(X_test_sc)
        selected_indices = reducer.selected_indices_.copy()

        print("Original train shape:", X_train.shape)
        print("Scaled train shape  :", X_train_sc.shape)
        print("Reduced train shape :", X_train_red.shape)
        print("Original test shape :", X_test.shape)
        print("Scaled test shape   :", X_test_sc.shape)
        print("Reduced test shape  :", X_test_red.shape)
        print("Selected indices:", selected_indices)

        # -------------------------------------------------
        # Второй scaler уже НЕ нужен:
        # выбранные признаки взяты из X_train_sc / X_test_sc,
        # то есть они уже стандартизованы по train
        # -------------------------------------------------

        # ---- L1 logistic regression ----
        clf = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=1.0,
            max_iter=5000,
            random_state=clf_seed + fold_id,
        )

        clf.fit(X_train_red, y_train)

        # ---- предсказания ----
        y_pred = clf.predict(X_test_red)

        acc = accuracy_score(y_test, y_pred)
        err = 1.0 - acc
        f1 = f1_score(y_test, y_pred, pos_label=1)

        print(f"Fold {fold_id} accuracy            : {acc:.6f}")
        print(f"Fold {fold_id} classification error: {err:.6f}")
        print(f"Fold {fold_id} F1-score            : {f1:.6f}")

        # ---- сохранить подробности по фолду ----
        fold_row = {
            "fold": fold_id,
            "train_size": int(len(train_idx)),
            "test_size": int(len(test_idx)),
            "accuracy": float(acc),
            "classification_error": float(err),
            "f1_score": float(f1),
            "n_selected": int(len(selected_indices)),
            "selected_indices": ",".join(map(str, selected_indices.tolist())),
            "test_indices_file": str(indices_dir / f"fold_{fold_id:02d}_test_indices.txt"),
            "eta_npy_file": str(eta_dir / f"fold_{fold_id:02d}_eta.npy"),
            "eta_csv_file": str(eta_dir / f"fold_{fold_id:02d}_eta.csv"),
        }
        fold_rows.append(fold_row)

        fold_json = {
            "fold": fold_id,
            "test_indices": test_idx.tolist(),
            "selected_indices": selected_indices.tolist(),
            "accuracy": float(acc),
            "classification_error": float(err),
            "f1_score": float(f1),
        }
        with open(results_path / f"fold_{fold_id:02d}_summary.json", "w", encoding="utf-8") as f:
            json.dump(fold_json, f, ensure_ascii=False, indent=2)

    # -----------------------------
    # 6. итоговая таблица
    # -----------------------------
    results_df = pd.DataFrame(fold_rows)

    summary_row = {
        "fold": "mean",
        "train_size": results_df["train_size"].mean(),
        "test_size": results_df["test_size"].mean(),
        "accuracy": results_df["accuracy"].mean(),
        "classification_error": results_df["classification_error"].mean(),
        "f1_score": results_df["f1_score"].mean(),
        "n_selected": results_df["n_selected"].mean(),
        "selected_indices": "",
        "test_indices_file": "",
        "eta_npy_file": "",
        "eta_csv_file": "",
    }
    std_row = {
        "fold": "std",
        "train_size": results_df["train_size"].std(ddof=1),
        "test_size": results_df["test_size"].std(ddof=1),
        "accuracy": results_df["accuracy"].std(ddof=1),
        "classification_error": results_df["classification_error"].std(ddof=1),
        "f1_score": results_df["f1_score"].std(ddof=1),
        "n_selected": results_df["n_selected"].std(ddof=1),
        "selected_indices": "",
        "test_indices_file": "",
        "eta_npy_file": "",
        "eta_csv_file": "",
    }

    results_df_full = pd.concat(
        [results_df, pd.DataFrame([summary_row, std_row])],
        ignore_index=True,
    )

    results_csv = results_path / "results.csv"
    results_df_full.to_csv(results_csv, index=False)

    print("=" * 80)
    print("CV finished")
    print(f"Mean accuracy: {results_df['accuracy'].mean():.6f}")
    print(f"Mean F1      : {results_df['f1_score'].mean():.6f}")
    print(f"Saved to     : {results_csv}")

    return results_df_full

In [ ]:
results_df2 = run_colon_10fold_cv_upd(
    cv_seed=42,
    reducer_seed=42,
    clf_seed=42,
    results_dir="results_colon_cv_seed42_5_80_2_64_5000_0.25",
)

print(results_df)

device: cuda
Fold 1/10
Train shape: (55, 2000)
Test shape : (7, 2000)
Test indices: [13, 17, 30, 34, 37, 41, 43]
[outer=1/5 model=1/80] d=7 epoch_best=80 S_par=0.587751 F12_par=0.587751 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.350739 beta_neg=0.461046 cG=0.250000
[outer=1/5 model=2/80] d=3 epoch_best=5 S_par=0.922580 F12_par=0.922580 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.262120 beta_neg=0.380318 cG=0.250000
[outer=1/5 model=3/80] d=4 epoch_best=15 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.490682 beta_neg=0.377502 cG=0.250000
[outer=1/5 model=4/80] d=1 epoch_best=5 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.999500 F12_val=0.999500 G12_val=0.000000 beta_pos=0.451005 beta_neg=0.572343 cG=0.250000
[outer=1/5 model=5/80] d=6 epoch_best=10 S_par=0.731208 F12_par=0.731208 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 be

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


[outer=1/5 model=1/80] d=4 epoch_best=10 S_par=0.631114 F12_par=0.631114 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.395908 beta_neg=0.465663 cG=0.250000
[outer=1/5 model=2/80] d=4 epoch_best=5 S_par=0.461042 F12_par=0.461042 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.138770 beta_neg=0.452831 cG=0.250000
[outer=1/5 model=3/80] d=3 epoch_best=10 S_par=0.832847 F12_par=0.832847 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.486915 beta_neg=0.495460 cG=0.250000
[outer=1/5 model=4/80] d=3 epoch_best=10 S_par=0.648158 F12_par=0.648158 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.202556 beta_neg=0.698213 cG=0.250000
[outer=1/5 model=5/80] d=2 epoch_best=5 S_par=0.726810 F12_par=0.726810 G12_par=0.000000 S_val=0.922580 F12_val=0.922580 G12_val=0.000000 beta_pos=0.430587 beta_neg=0.633963 cG=0.250000
[outer=1/5 model=6/80] d=7 epoch_best=45 S_par=0.731208 F12_par=0.7

/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/eliseev/venvs/jupyter/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [85]:
def unary_predict_detailed(
    X: torch.Tensor,
    model_pos: nn.Module,
    model_neg: nn.Module,
    beta_pos: float,
    beta_neg: float,
) -> dict:
    """
    Детальная раскладка решений unary-пары.

    Возвращает:
      +1 : pos-only
      -1 : neg-only
       0 : non-decision (both или none)

    both и none возвращаются отдельно.
    """
    device = next(model_pos.parameters()).device
    X = X.to(device=device, dtype=torch.float32)

    model_pos.eval()
    model_neg.eval()

    with torch.no_grad():
        s1 = model_pos(X).flatten()
        s2 = model_neg(X).flatten()

    pos_hit = s1 >= float(beta_pos)
    neg_hit = s2 >= float(beta_neg)

    pos_only = pos_hit & (~neg_hit)
    neg_only = neg_hit & (~pos_hit)
    both = pos_hit & neg_hit
    none = (~pos_hit) & (~neg_hit)

    y_pred_strict = torch.zeros(X.shape[0], dtype=torch.int64, device=device)
    y_pred_strict[pos_only] = 1
    y_pred_strict[neg_only] = -1

    return {
        "y_pred_strict": y_pred_strict.detach().cpu().numpy(),   # {-1, 0, +1}
        "pos_only_mask": pos_only.detach().cpu().numpy(),
        "neg_only_mask": neg_only.detach().cpu().numpy(),
        "both_mask": both.detach().cpu().numpy(),
        "none_mask": none.detach().cpu().numpy(),
        "decided_mask": (pos_only | neg_only).detach().cpu().numpy(),
        "scores_pos": s1.detach().cpu().numpy(),
        "scores_neg": s2.detach().cpu().numpy(),
    }


In [288]:
def strict_unary_metrics(
    y_true: np.ndarray,
    pred: dict,
    *,
    cG: float,
    h_tau: float = 1e-3,
) -> dict:
    """
    Criterion-aligned + интерпретационные метрики для unary-модели.

    Главные:
      a1, a2, b1, b2, F12, G12, S

    Дополнительные:
      coverage, reject_rate, conflict_rate,
      selective_accuracy, selective_f1,
      conservative_accuracy, conservative_error.
    """
    y_true = np.asarray(y_true, dtype=int).ravel()

    pos_only = np.asarray(pred["pos_only_mask"], dtype=bool).ravel()
    neg_only = np.asarray(pred["neg_only_mask"], dtype=bool).ravel()
    both = np.asarray(pred["both_mask"], dtype=bool).ravel()
    none = np.asarray(pred["none_mask"], dtype=bool).ravel()
    y_pred_strict = np.asarray(pred["y_pred_strict"], dtype=int).ravel()

    n = y_true.shape[0]
    if not (pos_only.shape[0] == neg_only.shape[0] == both.shape[0] == none.shape[0] == n):
        raise ValueError("Prediction masks and y_true must have the same length")

    total = pos_only.astype(int) + neg_only.astype(int) + both.astype(int) + none.astype(int)
    if not np.all(total == 1):
        raise ValueError("Each object must belong to exactly one of: pos_only, neg_only, both, none")

    pos = (y_true == 1)
    neg = (y_true == -1)

    n1 = int(pos.sum())
    n2 = int(neg.sum())

    # 8-клеточная раскладка
    c_pos = int((pos_only & pos).sum())   # correct exclusive on + class
    w_pos = int((neg_only & pos).sum())   # wrong exclusive on + class
    b_pos = int((both & pos).sum())       # conflict on + class
    r_pos = int((none & pos).sum())       # abstain on + class

    c_neg = int((neg_only & neg).sum())   # correct exclusive on - class
    w_neg = int((pos_only & neg).sum())   # wrong exclusive on - class
    b_neg = int((both & neg).sum())       # conflict on - class
    r_neg = int((none & neg).sum())       # abstain on - class

    # criterion-aligned quantities
    a1 = c_pos / n1 if n1 > 0 else np.nan
    a2 = c_neg / n2 if n2 > 0 else np.nan
    b1 = b_pos / n1 if n1 > 0 else np.nan
    b2 = b_neg / n2 if n2 > 0 else np.nan

    if n1 > 0 and n2 > 0:
        F12 = 2.0 * a1 * a2 / (a1 + a2 + h_tau)
        G12 = 2.0 * b1 * b2 / (b1 + b2 + h_tau)
        #G12 = (b1 + b2) / 2
        #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2
        S = F12 - float(cG) * G12
    else:
        F12 = np.nan
        G12 = np.nan
        S = np.nan

    decided_count = int((pos_only | neg_only).sum())
    conflict_count = int(both.sum())
    reject_count = int(none.sum())
    nondecision_count = conflict_count + reject_count

    correct_exclusive_count = c_pos + c_neg
    wrong_exclusive_count = w_pos + w_neg

    coverage = decided_count / n if n > 0 else np.nan
    conflict_rate = conflict_count / n if n > 0 else np.nan
    reject_rate = reject_count / n if n > 0 else np.nan
    nondecision_rate = nondecision_count / n if n > 0 else np.nan
    wrong_exclusive_rate = wrong_exclusive_count / n if n > 0 else np.nan
    correct_exclusive_rate = correct_exclusive_count / n if n > 0 else np.nan

    # selective metrics: только на decided subset
    decided_mask = (y_pred_strict != 0)
    if decided_count > 0:
        y_true_dec = y_true[decided_mask]
        y_pred_dec = y_pred_strict[decided_mask]

        selective_accuracy = accuracy_score(y_true_dec, y_pred_dec)
        selective_error = 1.0 - selective_accuracy
        selective_f1 = f1_score(y_true_dec, y_pred_dec, pos_label=1, zero_division=0)
    else:
        selective_accuracy = np.nan
        selective_error = np.nan
        selective_f1 = np.nan

    # conservative metrics: both и none считаем ошибкой
    conservative_accuracy = correct_exclusive_count / n if n > 0 else np.nan
    conservative_error = 1.0 - conservative_accuracy if n > 0 else np.nan

    return {
        "n_test": int(n),
        "n_pos": int(n1),
        "n_neg": int(n2),

        "c_pos": int(c_pos),
        "w_pos": int(w_pos),
        "b_pos": int(b_pos),
        "r_pos": int(r_pos),
        "c_neg": int(c_neg),
        "w_neg": int(w_neg),
        "b_neg": int(b_neg),
        "r_neg": int(r_neg),

        "a1": float(a1) if not np.isnan(a1) else np.nan,
        "a2": float(a2) if not np.isnan(a2) else np.nan,
        "b1": float(b1) if not np.isnan(b1) else np.nan,
        "b2": float(b2) if not np.isnan(b2) else np.nan,
        "F12": float(F12) if not np.isnan(F12) else np.nan,
        "G12": float(G12) if not np.isnan(G12) else np.nan,
        "S": float(S) if not np.isnan(S) else np.nan,
        "cG": float(cG),

        "decided_count": int(decided_count),
        "conflict_count": int(conflict_count),
        "reject_count": int(reject_count),
        "nondecision_count": int(nondecision_count),
        "correct_exclusive_count": int(correct_exclusive_count),
        "wrong_exclusive_count": int(wrong_exclusive_count),

        "coverage": float(coverage),
        "conflict_rate": float(conflict_rate),
        "reject_rate": float(reject_rate),
        "nondecision_rate": float(nondecision_rate),
        "correct_exclusive_rate": float(correct_exclusive_rate),
        "wrong_exclusive_rate": float(wrong_exclusive_rate),

        "selective_accuracy": float(selective_accuracy) if not np.isnan(selective_accuracy) else np.nan,
        "selective_error": float(selective_error) if not np.isnan(selective_error) else np.nan,
        "selective_f1": float(selective_f1) if not np.isnan(selective_f1) else np.nan,
        "conservative_accuracy": float(conservative_accuracy) if not np.isnan(conservative_accuracy) else np.nan,
        "conservative_error": float(conservative_error) if not np.isnan(conservative_error) else np.nan,
    }


In [87]:
def _make_noise_like(
    fmin: torch.Tensor,
    fmax: torch.Tensor,
    n: int,
    device: torch.device,
) -> torch.Tensor:
    fmin = fmin.to(device=device, dtype=torch.float32)
    fmax = fmax.to(device=device, dtype=torch.float32)
    u = torch.rand(n, fmin.numel(), device=device, dtype=torch.float32)
    return fmin.unsqueeze(0) + u * (fmax - fmin).unsqueeze(0)


In [88]:
def _train_one_epoch_unary_side(
    model: nn.Module,
    X_real: torch.Tensor,
    fmin: torch.Tensor,
    fmax: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    batch_size: int,
):
    """
    Одна эпоха обучения одной стороны:
    реальные объекты -> target 1
    равномерный шум   -> target 0
    """
    model.train()
    device = X_real.device
    n = int(X_real.shape[0])
    d = int(X_real.shape[1])

    criterion = nn.MSELoss()
    perm = torch.randperm(n, device=device)
    span = (fmax - fmin).to(device=device, dtype=torch.float32)

    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        xb_real = X_real.index_select(0, idx)

        cur_bs = int(xb_real.shape[0])
        u = torch.rand(cur_bs, d, device=device, dtype=torch.float32)
        xb_noise = fmin.unsqueeze(0) + u * span.unsqueeze(0)

        xb = torch.cat([xb_real, xb_noise], dim=0)
        yb = torch.cat(
            [
                torch.ones(cur_bs, device=device, dtype=torch.float32),
                torch.zeros(cur_bs, device=device, dtype=torch.float32),
            ],
            dim=0,
        ).unsqueeze(1)

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

In [289]:
def _select_betas_on_par_fixed_cG(
    X_par: torch.Tensor,
    y_par: torch.Tensor,
    model_pos: nn.Module,
    model_neg: nn.Module,
    fmin: torch.Tensor,
    fmax: torch.Tensor,
    *,
    cG: float,
    checkpoint_n_grid: int = 19,
    h_tau: float = 1e-3,
) -> dict:
    """
    На D_par совместно подбираем ТОЛЬКО beta_pos и beta_neg
    по максимуму S = F12 - cG * G12 при фиксированном cG.
    """
    device = next(model_pos.parameters()).device

    X_par = X_par.to(device=device, dtype=torch.float32)
    y_par = y_par.to(device=device, dtype=torch.int64)

    pos = (y_par == 1)
    neg = (y_par == -1)

    n1 = int(pos.sum().item())
    n2 = int(neg.sum().item())

    if n1 == 0 or n2 == 0:
        raise ValueError("D_par must contain both classes")

    model_pos.eval()
    model_neg.eval()

    with torch.no_grad():
        s1_all = model_pos(X_par).flatten()
        s2_all = model_neg(X_par).flatten()

        X_noise_1 = _make_noise_like(fmin, fmax, n1, device=device)
        X_noise_2 = _make_noise_like(fmin, fmax, n2, device=device)

        s1_no = model_pos(X_noise_1).flatten()
        s2_no = model_neg(X_noise_2).flatten()

        q = torch.linspace(0.05, 0.95, steps=checkpoint_n_grid, device=device)

        pool1 = torch.quantile(torch.cat([s1_all[pos], s1_all[neg], s1_no]), q)
        pool2 = torch.quantile(torch.cat([s2_all[neg], s2_all[pos], s2_no]), q)

    best_S = -np.inf
    best_bp = 0.0
    best_bn = 0.0
    best_metrics = None

    for bp_t in pool1:
        bp = float(bp_t.item())
        m1 = (s1_all >= bp)

        for bn_t in pool2:
            bn = float(bn_t.item())
            m2 = (s2_all >= bn)

            n02_1 = int(((m1 & ~m2) & pos).sum().item())
            n02_2 = int(((m2 & ~m1) & neg).sum().item())

            a1 = n02_1 / n1
            a2 = n02_2 / n2
            F12 = 2.0 * a1 * a2 / (a1 + a2 + h_tau)

            n12_1 = int(((m1 & m2) & pos).sum().item())
            n12_2 = int(((m2 & m1) & neg).sum().item())

            b1 = n12_1 / n1
            b2 = n12_2 / n2
            G12 = 2.0 * b1 * b2 / (b1 + b2 + h_tau)
            #G12 = (b1 + b2) / 2
            #G12 = 0.7 * 2 * b1 * b2 / (b1 + b2 + H_TAU) + 0.3 * (b1 + b2) / 2

            S = F12 - float(cG) * G12

            if S > best_S:
                best_S = float(S)
                best_bp = float(bp)
                best_bn = float(bn)
                best_metrics = {
                    "a1_par": float(a1),
                    "a2_par": float(a2),
                    "b1_par": float(b1),
                    "b2_par": float(b2),
                    "F12_par": float(F12),
                    "G12_par": float(G12),
                }

    return {
        "beta_pos": float(best_bp),
        "beta_neg": float(best_bn),
        "cG": float(cG),
        "S_par": float(best_S),
        **best_metrics,
    }


In [90]:
def _score_fixed_operating_point(
    X_eval: torch.Tensor,
    y_eval: torch.Tensor,
    model_pos: nn.Module,
    model_neg: nn.Module,
    *,
    beta_pos: float,
    beta_neg: float,
    cG: float,
) -> dict:
    """
    Считает полный набор метрик
    на фиксированных beta_pos, beta_neg и фиксированном cG.
    """
    pred = unary_predict_detailed(
        X_eval,
        model_pos,
        model_neg,
        beta_pos=beta_pos,
        beta_neg=beta_neg,
    )
    return strict_unary_metrics(
        y_true=y_eval.detach().cpu().numpy(),
        pred=pred,
        cG=cG,
    )


In [91]:
def train_unary_pair_three_splits(
    X_train_sc: np.ndarray,
    y_train: np.ndarray,
    *,
    d_hidden: int = 64,
    n_hidden_layers: int = 2,
    num_epochs: int = 250,
    batch_size: int = 8,
    lr: float = 3e-3,
    weight_decay: float = 1e-2,
    checkpoint_every: int = 5,
    checkpoint_n_grid: int = 19,
    cG: float = 1.0,
    par_fraction: float = 0.20,
    ckpt_fraction: float = 0.20,
    seed: int = 42,
    device: Optional[str] = None,
) -> dict:
    """
    Разбиение train fold на:
      D_fit   - обучение сети
      D_par   - подбор beta_pos / beta_neg при фиксированном cG
      D_ckpt  - выбор лучшего checkpoint по S_ckpt

    Главный критерий выбора checkpoint:
      S_ckpt = F12_ckpt - cG * G12_ckpt
    """
    X_train_sc = np.asarray(X_train_sc, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64).ravel()

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    if par_fraction <= 0 or ckpt_fraction <= 0 or par_fraction + ckpt_fraction >= 1:
        raise ValueError("Need 0 < par_fraction, ckpt_fraction and par_fraction + ckpt_fraction < 1")

    # 1) отделяем D_ckpt
    sss_ckpt = StratifiedShuffleSplit(
        n_splits=1,
        test_size=ckpt_fraction,
        random_state=seed,
    )
    (fitpar_idx, ckpt_idx), = sss_ckpt.split(X_train_sc, (y_train == 1).astype(int))

    X_fitpar = X_train_sc[fitpar_idx]
    y_fitpar = y_train[fitpar_idx]

    X_ckpt = X_train_sc[ckpt_idx]
    y_ckpt = y_train[ckpt_idx]

    # 2) из оставшегося отделяем D_par
    par_rel = par_fraction / (1.0 - ckpt_fraction)
    sss_par = StratifiedShuffleSplit(
        n_splits=1,
        test_size=par_rel,
        random_state=seed + 1,
    )
    (fit_idx_local, par_idx_local), = sss_par.split(X_fitpar, (y_fitpar == 1).astype(int))

    X_fit = X_fitpar[fit_idx_local]
    y_fit = y_fitpar[fit_idx_local]

    X_par = X_fitpar[par_idx_local]
    y_par = y_fitpar[par_idx_local]

    X_fit_t = torch.as_tensor(X_fit, dtype=torch.float32, device=device)
    y_fit_t = torch.as_tensor(y_fit, dtype=torch.int64, device=device)

    X_par_t = torch.as_tensor(X_par, dtype=torch.float32, device=device)
    y_par_t = torch.as_tensor(y_par, dtype=torch.int64, device=device)

    X_ckpt_t = torch.as_tensor(X_ckpt, dtype=torch.float32, device=device)
    y_ckpt_t = torch.as_tensor(y_ckpt, dtype=torch.int64, device=device)

    ds_fit = TensorDataset(X_fit_t, y_fit_t)
    fmin, fmax = compute_feature_bounds(ds_fit)
    fmin = fmin.to(device=device, dtype=torch.float32)
    fmax = fmax.to(device=device, dtype=torch.float32)

    X_pos_fit = X_fit_t[y_fit_t == 1]
    X_neg_fit = X_fit_t[y_fit_t == -1]

    if X_pos_fit.shape[0] == 0 or X_neg_fit.shape[0] == 0:
        raise ValueError("D_fit must contain both classes")

    d_in = int(X_fit_t.shape[1])

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model_pos = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(device)
    model_neg = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(device)

    opt_pos = torch.optim.AdamW(model_pos.parameters(), lr=lr, weight_decay=weight_decay)
    opt_neg = torch.optim.AdamW(model_neg.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_S_ckpt = -np.inf
    history = []

    for epoch in range(1, num_epochs + 1):
        _train_one_epoch_unary_side(
            model=model_pos,
            X_real=X_pos_fit,
            fmin=fmin,
            fmax=fmax,
            optimizer=opt_pos,
            batch_size=batch_size,
        )
        _train_one_epoch_unary_side(
            model=model_neg,
            X_real=X_neg_fit,
            fmin=fmin,
            fmax=fmax,
            optimizer=opt_neg,
            batch_size=batch_size,
        )

        if epoch % checkpoint_every != 0 and epoch != num_epochs:
            continue

        op_par = _select_betas_on_par_fixed_cG(
            X_par_t,
            y_par_t,
            model_pos,
            model_neg,
            fmin=fmin,
            fmax=fmax,
            cG=cG,
            checkpoint_n_grid=checkpoint_n_grid,
        )

        m_ckpt = _score_fixed_operating_point(
            X_ckpt_t,
            y_ckpt_t,
            model_pos,
            model_neg,
            beta_pos=op_par["beta_pos"],
            beta_neg=op_par["beta_neg"],
            cG=float(cG),
        )

        row = {
            "epoch": int(epoch),
            "beta_pos": float(op_par["beta_pos"]),
            "beta_neg": float(op_par["beta_neg"]),
            "cG": float(cG),

            "S_par": float(op_par["S_par"]),
            "F12_par": float(op_par["F12_par"]),
            "G12_par": float(op_par["G12_par"]),
            "a1_par": float(op_par["a1_par"]),
            "a2_par": float(op_par["a2_par"]),
            "b1_par": float(op_par["b1_par"]),
            "b2_par": float(op_par["b2_par"]),

            "S_ckpt": float(m_ckpt["S"]),
            "F12_ckpt": float(m_ckpt["F12"]),
            "G12_ckpt": float(m_ckpt["G12"]),
            "a1_ckpt": float(m_ckpt["a1"]),
            "a2_ckpt": float(m_ckpt["a2"]),
            "b1_ckpt": float(m_ckpt["b1"]),
            "b2_ckpt": float(m_ckpt["b2"]),

            "coverage_ckpt": float(m_ckpt["coverage"]),
            "reject_rate_ckpt": float(m_ckpt["reject_rate"]),
            "conflict_rate_ckpt": float(m_ckpt["conflict_rate"]),
            "wrong_exclusive_rate_ckpt": float(m_ckpt["wrong_exclusive_rate"]),
            "selective_accuracy_ckpt": (
                float(m_ckpt["selective_accuracy"])
                if not np.isnan(m_ckpt["selective_accuracy"]) else np.nan
            ),
            "conservative_accuracy_ckpt": float(m_ckpt["conservative_accuracy"]),
        }
        history.append(row)

        if row["S_ckpt"] > best_S_ckpt:
            best_S_ckpt = row["S_ckpt"]
            best_state = {
                "epoch": int(epoch),
                "beta_pos": float(op_par["beta_pos"]),
                "beta_neg": float(op_par["beta_neg"]),
                "cG": float(cG),

                "S_par": float(op_par["S_par"]),
                "F12_par": float(op_par["F12_par"]),
                "G12_par": float(op_par["G12_par"]),

                "S_ckpt": float(m_ckpt["S"]),
                "F12_ckpt": float(m_ckpt["F12"]),
                "G12_ckpt": float(m_ckpt["G12"]),
                "a1_ckpt": float(m_ckpt["a1"]),
                "a2_ckpt": float(m_ckpt["a2"]),
                "b1_ckpt": float(m_ckpt["b1"]),
                "b2_ckpt": float(m_ckpt["b2"]),
                "coverage_ckpt": float(m_ckpt["coverage"]),
                "reject_rate_ckpt": float(m_ckpt["reject_rate"]),
                "conflict_rate_ckpt": float(m_ckpt["conflict_rate"]),
                "wrong_exclusive_rate_ckpt": float(m_ckpt["wrong_exclusive_rate"]),
                "selective_accuracy_ckpt": (
                    float(m_ckpt["selective_accuracy"])
                    if not np.isnan(m_ckpt["selective_accuracy"]) else np.nan
                ),
                "conservative_accuracy_ckpt": float(m_ckpt["conservative_accuracy"]),

                "model_pos_state": copy.deepcopy(model_pos.state_dict()),
                "model_neg_state": copy.deepcopy(model_neg.state_dict()),
            }

    if best_state is None:
        raise RuntimeError("No checkpoint was saved. Increase num_epochs or change checkpoint_every.")

    best_model_pos = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(device)
    best_model_neg = MLP(d_in=d_in, d_hidden=d_hidden, n_hidden_layers=n_hidden_layers).to(device)

    best_model_pos.load_state_dict(best_state["model_pos_state"])
    best_model_neg.load_state_dict(best_state["model_neg_state"])

    best_model_pos.eval()
    best_model_neg.eval()

    return {
        "model_pos": best_model_pos,
        "model_neg": best_model_neg,
        "beta_pos": float(best_state["beta_pos"]),
        "beta_neg": float(best_state["beta_neg"]),
        "cG": float(best_state["cG"]),
        "best_epoch": int(best_state["epoch"]),

        "best_S_par": float(best_state["S_par"]),
        "best_F12_par": float(best_state["F12_par"]),
        "best_G12_par": float(best_state["G12_par"]),

        "best_S_ckpt": float(best_state["S_ckpt"]),
        "best_F12_ckpt": float(best_state["F12_ckpt"]),
        "best_G12_ckpt": float(best_state["G12_ckpt"]),

        "best_a1_ckpt": float(best_state["a1_ckpt"]),
        "best_a2_ckpt": float(best_state["a2_ckpt"]),
        "best_b1_ckpt": float(best_state["b1_ckpt"]),
        "best_b2_ckpt": float(best_state["b2_ckpt"]),

        "coverage_ckpt": float(best_state["coverage_ckpt"]),
        "reject_rate_ckpt": float(best_state["reject_rate_ckpt"]),
        "conflict_rate_ckpt": float(best_state["conflict_rate_ckpt"]),
        "wrong_exclusive_rate_ckpt": float(best_state["wrong_exclusive_rate_ckpt"]),
        "selective_accuracy_ckpt": best_state["selective_accuracy_ckpt"],
        "conservative_accuracy_ckpt": float(best_state["conservative_accuracy_ckpt"]),

        "history": pd.DataFrame(history),
        "fit_size": int(len(X_fit)),
        "par_size": int(len(X_par)),
        "ckpt_size": int(len(X_ckpt)),
    }

In [92]:
def run_unary_on_saved_folds_strict(
    results_dir: str,
    *,
    d_hidden: int = 64,
    n_hidden_layers: int = 2,
    num_epochs: int = 250,
    batch_size: int = 8,
    lr: float = 3e-3,
    weight_decay: float = 1e-2,
    checkpoint_every: int = 5,
    checkpoint_n_grid: int = 19,
    cG: float = 1.0,
    par_fraction: float = 0.20,
    ckpt_fraction: float = 0.20,
    seed: int = 42,
    device: Optional[str] = None,
    save_name: str = "unary_strict_results.csv",
) -> pd.DataFrame:
    """
    Читает fold_*_summary.json, обучает unary-pair на выбранных координатах,
    на test считает criterion-aligned и интерпретационные метрики.

    cG здесь фиксирован и не подбирается.
    """
    results_path = Path(results_dir)
    if not results_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {results_dir}")

    fold_files = sorted(results_path.glob("fold_*_summary.json"))
    if len(fold_files) == 0:
        raise FileNotFoundError(f"No fold_*_summary.json found in {results_dir}")

    X, y = make_dataset_colon()
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=int).ravel()

    n = X.shape[0]
    all_idx = np.arange(n, dtype=int)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    detail_dir = results_path / "unary_strict_details"
    detail_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    all_test_indices = []

    for fold_file in fold_files:
        with open(fold_file, "r", encoding="utf-8") as f:
            fold_info = json.load(f)

        fold_id = int(fold_info["fold"])
        test_idx = np.asarray(fold_info["test_indices"], dtype=int)
        selected_indices = np.asarray(fold_info["selected_indices"], dtype=int)

        train_idx = np.setdiff1d(all_idx, test_idx, assume_unique=False)

        if np.intersect1d(train_idx, test_idx).size != 0:
            raise ValueError(f"Fold {fold_id}: overlap between train and test")

        X_train = X[train_idx][:, selected_indices]
        X_test = X[test_idx][:, selected_indices]
        y_train = y[train_idx]
        y_test = y[test_idx]

        all_test_indices.extend(test_idx.tolist())

        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)

        trained = train_unary_pair_three_splits(
            X_train_sc,
            y_train,
            d_hidden=d_hidden,
            n_hidden_layers=n_hidden_layers,
            num_epochs=num_epochs,
            batch_size=batch_size,
            lr=lr,
            weight_decay=weight_decay,
            checkpoint_every=checkpoint_every,
            checkpoint_n_grid=checkpoint_n_grid,
            cG=cG,
            par_fraction=par_fraction,
            ckpt_fraction=ckpt_fraction,
            seed=seed + fold_id,
            device=device,
        )

        X_test_t = torch.as_tensor(X_test_sc, dtype=torch.float32)
        pred = unary_predict_detailed(
            X_test_t,
            trained["model_pos"],
            trained["model_neg"],
            beta_pos=trained["beta_pos"],
            beta_neg=trained["beta_neg"],
        )

        m = strict_unary_metrics(
            y_true=y_test,
            pred=pred,
            cG=float(cG),
        )

        row = {
            "fold": fold_id,
            "train_size": int(train_idx.size),
            "test_size": int(test_idx.size),
            "n_selected": int(selected_indices.size),
            "selected_indices": ",".join(map(str, selected_indices.tolist())),

            "fit_size_inner": int(trained["fit_size"]),
            "par_size_inner": int(trained["par_size"]),
            "ckpt_size_inner": int(trained["ckpt_size"]),

            "best_epoch": int(trained["best_epoch"]),
            "beta_pos": float(trained["beta_pos"]),
            "beta_neg": float(trained["beta_neg"]),
            "cG": float(trained["cG"]),

            "S_par": float(trained["best_S_par"]),
            "F12_par": float(trained["best_F12_par"]),
            "G12_par": float(trained["best_G12_par"]),

            "S_ckpt": float(trained["best_S_ckpt"]),
            "F12_ckpt": float(trained["best_F12_ckpt"]),
            "G12_ckpt": float(trained["best_G12_ckpt"]),

            "S_test": float(m["S"]),
            "F12_test": float(m["F12"]),
            "G12_test": float(m["G12"]),
            "a1_test": float(m["a1"]),
            "a2_test": float(m["a2"]),
            "b1_test": float(m["b1"]),
            "b2_test": float(m["b2"]),

            "decided_count": int(m["decided_count"]),
            "conflict_count": int(m["conflict_count"]),
            "reject_count": int(m["reject_count"]),
            "wrong_exclusive_count": int(m["wrong_exclusive_count"]),
            "correct_exclusive_count": int(m["correct_exclusive_count"]),

            "coverage": float(m["coverage"]),
            "conflict_rate": float(m["conflict_rate"]),
            "reject_rate": float(m["reject_rate"]),
            "wrong_exclusive_rate": float(m["wrong_exclusive_rate"]),
            "correct_exclusive_rate": float(m["correct_exclusive_rate"]),

            "selective_accuracy": m["selective_accuracy"],
            "selective_error": m["selective_error"],
            "selective_f1": m["selective_f1"],
            "conservative_accuracy": float(m["conservative_accuracy"]),
            "conservative_error": float(m["conservative_error"]),
        }
        rows.append(row)

        hist_path = detail_dir / f"fold_{fold_id:02d}_checkpoint_history.csv"
        trained["history"].to_csv(hist_path, index=False)

        fold_detail = {
            "fold": fold_id,
            "train_indices": train_idx.tolist(),
            "test_indices": test_idx.tolist(),
            "selected_indices": selected_indices.tolist(),

            "fit_size_inner": int(trained["fit_size"]),
            "par_size_inner": int(trained["par_size"]),
            "ckpt_size_inner": int(trained["ckpt_size"]),

            "best_epoch": int(trained["best_epoch"]),
            "beta_pos": float(trained["beta_pos"]),
            "beta_neg": float(trained["beta_neg"]),
            "cG": float(trained["cG"]),

            "S_par": float(trained["best_S_par"]),
            "F12_par": float(trained["best_F12_par"]),
            "G12_par": float(trained["best_G12_par"]),

            "S_ckpt": float(trained["best_S_ckpt"]),
            "F12_ckpt": float(trained["best_F12_ckpt"]),
            "G12_ckpt": float(trained["best_G12_ckpt"]),

            "S_test": float(m["S"]),
            "F12_test": float(m["F12"]),
            "G12_test": float(m["G12"]),
            "a1_test": float(m["a1"]),
            "a2_test": float(m["a2"]),
            "b1_test": float(m["b1"]),
            "b2_test": float(m["b2"]),

            "decided_count": int(m["decided_count"]),
            "conflict_count": int(m["conflict_count"]),
            "reject_count": int(m["reject_count"]),
            "wrong_exclusive_count": int(m["wrong_exclusive_count"]),
            "correct_exclusive_count": int(m["correct_exclusive_count"]),

            "coverage": float(m["coverage"]),
            "conflict_rate": float(m["conflict_rate"]),
            "reject_rate": float(m["reject_rate"]),
            "wrong_exclusive_rate": float(m["wrong_exclusive_rate"]),
            "correct_exclusive_rate": float(m["correct_exclusive_rate"]),

            "selective_accuracy": m["selective_accuracy"],
            "selective_error": m["selective_error"],
            "selective_f1": m["selective_f1"],
            "conservative_accuracy": float(m["conservative_accuracy"]),
            "conservative_error": float(m["conservative_error"]),

            "checkpoint_history_file": str(hist_path),
        }

        with open(detail_dir / f"fold_{fold_id:02d}_strict_unary_detail.json", "w", encoding="utf-8") as f:
            json.dump(fold_detail, f, ensure_ascii=False, indent=2)

        print("=" * 90)
        print(f"Fold {fold_id}")
        print(f"selected_indices            : {selected_indices.tolist()}")
        print(f"best_epoch                  : {trained['best_epoch']}")
        print(f"cG                          : {trained['cG']:.6f}")
        print(f"S_test                      : {m['S']:.6f}")
        print(f"F12_test                    : {m['F12']:.6f}")
        print(f"G12_test                    : {m['G12']:.6f}")
        print(f"a1_test, a2_test            : {m['a1']:.6f}, {m['a2']:.6f}")
        print(f"b1_test, b2_test            : {m['b1']:.6f}, {m['b2']:.6f}")
        print(f"coverage                    : {m['coverage']:.6f}")
        print(f"conflict_rate               : {m['conflict_rate']:.6f}")
        print(f"reject_rate                 : {m['reject_rate']:.6f}")
        print(f"wrong_exclusive_rate        : {m['wrong_exclusive_rate']:.6f}")
        print(f"selective_accuracy          : {m['selective_accuracy']}")
        print(f"selective_f1                : {m['selective_f1']}")
        print(f"conservative_accuracy       : {m['conservative_accuracy']:.6f}")
        print(f"conservative_error          : {m['conservative_error']:.6f}")

    df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    all_test_indices = np.asarray(all_test_indices, dtype=int)
    uniq_test = np.unique(all_test_indices)
    if uniq_test.size != n:
        print(f"WARNING: union of test indices over folds has size {uniq_test.size}, expected {n}")
    if all_test_indices.size != n:
        print(f"WARNING: total collected test indices = {all_test_indices.size}, expected {n}")

    metric_cols = [
        "train_size", "test_size", "n_selected",
        "fit_size_inner", "par_size_inner", "ckpt_size_inner",
        "best_epoch", "beta_pos", "beta_neg", "cG",
        "S_par", "F12_par", "G12_par",
        "S_ckpt", "F12_ckpt", "G12_ckpt",
        "S_test", "F12_test", "G12_test",
        "a1_test", "a2_test", "b1_test", "b2_test",
        "decided_count", "conflict_count", "reject_count",
        "wrong_exclusive_count", "correct_exclusive_count",
        "coverage", "conflict_rate", "reject_rate",
        "wrong_exclusive_rate", "correct_exclusive_rate",
        "selective_accuracy", "selective_error", "selective_f1",
        "conservative_accuracy", "conservative_error",
    ]

    mean_row = {"fold": "mean", "selected_indices": ""}
    std_row = {"fold": "std", "selected_indices": ""}

    for col in metric_cols:
        mean_row[col] = float(df[col].mean(skipna=True))
        std_row[col] = float(df[col].std(ddof=1, skipna=True))

    df_full = pd.concat([df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

    out_csv = results_path / save_name
    df_full.to_csv(out_csv, index=False)

    print("=" * 90)
    print("STRICT UNARY CV FINISHED")
    print(f"Fixed cG                   : {cG:.6f}")
    print(f"Mean S_test                : {df['S_test'].mean(skipna=True):.6f}")
    print(f"Mean F12_test              : {df['F12_test'].mean(skipna=True):.6f}")
    print(f"Mean G12_test              : {df['G12_test'].mean(skipna=True):.6f}")
    print(f"Mean coverage              : {df['coverage'].mean(skipna=True):.6f}")
    print(f"Mean conflict_rate         : {df['conflict_rate'].mean(skipna=True):.6f}")
    print(f"Mean reject_rate           : {df['reject_rate'].mean(skipna=True):.6f}")
    print(f"Mean selective_accuracy    : {df['selective_accuracy'].mean(skipna=True):.6f}")
    print(f"Mean selective_f1          : {df['selective_f1'].mean(skipna=True):.6f}")
    print(f"Mean conservative_accuracy : {df['conservative_accuracy'].mean(skipna=True):.6f}")
    print(f"Saved to                   : {out_csv}")

    return df_full

In [268]:
df_strict = run_unary_on_saved_folds_strict(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    d_hidden=64,
    n_hidden_layers=2,
    num_epochs=250,
    batch_size=8,
    checkpoint_every=5,
    checkpoint_n_grid=19,
    cG=0.25,
    par_fraction=0.20,
    ckpt_fraction=0.20,
    seed=42,
)
print(df_strict)

Fold 1
selected_indices            : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
best_epoch                  : 50
cG                          : 0.250000
S_test                      : 0.633595
F12_test                    : 0.705384
G12_test                    : 0.287158
a1_test, a2_test            : 0.666667, 0.750000
b1_test, b2_test            : 0.333333, 0.250000
coverage                    : 0.714286
conflict_rate               : 0.285714
reject_rate                 : 0.000000
wrong_exclusive_rate        : 0.000000
selective_accuracy          : 1.0
selective_f1                : 1.0
conservative_accuracy       : 0.714286
conservative_error          : 0.285714
Fold 2
selected_indices            : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
best_epoch                  : 45
cG                          : 0.250000
S_test                      : 0.000000
F12_test                    : 0.000000
G12_test                    : 0.0000

In [ ]:
def run_all_A_real(
    include_unary: bool = False,
    unary_params: dict | None = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    configs = [
        ("DS-A-real", make_dataset_A_real, [2, 3, 5]),
    ]

    methods = [
        ("PCA",        lambda q, **_: PCAReducer(q)),
        ("PLS",        lambda q, **_: PLSReducer(q)),
        ("UMAP_sup",   lambda q, **_: UMAPReducer(q)),
        ("HSIC_Lasso", lambda q, **_: HSICSelector(q)),
        ("MLP_unar",   lambda q, **kw: MLPReducer(
            d_hidden=64,
            n_hidden_layers=2,
            num_epochs=40,
            batch_size=256,
            n_select=q,
            seed=0,
            output=False,
            **kw,
        )),
    ]

    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    else:
        DEVICE = torch.device("cpu")

    print("Using device:", DEVICE)
    if DEVICE.type == "cuda":
        print("CUDA name:", torch.cuda.get_device_name(0))

    base_unary_params = dict(
        d_hidden=32,
        n_hidden_layers=1,
        num_epochs=50,
        batch_size=256,
    )
    if unary_params is not None:
        base_unary_params.update(unary_params)
    base_unary_params["device"] = str(DEVICE) 

    rows_cls_all: List[Dict[str, Any]] = []

    for dname, maker, qgrid in configs:
        X, y = maker()  
        for name, ctor in methods:
            for q in qgrid:
                if name == "MLP_unar":
                    reducer_kwargs = {"device": str(DEVICE)} 
                else:
                    reducer_kwargs = {}

                cls_rows, _un_rows = evaluate_reduction(
                    X, y,
                    reducer_name=name,
                    reducer_ctor=ctor,
                    q=q,
                    dataset_name=dname,
                    reducer_kwargs=reducer_kwargs,
                    compute_unary_on_reduced=include_unary,
                    unary_params=base_unary_params,
                )
                rows_cls_all.extend(cls_rows)

    df = pd.DataFrame(rows_cls_all)

    for col in ["AUC", "PR_AUC", "ΔAUC", "ΔPR_AUC",
                "t_reducer", "t_model", "t_model_base"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    agg = (
        df.groupby(["dataset", "method", "q", "model"], dropna=False)
          .agg(
              AUC_mean=("AUC", "mean"),
              PR_AUC_mean=("PR_AUC", "mean"),
              dAUC_mean=("ΔAUC", "mean"),
              dPR_mean=("ΔPR_AUC", "mean"),
              t_red_med=("t_reducer", "median"),
              t_mod_med=("t_model", "median"),
          )
          .reset_index()
          .sort_values(["dataset", "model", "dAUC_mean"], ascending=[True, True, False])
    )

    return df, agg

In [ ]:
df_A_real, agg_A_real = run_all_A_real()

In [ ]:
agg_A_real

In [47]:
def run_all_B_real(
    include_unary: bool = True,
    unary_params: dict | None = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    configs = [
        ("DS-B-real", make_dataset_B_real, [5, 10, 15]),
    ]

    methods = [
        ("PCA",        lambda q, **_: PCAReducer(q)),
        ("PLS",        lambda q, **_: PLSReducer(q)),
        ("UMAP_sup",   lambda q, **_: UMAPReducer(q)),
        ("HSIC_Lasso", lambda q, **_: HSICSelector(q)),
        ("Ensembleunar", lambda q, **kw: EnsembleReducer(
            d_hidden=32,
            n_hidden_layers=2,
            num_epochs=8,
            batch_size=1024,
            n_select=q,
            num_models=4,
            num_attempts=20,
            num_coords=4,
            num_samples=10,
            seed=42,
            output=True,
            **kw,
        )),
    ]

    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
    else:
        DEVICE = torch.device("cpu")

    print("Using device:", DEVICE)
    if DEVICE.type == "cuda":
        print("CUDA name:", torch.cuda.get_device_name(0))

    base_unary_params = dict(
        d_hidden=32,
        n_hidden_layers=1,
        num_epochs=50,
        batch_size=256,
    )
    if unary_params is not None:
        base_unary_params.update(unary_params)
    base_unary_params["device"] = str(DEVICE)

    rows_cls_all: List[Dict[str, Any]] = []

    for dname, maker, qgrid in configs:
        X, y = maker() 
        for name, ctor in methods:
            for q in qgrid:
                if name == "Ensembleunar":
                    reducer_kwargs = {
                        "device": str(DEVICE), 
                        "use_gpu_shap": True,
                    }
                else:
                    reducer_kwargs = {}

                cls_rows, _un_rows = evaluate_reduction(
                    X, y,
                    reducer_name=name,
                    reducer_ctor=ctor,
                    q=q,
                    dataset_name=dname,
                    reducer_kwargs=reducer_kwargs,
                    compute_unary_on_reduced=include_unary,
                    unary_params=base_unary_params,
                )
                rows_cls_all.extend(cls_rows)

    df = pd.DataFrame(rows_cls_all)

    for col in [
        "AUC", "PR_AUC", "ΔAUC", "ΔPR_AUC",
        "t_reducer", "t_model", "t_model_base"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    agg = (
        df.groupby(["dataset", "method", "q", "model"], dropna=False)
          .agg(
              AUC_mean=("AUC", "mean"),
              PR_AUC_mean=("PR_AUC", "mean"),
              dAUC_mean=("ΔAUC", "mean"),
              dPR_mean=("ΔPR_AUC", "mean"),
              t_red_med=("t_reducer", "median"),
              t_mod_med=("t_model", "median"),
          )
          .reset_index()
          .sort_values(
              ["dataset", "model", "dAUC_mean"],
              ascending=[True, True, False],
          )
    )

    results = {
        "df": df,
        "agg": agg,
    }
    with open("results_B_real", "wb") as f:
        pickle.dump(results, f)

    return df, agg

In [ ]:
df_B_real, agg_B_real = run_all_B_real()

In [35]:
def _classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    Метрики для обычной бинарной классификации.
    Предполагается, что y_true, y_pred лежат в {-1, 1}.
    """
    acc = float(accuracy_score(y_true, y_pred))
    err = float(1.0 - acc)
    f1 = float(f1_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0))

    return {
        "accuracy": acc,
        "classification_error": err,
        "f1_score": f1,
    }

In [36]:
def _run_classifier_on_saved_folds(
    results_dir: str,
    *,
    model_builder: Callable[[int], Any],
    model_name: str,
    seed: int = 42,
    save_name: Optional[str] = None,
) -> pd.DataFrame:
    """
    Общий раннер:
    - читает fold_*_summary.json из results_dir,
    - на каждом фолде обучает обычный классификатор на selected_indices,
    - считает accuracy / classification_error / f1_score,
    - сохраняет csv и json-детали.
    """
    results_path = Path(results_dir)
    if not results_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {results_dir}")

    fold_files = sorted(results_path.glob("fold_*_summary.json"))
    if len(fold_files) == 0:
        raise FileNotFoundError(f"No fold_*_summary.json found in {results_dir}")

    if save_name is None:
        save_name = f"{model_name.lower()}_results.csv"

    X, y = make_dataset_colon()
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=int).ravel()

    uniq = np.unique(y)
    if set(uniq) != {-1, 1}:
        raise ValueError(f"Expected y in {{-1, 1}}, got {uniq}")

    n = X.shape[0]
    all_idx = np.arange(n, dtype=int)

    detail_dir = results_path / f"{model_name.lower()}_details"
    detail_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    all_test_indices = []

    for fold_file in fold_files:
        with open(fold_file, "r", encoding="utf-8") as f:
            fold_info = json.load(f)

        fold_id = int(fold_info["fold"])
        test_idx = np.asarray(fold_info["test_indices"], dtype=int)
        selected_indices = np.asarray(fold_info["selected_indices"], dtype=int)

        train_idx = np.setdiff1d(all_idx, test_idx, assume_unique=False)

        if np.intersect1d(train_idx, test_idx).size != 0:
            raise ValueError(f"Fold {fold_id}: overlap between train and test")

        X_train = X[train_idx][:, selected_indices]
        X_test = X[test_idx][:, selected_indices]
        y_train = y[train_idx]
        y_test = y[test_idx]

        all_test_indices.extend(test_idx.tolist())

        model = model_builder(seed + fold_id)

        # Масштабирование делаем внутри pipeline, чтобы не было утечки.
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model),
        ])

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        m = _classification_metrics(y_true=y_test, y_pred=y_pred)

        row = {
            "fold": fold_id,
            "train_size": int(train_idx.size),
            "test_size": int(test_idx.size),
            "n_selected": int(selected_indices.size),
            "selected_indices": ",".join(map(str, selected_indices.tolist())),

            "accuracy": float(m["accuracy"]),
            "classification_error": float(m["classification_error"]),
            "f1_score": float(m["f1_score"]),
        }
        rows.append(row)

        fold_detail = {
            "fold": fold_id,
            "model_name": model_name,
            "train_indices": train_idx.tolist(),
            "test_indices": test_idx.tolist(),
            "selected_indices": selected_indices.tolist(),

            "train_size": int(train_idx.size),
            "test_size": int(test_idx.size),
            "n_selected": int(selected_indices.size),

            "accuracy": float(m["accuracy"]),
            "classification_error": float(m["classification_error"]),
            "f1_score": float(m["f1_score"]),
        }

        with open(detail_dir / f"fold_{fold_id:02d}_{model_name.lower()}_detail.json", "w", encoding="utf-8") as f:
            json.dump(fold_detail, f, ensure_ascii=False, indent=2)

        print("=" * 90)
        print(f"{model_name} | Fold {fold_id}")
        print(f"selected_indices       : {selected_indices.tolist()}")
        print(f"accuracy               : {m['accuracy']:.6f}")
        print(f"classification_error   : {m['classification_error']:.6f}")
        print(f"f1_score               : {m['f1_score']:.6f}")

    df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    all_test_indices = np.asarray(all_test_indices, dtype=int)
    uniq_test = np.unique(all_test_indices)
    if uniq_test.size != n:
        print(f"WARNING: union of test indices over folds has size {uniq_test.size}, expected {n}")
    if all_test_indices.size != n:
        print(f"WARNING: total collected test indices = {all_test_indices.size}, expected {n}")

    metric_cols = [
        "train_size",
        "test_size",
        "n_selected",
        "accuracy",
        "classification_error",
        "f1_score",
    ]

    mean_row = {"fold": "mean", "selected_indices": ""}
    std_row = {"fold": "std", "selected_indices": ""}

    for col in metric_cols:
        mean_row[col] = float(df[col].mean(skipna=True))
        std_row[col] = float(df[col].std(ddof=1, skipna=True))

    df_full = pd.concat([df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

    out_csv = results_path / save_name
    df_full.to_csv(out_csv, index=False)

    print("=" * 90)
    print(f"{model_name} CV FINISHED")
    print(f"Mean accuracy             : {df['accuracy'].mean(skipna=True):.6f}")
    print(f"Mean classification_error : {df['classification_error'].mean(skipna=True):.6f}")
    print(f"Mean f1_score             : {df['f1_score'].mean(skipna=True):.6f}")
    print(f"Saved to                  : {out_csv}")

    return df_full

In [37]:

def run_svm_on_saved_folds(
    results_dir: str,
    *,
    C: float = 1.0,
    kernel: str = "linear",
    gamma: str | float = "scale",
    class_weight: Optional[dict[int, float] | str] = None,
    seed: int = 42,
    save_name: str = "svm_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для SVM.

    По умолчанию стоит linear SVM, что для вашей задачи отбора признаков
    обычно наиболее естественно как базовый ориентир.
    """
    def build_model(fold_seed: int) -> SVC:
        return SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            class_weight=class_weight,
            random_state=fold_seed,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="SVM",
        seed=seed,
        save_name=save_name,
    )

In [38]:



def run_knn_on_saved_folds(
    results_dir: str,
    *,
    n_neighbors: int = 5,
    weights: str = "uniform",
    metric: str = "minkowski",
    p: int = 2,
    seed: int = 42,
    save_name: str = "knn_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для KNN.

    seed оставлен для единообразия интерфейса, хотя сам KNN детерминирован.
    """
    def build_model(_: int) -> KNeighborsClassifier:
        return KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            metric=metric,
            p=p,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="KNN",
        seed=seed,
        save_name=save_name,
    )

In [269]:
df_svm_rbf = run_svm_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    C=1.0,
    kernel="rbf",
    gamma='scale',
    seed=42,
)


SVM | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
SVM | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.857143
classification_error   : 0.142857
f1_score               : 0.800000
SVM | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 0.833333
classification_error   : 0.166667
f1_score               : 0.800000
SVM | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
SVM | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 1230, 1548, 1647, 1678, 1892]
accuracy            

In [270]:
df_knn = run_knn_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    n_neighbors=5,
    weights="uniform",
    seed=42,
)

KNN | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
KNN | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.857143
classification_error   : 0.142857
f1_score               : 0.800000
KNN | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 0.833333
classification_error   : 0.166667
f1_score               : 0.800000
KNN | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
KNN | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 1230, 1548, 1647, 1678, 1892]
accuracy            

In [293]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import KernelDensity

class KDEBayesClassifier(BaseEstimator, ClassifierMixin):
    """
    Простая байесовская классификация через поклассовые KDE.

    Для каждого класса y in classes_ оценивается log p(x | y),
    затем добавляется log prior(y).
    """

    def __init__(
        self,
        bandwidth: float = 1.0,
        kernel: str = "gaussian",
        priors: Optional[dict[int, float]] = None,
    ) -> None:
        self.bandwidth = bandwidth
        self.kernel = kernel
        self.priors = priors

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KDEBayesClassifier":
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        if self.classes_.size < 2:
            raise ValueError("KDEBayesClassifier requires at least 2 classes.")

        self.models_ = {}
        self.log_priors_ = {}

        n = len(y)
        for cls in self.classes_:
            Xc = X[y == cls]
            if Xc.shape[0] == 0:
                raise ValueError(f"Empty class encountered: {cls}")

            kde = KernelDensity(
                bandwidth=self.bandwidth,
                kernel=self.kernel,
            )
            kde.fit(Xc)
            self.models_[cls] = kde

            if self.priors is None:
                prior = Xc.shape[0] / n
            else:
                if cls not in self.priors:
                    raise ValueError(f"Prior for class {cls} is missing in priors.")
                prior = float(self.priors[cls])

            if prior <= 0:
                raise ValueError(f"Prior must be positive, got {prior} for class {cls}.")

            self.log_priors_[cls] = float(np.log(prior))

        return self

    def _joint_log_posteriors(self, X: np.ndarray) -> np.ndarray:
        """
        Возвращает матрицу [n_samples, n_classes]:
            log p(x | y=c) + log p(y=c)
        """
        X = np.asarray(X, dtype=np.float64)

        log_post_cols = []
        for cls in self.classes_:
            log_lik = self.models_[cls].score_samples(X)
            log_post = log_lik + self.log_priors_[cls]
            log_post_cols.append(log_post)

        return np.column_stack(log_post_cols)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Возвращает апостериорные вероятности классов.
        """
        log_post = self._joint_log_posteriors(X)

        # стабильный softmax построчно
        row_max = np.max(log_post, axis=1, keepdims=True)
        exp_shifted = np.exp(log_post - row_max)
        probs = exp_shifted / exp_shifted.sum(axis=1, keepdims=True)
        return probs

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        """
        Для бинарного случая возвращает score положительного класса
        как разность log-posteriors:
            score = log_post(class_1) - log_post(class_0)
        """
        log_post = self._joint_log_posteriors(X)

        if log_post.shape[1] != 2:
            raise ValueError("decision_function is implemented only for binary classification.")

        return log_post[:, 1] - log_post[:, 0]

    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        best_idx = np.argmax(probs, axis=1)
        return self.classes_[best_idx]

In [105]:
def run_kde_bayes_on_saved_folds(
    results_dir: str,
    *,
    bandwidth: float = 1.0,
    kernel: str = "gaussian",
    priors: Optional[dict[int, float]] = None,
    seed: int = 42,
    save_name: str = "kde_bayes_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для KDE-Bayes.
    """

    def build_model(_: int) -> KDEBayesClassifier:
        return KDEBayesClassifier(
            bandwidth=bandwidth,
            kernel=kernel,
            priors=priors,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="KDE_Bayes",
        seed=seed,
        save_name=save_name,
    )

In [106]:
def run_qda_on_saved_folds(
    results_dir: str,
    *,
    reg_param: float = 0.0,
    priors: Optional[list[float]] = None,
    seed: int = 42,
    save_name: str = "qda_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для QDA.
    """

    def build_model(_: int) -> QuadraticDiscriminantAnalysis:
        return QuadraticDiscriminantAnalysis(
            priors=priors,
            reg_param=reg_param,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="QDA",
        seed=seed,
        save_name=save_name,
    )

In [107]:
def run_shrinkage_lda_on_saved_folds(
    results_dir: str,
    *,
    shrinkage: str | float = "auto",
    solver: str = "lsqr",
    priors: Optional[list[float]] = None,
    seed: int = 42,
    save_name: str = "shrinkage_lda_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для shrinkage-LDA.

    Важно:
    - shrinkage работает только с solver='lsqr' или solver='eigen'.
    """

    if solver not in {"lsqr", "eigen"}:
        raise ValueError("For shrinkage-LDA, solver must be 'lsqr' or 'eigen'.")

    def build_model(_: int) -> LinearDiscriminantAnalysis:
        return LinearDiscriminantAnalysis(
            solver=solver,
            shrinkage=shrinkage,
            priors=priors,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="Shrinkage_LDA",
        seed=seed,
        save_name=save_name,
    )

In [108]:
def run_gaussiannb_on_saved_folds(
    results_dir: str,
    *,
    var_smoothing: float = 1e-9,
    priors: Optional[list[float]] = None,
    seed: int = 42,
    save_name: str = "gaussiannb_results.csv",
) -> pd.DataFrame:
    """
    Запуск по всем сохранённым фолдам для GaussianNB.
    """

    def build_model(_: int) -> GaussianNB:
        return GaussianNB(
            priors=priors,
            var_smoothing=var_smoothing,
        )

    return _run_classifier_on_saved_folds(
        results_dir=results_dir,
        model_builder=build_model,
        model_name="GaussianNB",
        seed=seed,
        save_name=save_name,
    )

In [271]:
df_kde = run_kde_bayes_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    bandwidth=0.5,
    seed=42,
)

KDE_Bayes | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
KDE_Bayes | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.857143
classification_error   : 0.142857
f1_score               : 0.800000
KDE_Bayes | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
KDE_Bayes | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
KDE_Bayes | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 1230, 1548, 1647, 16

In [272]:
df_qda = run_qda_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    reg_param=0.0,
    seed=42,
)

QDA | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 0.571429
classification_error   : 0.428571
f1_score               : 0.000000
QDA | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.714286
classification_error   : 0.285714
f1_score               : 0.500000
QDA | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
QDA | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 0.833333
classification_error   : 0.166667
f1_score               : 0.666667
QDA | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 1230, 1548, 1647, 1678, 1892]
accuracy            

In [273]:
df_lda = run_shrinkage_lda_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    shrinkage="auto",
    solver="lsqr",
    seed=42,
)

Shrinkage_LDA | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
Shrinkage_LDA | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.857143
classification_error   : 0.142857
f1_score               : 0.800000
Shrinkage_LDA | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 0.833333
classification_error   : 0.166667
f1_score               : 0.800000
Shrinkage_LDA | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
Shrinkage_LDA | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 

In [274]:
df_gnb = run_gaussiannb_on_saved_folds(
    "results_colon_cv_seed42_4_100_2_64_5000_0.25_cG_0.7_harm",
    var_smoothing=1e-9,
    seed=42,
)

GaussianNB | Fold 1
selected_indices       : [14, 240, 266, 376, 492, 512, 779, 896, 1041, 1581, 1634, 1771, 1883, 1893]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
GaussianNB | Fold 2
selected_indices       : [42, 81, 142, 240, 266, 512, 558, 738, 1003, 1422, 1581, 1633, 1771, 1872]
accuracy               : 0.857143
classification_error   : 0.142857
f1_score               : 0.800000
GaussianNB | Fold 3
selected_indices       : [13, 244, 266, 364, 492, 764, 779, 1003, 1422, 1581, 1634, 1671, 1872, 1916]
accuracy               : 0.833333
classification_error   : 0.166667
f1_score               : 0.800000
GaussianNB | Fold 4
selected_indices       : [46, 285, 359, 410, 575, 963, 1472, 1548, 1667, 1771, 1807, 1835, 1883, 1959]
accuracy               : 1.000000
classification_error   : 0.000000
f1_score               : 1.000000
GaussianNB | Fold 5
selected_indices       : [46, 390, 398, 410, 488, 514, 624, 993, 1001, 1230, 1548, 164

In [147]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, Tuple
import pandas as pd


def _read_precomputed_stats(csv_path: Path, metrics: Iterable[str]) -> Dict[str, Tuple[float, float]]:
    """
    Читает готовые mean/std из CSV, где есть строки с first-column == 'mean' и 'std'.

    Возвращает:
        {
            metric_name: (mean_value, std_value),
            ...
        }
    """
    df = pd.read_csv(csv_path)

    if df.empty:
        raise ValueError(f"Файл пустой: {csv_path}")

    key_col = df.columns[0]
    key_vals = df[key_col].astype(str).str.strip().str.lower()

    mean_rows = df.loc[key_vals == "mean"]
    std_rows = df.loc[key_vals == "std"]

    if len(mean_rows) != 1 or len(std_rows) != 1:
        raise ValueError(
            f"В файле {csv_path} не удалось однозначно найти строки 'mean' и 'std'"
        )

    mean_row = mean_rows.iloc[0]
    std_row = std_rows.iloc[0]

    out: Dict[str, Tuple[float, float]] = {}
    for metric in metrics:
        if metric not in df.columns:
            raise ValueError(f"В файле {csv_path} нет колонки '{metric}'")

        out[metric] = (float(mean_row[metric]), float(std_row[metric]))

    return out


def print_ablation_summary(root_dir: str | Path) -> str:
    """
    Проходит по всем подпапкам results_colon_cv_seed* внутри root_dir
    и печатает сводку по готовым mean/std из csv-файлов.

    Ничего не пересчитывает по фолдам.
    """
    root = Path(root_dir).expanduser()

    if not root.exists():
        raise FileNotFoundError(f"Папка не найдена: {root}")

    ablation_dirs = sorted(
        p for p in root.iterdir()
        if p.is_dir() and p.name.startswith("results_colon_cv_seed")
    )

    if not ablation_dirs:
        text = f"В папке {root} не найдено подпапок results_colon_cv_seed*"
        print(text)
        return text

    classical_models = [
        ("results.csv", "Логистическая регрессия", ["accuracy", "f1_score"]),
        ("svm_results.csv", "SVM", ["accuracy", "f1_score"]),
        ("knn_results.csv", "KNN", ["accuracy", "f1_score"]),
        ("gaussiannb_results.csv", "GaussianNB", ["accuracy", "f1_score"]),
        ("kde_bayes_results.csv", "KDE Bayes", ["accuracy", "f1_score"]),
        ("qda_results.csv", "QDA", ["accuracy", "f1_score"]),
        ("shrinkage_lda_results.csv", "Shrinkage LDA", ["accuracy", "f1_score"]),
    ]

    unary_metrics = [
        "S_test",
        "F12_test",
        "G12_test",
        "coverage",
        "conflict_rate",
        "reject_rate",
        "selective_accuracy",
        "selective_f1",
        "conservative_accuracy",
    ]

    blocks = []

    for ablation_dir in ablation_dirs:
        lines = [f"Абляция: {ablation_dir.name}", ""]

        for filename, model_name, metrics in classical_models:
            file_path = ablation_dir / filename

            if not file_path.exists():
                lines.append(f"{model_name}: файл {filename} не найден")
                continue

            try:
                stats = _read_precomputed_stats(file_path, metrics)
                acc_mean, acc_std = stats["accuracy"]
                f1_mean, f1_std = stats["f1_score"]

                lines.append(
                    f"{model_name}: accuracy = {acc_mean:.4f} ± {acc_std:.4f}, "
                    f"f1_score = {f1_mean:.4f} ± {f1_std:.4f}"
                )
            except Exception as e:
                lines.append(f"{model_name}: ошибка чтения {filename}: {e}")

        lines.append("")

        unary_path = ablation_dir / "unary_strict_results.csv"
        if not unary_path.exists():
            lines.append("Unary strict: файл unary_strict_results.csv не найден")
        else:
            try:
                stats = _read_precomputed_stats(unary_path, unary_metrics)
                parts = [
                    f"{metric} = {stats[metric][0]:.4f} ± {stats[metric][1]:.4f}"
                    for metric in unary_metrics
                ]
                lines.append("Unary strict: " + ", ".join(parts))
            except Exception as e:
                lines.append(f"Unary strict: ошибка чтения unary_strict_results.csv: {e}")

        blocks.append("\n".join(lines))

    result_text = "\n\n".join(blocks)
    print(result_text)
    return result_text

In [275]:
print_ablation_summary("/home/eliseev")

Абляция: results_colon_cv_seed42_1_400_2_64_5000_1

Логистическая регрессия: accuracy = 0.8524 ± 0.1656, f1_score = 0.8138 ± 0.1880
SVM: accuracy = 0.8238 ± 0.1620, f1_score = 0.7805 ± 0.1807
KNN: accuracy = 0.8381 ± 0.1522, f1_score = 0.7433 ± 0.3151
GaussianNB: accuracy = 0.8381 ± 0.1522, f1_score = 0.7433 ± 0.3151
KDE Bayes: accuracy = 0.7738 ± 0.2240, f1_score = 0.6938 ± 0.3345
QDA: accuracy = 0.7929 ± 0.1459, f1_score = 0.6067 ± 0.3627
Shrinkage LDA: accuracy = 0.8214 ± 0.1797, f1_score = 0.7338 ± 0.3191

Unary strict: S_test = 0.3323 ± 0.4478, F12_test = 0.3989 ± 0.3116, G12_test = 0.0666 ± 0.2107, coverage = 0.5333 ± 0.1922, conflict_rate = 0.1595 ± 0.2042, reject_rate = 0.3071 ± 0.1836, selective_accuracy = 0.9667 ± 0.1054, selective_f1 = 0.7800 ± 0.4158, conservative_accuracy = 0.5167 ± 0.2024

Абляция: results_colon_cv_seed42_2_200_2_64_5000_0.25

Логистическая регрессия: файл results.csv не найден
SVM: accuracy = 0.8548 ± 0.1158, f1_score = 0.7900 ± 0.1707
KNN: accuracy = 0.

'Абляция: results_colon_cv_seed42_1_400_2_64_5000_1\n\nЛогистическая регрессия: accuracy = 0.8524 ± 0.1656, f1_score = 0.8138 ± 0.1880\nSVM: accuracy = 0.8238 ± 0.1620, f1_score = 0.7805 ± 0.1807\nKNN: accuracy = 0.8381 ± 0.1522, f1_score = 0.7433 ± 0.3151\nGaussianNB: accuracy = 0.8381 ± 0.1522, f1_score = 0.7433 ± 0.3151\nKDE Bayes: accuracy = 0.7738 ± 0.2240, f1_score = 0.6938 ± 0.3345\nQDA: accuracy = 0.7929 ± 0.1459, f1_score = 0.6067 ± 0.3627\nShrinkage LDA: accuracy = 0.8214 ± 0.1797, f1_score = 0.7338 ± 0.3191\n\nUnary strict: S_test = 0.3323 ± 0.4478, F12_test = 0.3989 ± 0.3116, G12_test = 0.0666 ± 0.2107, coverage = 0.5333 ± 0.1922, conflict_rate = 0.1595 ± 0.2042, reject_rate = 0.3071 ± 0.1836, selective_accuracy = 0.9667 ± 0.1054, selective_f1 = 0.7800 ± 0.4158, conservative_accuracy = 0.5167 ± 0.2024\n\nАбляция: results_colon_cv_seed42_2_200_2_64_5000_0.25\n\nЛогистическая регрессия: файл results.csv не найден\nSVM: accuracy = 0.8548 ± 0.1158, f1_score = 0.7900 ± 0.1707\nK